### **T2.1 - extract relevant MIMIC-IV tables**

This notebook links the psychiatric readmission cohort to structured MIMIC-IV hospital and ICU tables, creating one admission-level dataset for later cleaning and feature engineering.


##### **Notebook Outline**
- **setup and cohort load:** load the T1.4 cohort and define hospital, ICU, and output paths.
- **core admission features:** extract demographics, admission pathway, prior utilisation, ICU exposure, transfers, services, procedures, DRG, diagnoses, medications, labs, and vitals.
- **raw-table feature blocks:** add richer admission-level signals from POE, POE detail, pharmacy, EMAR, HCPCS, microbiology, OMR, and ICU event tables.
- **output save:** write the extracted admission-level and hourly clinical datasets for WP2.2 and WP2.3.


In [1]:
#load packages and shared helpers for this notebook
import sys
import pandas as pd
import re
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
import numpy as np
from statsmodels.stats.contingency_tables import Table2x2
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
print(sys.executable)

/scratch/DissProject/my_env/bin/python


**Load the T1.4 cohort and define paths**

The cohort created in T1.4 is loaded first. This dataset already contains the psychiatric admissions and the 30-day readmission outcome, so it acts as the base dataset for feature extraction. Relevant MIMIC-IV tables are then linked to this cohort using `subject_id` and `hadm_id`.

In [2]:
#base directory where mimic-iv is stored
base_path = Path("/scratch/DissProject/mimic-iv-3.1")

#paths to mimic-iv modules
hosp_path = base_path / "hosp"
icu_path = base_path / "icu"

#folder where previous outputs are saved
output_path = Path("/scratch/DissProject/outputs")

#load the cohort created in t1.4
cohort_path = output_path / "psychiatric_readmission_cohort.csv"
df = pd.read_csv(cohort_path)
print("Cohort shape:", df.shape)
print("Cohort columns:")
print(df.columns.tolist())
print(df.head())

Cohort shape: (238565, 9)
Cohort columns:
['subject_id', 'hadm_id', 'admittime', 'dischtime', 'is_psych', 'next_admittime', 'next_is_psych', 'days_to_next', 'readmitted_30d']
   subject_id   hadm_id            admittime            dischtime  is_psych  \
0    10000032  22595853  2180-05-06 22:23:00  2180-05-07 17:15:00       1.0   
1    10000032  22841357  2180-06-26 18:27:00  2180-06-27 18:49:00       1.0   
2    10000032  29079034  2180-07-23 12:35:00  2180-07-25 17:55:00       1.0   
3    10000032  25742920  2180-08-05 23:44:00  2180-08-07 17:50:00       1.0   
4    10000068  25022803  2160-03-03 23:16:00  2160-03-04 06:26:00       1.0   

        next_admittime  next_is_psych  days_to_next  readmitted_30d  
0  2180-06-26 18:27:00            1.0          50.0               0  
1  2180-07-23 12:35:00            1.0          25.0               1  
2  2180-08-05 23:44:00            1.0          11.0               1  
3                  NaN            NaN           NaN               0  


In [3]:
#check loaded cohort shape and existing diagnosis-related fields
print("Loaded cohort shape:", df.shape)
print("Loaded cohort columns:")
print(df.columns.tolist())

#identify any diagnosis or psychiatric columns already carried forward from earlier extraction
diagnosis_related_cols = [col for col in df.columns if "diagnos" in col.lower() or "psychotic" in col.lower() or "bipolar" in col.lower() or "depression" in col.lower()]
print("\nDiagnosis-related columns already present:")
print(diagnosis_related_cols)

Loaded cohort shape: (238565, 9)
Loaded cohort columns:
['subject_id', 'hadm_id', 'admittime', 'dischtime', 'is_psych', 'next_admittime', 'next_is_psych', 'days_to_next', 'readmitted_30d']

Diagnosis-related columns already present:
[]


In [4]:
#check whether diagnosis feature names already exist before adding grouped diagnosis flags
print(df.shape)

#count exact or prefixed diagnosis columns so duplicate feature creation can be spotted early
diagnosis_cols = ["num_total_diagnoses", "num_psych_diagnoses", "num_nonpsych_diagnoses",
    "has_anxiety_ptsd", "has_bipolar_disorder", "has_cognitive_delirium", "has_depression",
    "has_other_psych_diagnosis", "has_personality_disorder", "has_psychotic_disorder", "has_substance_use"]

for col in diagnosis_cols:
    print(col, sum(c.startswith(col) for c in df.columns))

(238565, 9)
num_total_diagnoses 0
num_psych_diagnoses 0
num_nonpsych_diagnoses 0
has_anxiety_ptsd 0
has_bipolar_disorder 0
has_cognitive_delirium 0
has_depression 0
has_other_psych_diagnosis 0
has_personality_disorder 0
has_psychotic_disorder 0
has_substance_use 0


**Select core cohort variables and load + extract patient-level variables**

Only relevant columns are kept from the T1.4 cohort. These include identifiers, admission/discharge times, the calculated time to next admission, and the 30-day readmission label. This keeps the dataset focused before adding demographic and admission-level features.

The `patients` table contains demographic variables. For this initial modelling dataset, `gender`, `anchor_age`, and `anchor_year_group` are extracted. These variables are linked to the cohort using `subject_id`, as they describe the patient rather than a specific admission.

In [5]:
#keep only useful columns from the cohort
df = df[["subject_id", "hadm_id", "admittime", "dischtime", "days_to_next", "readmitted_30d"]].copy()
print("Cohort after selecting useful columns:", df.shape)
print(df.head())

#load patients table
patients = pd.read_csv(hosp_path / "patients.csv.gz")
print("Patients shape:", patients.shape)
print("Patients columns:")
print(patients.columns.tolist())
print(patients.head())

#keep demographic variables
patients = patients[["subject_id", "gender", "anchor_age", "anchor_year_group"]].copy()
print("Patients after selecting variables:", patients.shape)
print(patients.head())

Cohort after selecting useful columns: (238565, 6)
   subject_id   hadm_id            admittime            dischtime  \
0    10000032  22595853  2180-05-06 22:23:00  2180-05-07 17:15:00   
1    10000032  22841357  2180-06-26 18:27:00  2180-06-27 18:49:00   
2    10000032  29079034  2180-07-23 12:35:00  2180-07-25 17:55:00   
3    10000032  25742920  2180-08-05 23:44:00  2180-08-07 17:50:00   
4    10000068  25022803  2160-03-03 23:16:00  2160-03-04 06:26:00   

   days_to_next  readmitted_30d  
0          50.0               0  
1          25.0               1  
2          11.0               1  
3           NaN               0  
4           NaN               0  
Patients shape: (364627, 6)
Patients columns:
['subject_id', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod']
   subject_id gender  anchor_age  anchor_year anchor_year_group         dod
0    10000032      F          52         2180       2014 - 2016  2180-09-09
1    10000048      F          23         2126     

**Merge patient variables with the cohort**

The patient-level variables are merged onto the psychiatric readmission cohort using `subject_id`. This adds demographic information to each psychiatric admission.

In [6]:
#merge cohort with patients table
df = df.merge(patients, on="subject_id", how="left")
print("Shape after merging patients:", df.shape)
print("Missing values after patient merge:")
print(df[["gender", "anchor_age", "anchor_year_group"]].isna().sum())
print("Duplicate subject_id + hadm_id rows:")
print(df.duplicated(subset=["subject_id", "hadm_id"]).sum())
print(df.head())

Shape after merging patients: (238565, 9)
Missing values after patient merge:
gender               0
anchor_age           0
anchor_year_group    0
dtype: int64
Duplicate subject_id + hadm_id rows:
0
   subject_id   hadm_id            admittime            dischtime  \
0    10000032  22595853  2180-05-06 22:23:00  2180-05-07 17:15:00   
1    10000032  22841357  2180-06-26 18:27:00  2180-06-27 18:49:00   
2    10000032  29079034  2180-07-23 12:35:00  2180-07-25 17:55:00   
3    10000032  25742920  2180-08-05 23:44:00  2180-08-07 17:50:00   
4    10000068  25022803  2160-03-03 23:16:00  2160-03-04 06:26:00   

   days_to_next  readmitted_30d gender  anchor_age anchor_year_group  
0          50.0               0      F          52       2014 - 2016  
1          25.0               1      F          52       2014 - 2016  
2          11.0               1      F          52       2014 - 2016  
3           NaN               0      F          52       2014 - 2016  
4           NaN               0

**Load and extract admission-level variables and merge with cohort**

The `admissions` table contains information specific to each hospital admission, including admission type, location, discharge location, insurance, language, marital status, race, and in-hospital mortality flag. These variables are linked using both `subject_id` and `hadm_id`. Admission-level variables are merged using `subject_id` and `hadm_id`, ensuring that each row receives information from the correct hospital admission.

In [7]:
#load admissions table
admissions = pd.read_csv(hosp_path / "admissions.csv.gz")
print("Admissions shape:", admissions.shape)
print("Admissions columns:")
print(admissions.columns.tolist())
print(admissions.head())

#keep admission-level variables
admissions = admissions[["subject_id", "hadm_id", "admission_type", "admission_location",
        "discharge_location", "insurance", "language", "marital_status", "race"]].copy()

print("Admissions after selecting variables:", admissions.shape)
print(admissions.head())

#merge with admission-level variables
df = df.merge(admissions, on=["subject_id", "hadm_id"], how="left")
print("Shape after merging admissions:", df.shape)
print("Missing values after admissions merge:")
print(df[["admission_type", "admission_location", "discharge_location", "insurance",
        "language",  "marital_status", "race"]].isna().sum())
print("Duplicate subject_id + hadm_id rows:")
print(df.duplicated(subset=["subject_id", "hadm_id"]).sum())
print(df.head())

Admissions shape: (546028, 16)
Admissions columns:
['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admit_provider_id', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'edregtime', 'edouttime', 'hospital_expire_flag']
   subject_id   hadm_id            admittime            dischtime deathtime  \
0    10000032  22595853  2180-05-06 22:23:00  2180-05-07 17:15:00       NaN   
1    10000032  22841357  2180-06-26 18:27:00  2180-06-27 18:49:00       NaN   
2    10000032  25742920  2180-08-05 23:44:00  2180-08-07 17:50:00       NaN   
3    10000032  29079034  2180-07-23 12:35:00  2180-07-25 17:55:00       NaN   
4    10000068  25022803  2160-03-03 23:16:00  2160-03-04 06:26:00       NaN   

   admission_type admit_provider_id      admission_location  \
0          URGENT            P49AFC  TRANSFER FROM HOSPITAL   
1        EW EMER.            P784FA          EMERGENCY ROOM   
2        EW EMER.            P19UTS 

**Create hospital length of stay**

Hospital length of stay is calculated using the difference between discharge time and admission time. This provides an admission-level measure of how long the patient remained in hospital during the index admission.

In [8]:
#convert timestamps to datetime
df["admittime"] = pd.to_datetime(df["admittime"])
df["dischtime"] = pd.to_datetime(df["dischtime"])

#calculate hospital length of stay in days
df["hospital_los_days"] = (df["dischtime"] - df["admittime"]
).dt.total_seconds() / (60 * 60 * 24)

print("Hospital length of stay stats:")
print(df["hospital_los_days"].describe())
print("Admissions with negative hospital length of stay:")
print((df["hospital_los_days"] < 0).sum())

#add admission/discharge, social-proxy, and los category indicators
admission_location_upper = df["admission_location"].fillna("Unknown").astype(str).str.upper()
discharge_location_upper = df["discharge_location"].fillna("Unknown").astype(str).str.upper()
insurance_upper = df["insurance"].fillna("Unknown").astype(str).str.upper()
language_upper = df["language"].fillna("Unknown").astype(str).str.upper()
marital_status_upper = df["marital_status"].fillna("Unknown").astype(str).str.upper()

df["discharged_against_advice"] = discharge_location_upper.eq("AGAINST ADVICE").astype(int)
df["not_married_flag"] = marital_status_upper.isin(["SINGLE", "DIVORCED", "WIDOWED", "SEPARATED"]).astype(int)
df["single_or_divorced_or_widowed"] = marital_status_upper.isin(["SINGLE", "DIVORCED", "WIDOWED"]).astype(int)
df["non_english_language_flag"] = (~language_upper.eq("ENGLISH")).astype(int)
df["public_insurance_flag"] = insurance_upper.isin(["MEDICAID", "MEDICARE"]).astype(int)
df["medicaid_flag"] = insurance_upper.eq("MEDICAID").astype(int)
df["medicare_flag"] = insurance_upper.eq("MEDICARE").astype(int)
df["insurance_missing_flag"] = df["insurance"].isna().astype(int)
df["admitted_from_facility_flag"] = admission_location_upper.str.contains("SKILLED NURSING|FACILITY|NURSING|REHAB", regex=True).astype(int)
df["admitted_from_hospital_transfer_flag"] = admission_location_upper.str.contains("TRANSFER FROM HOSPITAL|HOSPITAL", regex=True).astype(int)
df["emergency_room_admission_flag"] = admission_location_upper.str.contains("EMERGENCY ROOM", regex=False).astype(int)
df["transfer_from_hospital_flag"] = admission_location_upper.str.contains("TRANSFER FROM HOSPITAL|HOSPITAL", regex=True).astype(int)
df["transfer_from_snf_flag"] = admission_location_upper.str.contains("SKILLED NURSING", regex=False).astype(int)
df["internal_transfer_from_psych_flag"] = admission_location_upper.str.contains("INTERNAL TRANSFER TO OR FROM PSYCH|PSYCH", regex=True).astype(int)
df["discharged_home_flag"] = discharge_location_upper.eq("HOME").astype(int)
df["discharged_to_facility_flag"] = discharge_location_upper.str.contains("SKILLED NURSING|REHAB|LONG TERM|FACILITY|HEALTH CARE|CHRONIC", regex=True).astype(int)
df["discharged_to_psych_facility_flag"] = discharge_location_upper.str.contains("PSYCH", regex=False).astype(int)

df["los_under_2_days"] = (df["hospital_los_days"] < 2).astype(int)
df["los_under_7_days"] = (df["hospital_los_days"] < 7).astype(int)
df["los_7_to_30_days"] = ((df["hospital_los_days"] >= 7) & (df["hospital_los_days"] < 30)).astype(int)
df["los_30plus_days"] = (df["hospital_los_days"] >= 30).astype(int)
df["los_60plus_days"] = (df["hospital_los_days"] >= 60).astype(int)

admission_proxy_cols = ["discharged_against_advice", "not_married_flag", "single_or_divorced_or_widowed",
    "non_english_language_flag", "public_insurance_flag", "medicaid_flag", "medicare_flag",
    "insurance_missing_flag", "admitted_from_facility_flag", "admitted_from_hospital_transfer_flag",
    "emergency_room_admission_flag", "transfer_from_hospital_flag", "transfer_from_snf_flag",
    "internal_transfer_from_psych_flag", "discharged_home_flag", "discharged_to_facility_flag",
    "discharged_to_psych_facility_flag", "los_under_2_days", "los_under_7_days", "los_7_to_30_days",
    "los_30plus_days", "los_60plus_days"]
print("Admission, social-proxy, and LOS category indicators created:")
print(df[admission_proxy_cols].sum().sort_values(ascending=False))

Hospital length of stay stats:
count    238565.000000
mean          5.367630
std           8.264140
min          -0.943750
25%           1.136111
50%           3.011111
75%           6.220833
max         515.562500
Name: hospital_los_days, dtype: float64
Admissions with negative hospital length of stay:
74
Admission, social-proxy, and LOS category indicators created:
los_under_7_days                        187597
public_insurance_flag                   166774
single_or_divorced_or_widowed           158242
not_married_flag                        158242
emergency_room_admission_flag           113857
medicare_flag                           106156
los_under_2_days                         89419
discharged_to_facility_flag              83808
discharged_home_flag                     72498
medicaid_flag                            60618
los_7_to_30_days                         46640
transfer_from_hospital_flag              26726
admitted_from_hospital_transfer_flag     26726
non_english_languag

**Create prior hospital utilisation features**

Admission records were ordered chronologically for each patient and aggregated at the admission level. Three features were derived: the total number of previous hospital admissions, no. of previous psychiatric admissions, and no. of previous non-psychiatric admissions prior to the index admission. Only admissions occurring before the current admission were included to prevent information leakage from future hospital encounters. These variables were retained as repeated hospital utilisation has been associated with increased risk of subsequent readmission in both general and psychiatric populations and may provide complementary information beyond diagnosis, medication, laboratory, and physiological measures.

In [9]:
#reload admissions with timestamps so previous hospital use can be calculated
all_admissions_history = pd.read_csv(hosp_path / "admissions.csv.gz", usecols=["subject_id", "hadm_id", "admittime", "dischtime"])
all_admissions_history["admittime"] = pd.to_datetime(all_admissions_history["admittime"], errors="coerce")
all_admissions_history["dischtime"] = pd.to_datetime(all_admissions_history["dischtime"], errors="coerce")

#identify psychiatric admissions using the current cohort hadm_ids
psych_hadm_ids = set(df["hadm_id"].astype(int))
all_admissions_history["is_psych_admission"] = (all_admissions_history["hadm_id"].astype(int).isin(psych_hadm_ids).astype(int))

#sort admissions chronologically within each patient
all_admissions_history = all_admissions_history.sort_values(["subject_id", "admittime", "hadm_id"]).reset_index(drop=True)

#count all previous admissions before the current admission
all_admissions_history["previous_total_admissions"] = (all_admissions_history.groupby("subject_id").cumcount())

#count previous psychiatric admissions before the current admission
all_admissions_history["previous_psych_admissions_from_all_hosp"] = (
    all_admissions_history.groupby("subject_id")["is_psych_admission"] .transform(lambda x: x.shift(fill_value=0).cumsum()))

#count previous non-psychiatric admissions before the current admission
all_admissions_history["previous_nonpsych_admissions"] = (all_admissions_history["previous_total_admissions"]
    - all_admissions_history["previous_psych_admissions_from_all_hosp"])

#safety check: previous non-psychiatric admissions should never be negative
negative_nonpsych = (all_admissions_history["previous_nonpsych_admissions"] < 0).sum()
print("Negative previous non-psychiatric admission counts:", negative_nonpsych)
if negative_nonpsych > 0:
    raise ValueError("Previous non-psychiatric admission count should not be negative. Check admission ordering or psychiatric admission flag.")

prior_admission_features = all_admissions_history[["subject_id", "hadm_id", "previous_total_admissions",
        "previous_psych_admissions_from_all_hosp", "previous_nonpsych_admissions"]].copy()

#merge prior admission utilisation features into the main dataset
df = df.merge(prior_admission_features, on=["subject_id", "hadm_id"], how="left")
prior_admission_cols = ["previous_total_admissions",  "previous_psych_admissions_from_all_hosp",
    "previous_nonpsych_admissions"]

df[prior_admission_cols] = df[prior_admission_cols].fillna(0).astype(int)
print("Prior hospital utilisation features created.")
print(df[prior_admission_cols].describe())
print("\nPreview:")
print(df[["subject_id", "hadm_id", "admittime"] + prior_admission_cols].head(10))

Negative previous non-psychiatric admission counts: 0
Prior hospital utilisation features created.
       previous_total_admissions  previous_psych_admissions_from_all_hosp  \
count              238565.000000                            238565.000000   
mean                    4.547423                                 3.491736   
std                    10.572356                                 9.656373   
min                     0.000000                                 0.000000   
25%                     0.000000                                 0.000000   
50%                     1.000000                                 1.000000   
75%                     4.000000                                 3.000000   
max                   237.000000                               237.000000   

       previous_nonpsych_admissions  
count                 238565.000000  
mean                       1.055687  
std                        2.812366  
min                        0.000000  
25%              

**Load and aggregate ICU stay variables**

The `icustays` table contains ICU-specific information. Some hospital admissions may contain multiple ICU stays, so ICU data are aggregated to one row per hospital admission. This creates features for whether the patient had an ICU stay, the number of ICU stays, and total ICU length of stay.

In [10]:
#load icu stays table
icustays = pd.read_csv(icu_path / "icustays.csv.gz")

print("ICU stays shape:", icustays.shape)
print("ICU stays columns:")
print(icustays.columns.tolist())
print(icustays.head())

#aggregate icu stays to one row per hospital admission
icu_features = icustays.groupby(["subject_id", "hadm_id"]).agg(
    icu_stay_count=("stay_id", "nunique"),
    total_icu_los_days=("los", "sum"),
    first_icu_careunit=("first_careunit", "first"),
    last_icu_careunit=("last_careunit", "last")).reset_index()

#mark admissions with an icu stay
icu_features["had_icu_stay"] = 1

print("ICU features after aggregation:", icu_features.shape)
print(icu_features.head())
print("ICU stay count stats:")
print(icu_features["icu_stay_count"].describe())


ICU stays shape: (94458, 8)
ICU stays columns:
['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit', 'intime', 'outtime', 'los']
   subject_id   hadm_id   stay_id                       first_careunit  \
0    10000032  29079034  39553978   Medical Intensive Care Unit (MICU)   
1    10000690  25860671  37081114   Medical Intensive Care Unit (MICU)   
2    10000980  26913865  39765666   Medical Intensive Care Unit (MICU)   
3    10001217  24597018  37067082  Surgical Intensive Care Unit (SICU)   
4    10001217  27703517  34592300  Surgical Intensive Care Unit (SICU)   

                         last_careunit               intime  \
0   Medical Intensive Care Unit (MICU)  2180-07-23 14:00:00   
1   Medical Intensive Care Unit (MICU)  2150-11-02 19:37:00   
2   Medical Intensive Care Unit (MICU)  2189-06-27 08:42:00   
3  Surgical Intensive Care Unit (SICU)  2157-11-20 19:18:02   
4  Surgical Intensive Care Unit (SICU)  2157-12-19 15:42:24   

               outtime       

**Merge ICU variables with cohort**

The ICU table-derived columns are merged onto the cohort using `subject_id` and `hadm_id`. Admissions without ICU records are assigned zero for ICU-related numerical variables.

In [11]:
#merge icu features with the main dataset
df = df.merge(icu_features, on=["subject_id", "hadm_id"], how="left")

#fill admissions with no icu stay
df["had_icu_stay"] = df["had_icu_stay"].fillna(0).astype(int)
df["icu_stay_count"] = df["icu_stay_count"].fillna(0).astype(int)
df["total_icu_los_days"] = df["total_icu_los_days"].fillna(0)

print("Shape after merging ICU features:", df.shape)
print("Duplicate subject_id + hadm_id rows:")
print(df.duplicated(subset=["subject_id", "hadm_id"]).sum())
print("ICU indicator counts:")
print(df["had_icu_stay"].value_counts())
print("Total ICU LOS stats:")
print(df["total_icu_los_days"].describe())
print(df.head())

Shape after merging ICU features: (238565, 47)
Duplicate subject_id + hadm_id rows:
0
ICU indicator counts:
had_icu_stay
0    198054
1     40511
Name: count, dtype: int64
Total ICU LOS stats:
count    238565.000000
mean          0.728449
std           3.167756
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         226.537488
Name: total_icu_los_days, dtype: float64
   subject_id   hadm_id           admittime           dischtime  days_to_next  \
0    10000032  22595853 2180-05-06 22:23:00 2180-05-07 17:15:00          50.0   
1    10000032  22841357 2180-06-26 18:27:00 2180-06-27 18:49:00          25.0   
2    10000032  29079034 2180-07-23 12:35:00 2180-07-25 17:55:00          11.0   
3    10000032  25742920 2180-08-05 23:44:00 2180-08-07 17:50:00           NaN   
4    10000068  25022803 2160-03-03 23:16:00 2160-03-04 06:26:00           NaN   

   readmitted_30d gender  anchor_age anchor_year_group  admission_type  ...  \
0               0

**Create ICU type feature**

The ICU care unit variables were simplified into a broader ICU type feature. This provides a more interpretable representation of the type of critical care environment involved in the admission, such as medical ICU, surgical ICU, cardiac ICU, or neurological ICU.

In [12]:
#define broader icu type categories from first icu care unit
def classify_icu_type(careunit):
    if pd.isna(careunit):
        return "No ICU"

    careunit = str(careunit).lower()
    if "coronary" in careunit or "cardiac" in careunit or "cvicu" in careunit:
        return "Cardiac ICU"
    elif "neuro" in careunit:
        return "Neuro ICU"
    elif "surgical" in careunit or "sicu" in careunit or "trauma" in careunit:
        return "Surgical/Trauma ICU"
    elif "medical" in careunit or "micu" in careunit:
        return "Medical ICU"
    else:
        return "Other ICU"

df["icu_type"] = df["first_icu_careunit"].apply(classify_icu_type)

print("ICU type distribution:")
print(df["icu_type"].value_counts(dropna=False))
print("\nReadmission rate by ICU type:")
print(df.groupby("icu_type")["readmitted_30d"].mean().mul(100).round(2))

ICU type distribution:
icu_type
No ICU                 198054
Surgical/Trauma ICU     17110
Medical ICU             10685
Cardiac ICU              8952
Neuro ICU                3631
Other ICU                 133
Name: count, dtype: int64

Readmission rate by ICU type:
icu_type
Cardiac ICU            12.79
Medical ICU            17.28
Neuro ICU               9.23
No ICU                 20.90
Other ICU               7.52
Surgical/Trauma ICU    15.40
Name: readmitted_30d, dtype: float64


**Load and aggregate hospital service transfer features**

The `services` table records changes in hospital service during an admission. Service transfers were aggregated to admission level to capture care pathway complexity. Admissions involving multiple service records may represent more complex hospital journeys, which could be relevant to readmission risk.

In [13]:
#load services table
services = pd.read_csv(hosp_path / "services.csv.gz")

print("Services shape:", services.shape)
print("Services columns:")
print(services.columns.tolist())
print(services.head())

#restrict services to admissions in current cohort
services = services.merge(df[["subject_id", "hadm_id"]], on=["subject_id", "hadm_id"], how="inner")
print("\nServices after restricting to cohort admissions:", services.shape)

#standardise service names for service-involvement indicators
services["prev_service_clean"] = services["prev_service"].fillna("").astype(str).str.upper()
services["curr_service_clean"] = services["curr_service"].fillna("").astype(str).str.upper()
services["psych_service_involved_anytime"] = (
    services["prev_service_clean"].str.contains("PSYCH", regex=False) |
    services["curr_service_clean"].str.contains("PSYCH", regex=False)).astype(int)
services["medicine_service_involved_anytime"] = (
    services["prev_service_clean"].str.contains("MED", regex=False) |
    services["curr_service_clean"].str.contains("MED", regex=False)).astype(int)

#aggregate service records to admission level
service_features = services.groupby(["subject_id", "hadm_id"]).agg(
    num_service_records=("curr_service", "count"), num_unique_services=("curr_service", "nunique"),
    first_service=("curr_service", "first"), last_service=("curr_service", "last"),
    psych_service_involved_anytime=("psych_service_involved_anytime", "max"),
    medicine_service_involved_anytime=("medicine_service_involved_anytime", "max")).reset_index()

#number of transfers is service records minus first service assignment
service_features["num_service_transfers"] = (service_features["num_service_records"] - 1).clip(lower=0)

#indicator for whether the admission involved service transfer
service_features["had_service_transfer"] = (service_features["num_service_transfers"] > 0).astype(int)
service_features["medicine_and_psych_services_both_flag"] = ((service_features["medicine_service_involved_anytime"] == 1) &
    (service_features["psych_service_involved_anytime"] == 1)).astype(int)
print("\nService features shape:", service_features.shape)
print(service_features.head())
print("\nService transfer distribution:")
print(service_features["num_service_transfers"].describe())

#merge service features into main dataset
df = df.merge(service_features, on=["subject_id", "hadm_id"], how="left")

#fill admissions without service rows
service_count_cols = ["num_service_records", "num_unique_services", "num_service_transfers", "had_service_transfer",
    "psych_service_involved_anytime", "medicine_service_involved_anytime", "medicine_and_psych_services_both_flag"]

#loop through each item in this feature block
for col in service_count_cols:
    df[col] = df[col].fillna(0).astype(int)

for col in ["first_service", "last_service"]:
    df[col] = df[col].fillna("Unknown")

print("\nShape after merging service features:", df.shape)
print("\nMissing service feature values:")
print(df[service_count_cols + ["first_service", "last_service"]].isna().sum())

print("\nService feature preview:")
print(df[["subject_id", "hadm_id"] + service_count_cols + ["first_service", "last_service"]].head())

Services shape: (593071, 5)
Services columns:
['subject_id', 'hadm_id', 'transfertime', 'prev_service', 'curr_service']
   subject_id   hadm_id         transfertime prev_service curr_service
0    10000032  22595853  2180-05-06 22:24:57          NaN          MED
1    10000032  22841357  2180-06-26 18:28:08          NaN          MED
2    10000032  25742920  2180-08-05 23:44:50          NaN          MED
3    10000032  29079034  2180-07-23 12:36:04          NaN          MED
4    10000068  25022803  2160-03-03 23:17:17          NaN          MED

Services after restricting to cohort admissions: (259632, 5)

Service features shape: (238555, 11)
   subject_id   hadm_id  num_service_records  num_unique_services  \
0    10000032  22595853                    1                    1   
1    10000032  22841357                    1                    1   
2    10000032  25742920                    1                    1   
3    10000032  29079034                    1                    1   
4    1000

**Load and aggregate physical transfer features**

The `transfers` table records physical movement through hospital locations such as the emergency department, wards, ICU-related areas, and discharge locations. These records were aggregated to admission level to capture care pathway complexity beyond clinical service changes alone. Frequent transfers or exposure to multiple care units may indicate more complex inpatient trajectories, which could be relevant to psychiatric readmission risk.

In [14]:
#load physical hospital transfer table
transfers = pd.read_csv(hosp_path / "transfers.csv.gz")
print("Transfers shape:", transfers.shape)
print("Transfers columns:")
print(transfers.columns.tolist())
print(transfers.head())

#restrict transfers to admissions in current cohort
cohort_keys = df[["subject_id", "hadm_id"]].drop_duplicates()
transfers = transfers.dropna(subset=["hadm_id"]).copy()
transfers["hadm_id"] = transfers["hadm_id"].astype(int)
#join the derived features back to the admission table
transfers = transfers.merge(cohort_keys, on=["subject_id", "hadm_id"], how="inner")
print("\nTransfers after restricting to cohort admissions:", transfers.shape)

#standardise text fields for indicators
transfers["eventtype_clean"] = transfers["eventtype"].fillna("Unknown").astype(str).str.upper()
transfers["careunit_clean"] = transfers["careunit"].fillna("Unknown").astype(str)

#exclude unknown discharge placeholder from unique care-unit count
known_careunit = transfers.loc[~transfers["careunit_clean"].str.upper().eq("UNKNOWN")].copy()

#aggregate transfer records to admission level
transfer_features = transfers.groupby(["subject_id", "hadm_id"]).agg(
    num_transfer_events=("transfer_id", "count"),
    num_careunit_transfers=("eventtype_clean", lambda x: int((x == "TRANSFER").sum())),
    had_ed_transfer_record=("eventtype_clean", lambda x: int((x == "ED").any())),
    had_unknown_transfer_careunit=("careunit_clean", lambda x: int(x.str.upper().eq("UNKNOWN").any()))).reset_index()

#summarise row-level records to admission-level features
unique_careunits = known_careunit.groupby(["subject_id", "hadm_id"])["careunit_clean"].nunique().reset_index()
unique_careunits = unique_careunits.rename(columns={"careunit_clean": "num_unique_careunits"})
transfer_features = transfer_features.merge(unique_careunits, on=["subject_id", "hadm_id"], how="left")
transfer_features["num_unique_careunits"] = transfer_features["num_unique_careunits"].fillna(0).astype(int)

print("\nTransfer features shape:", transfer_features.shape)
print(transfer_features.head())
print("\nTransfer event count distribution:")
print(transfer_features["num_transfer_events"].describe())
print("\nUnique careunit count distribution:")
print(transfer_features["num_unique_careunits"].describe())

#merge transfer features into main dataset
df = df.merge(transfer_features, on=["subject_id", "hadm_id"], how="left")

transfer_count_cols = ["num_transfer_events", "num_careunit_transfers", "num_unique_careunits",
    "had_ed_transfer_record", "had_unknown_transfer_careunit"]

#loop through each item in this feature block
for col in transfer_count_cols:
    df[col] = df[col].fillna(0).astype(int)

print("\nShape after merging transfer features:", df.shape)
print("\nMissing transfer feature values:")
print(df[transfer_count_cols].isna().sum())
print("\nTransfer feature preview:")
print(df[["subject_id", "hadm_id"] + transfer_count_cols].head())

Transfers shape: (2413581, 7)
Transfers columns:
['subject_id', 'hadm_id', 'transfer_id', 'eventtype', 'careunit', 'intime', 'outtime']
   subject_id     hadm_id  transfer_id  eventtype              careunit  \
0    10000032  22595853.0     33258284         ED  Emergency Department   
1    10000032  22595853.0     35223874      admit            Transplant   
2    10000032  22595853.0     36904543  discharge               UNKNOWN   
3    10000032  22841357.0     34100253  discharge               UNKNOWN   
4    10000032  22841357.0     34703856      admit            Transplant   

                intime              outtime  
0  2180-05-06 19:17:00  2180-05-06 23:30:00  
1  2180-05-06 23:30:00  2180-05-07 17:21:27  
2  2180-05-07 17:21:27                  NaN  
3  2180-06-27 18:49:12                  NaN  
4  2180-06-26 21:31:00  2180-06-27 18:49:12  

Transfers after restricting to cohort admissions: (893570, 7)

Transfer features shape: (238561, 7)
   subject_id   hadm_id  num_transfe

**Load and aggregate procedure burden features**

The `procedures_icd` table was used to create simple admission-level procedure burden features. Procedure records provide an additional proxy for medical intervention and acute clinical complexity, complementing diagnosis, DRG, ICU, laboratory, and physiological features.

In [15]:
#load icd procedure table
procedures = pd.read_csv(hosp_path / "procedures_icd.csv.gz")
print("Procedures shape:", procedures.shape)
print("Procedures columns:")
print(procedures.columns.tolist())
print(procedures.head())

#restrict procedures to admissions in current cohort
procedures = procedures.merge(df[["subject_id", "hadm_id"]], on=["subject_id", "hadm_id"], how="inner")
print("\nProcedures after restricting to cohort admissions:", procedures.shape)

#aggregate procedure burden to admission level
procedure_features = procedures.groupby(["subject_id", "hadm_id"]).agg(
    num_procedures=("icd_code", "count"),
    num_unique_procedure_codes=("icd_code", "nunique")).reset_index()

procedure_features["had_procedure"] = (procedure_features["num_procedures"] > 0).astype(int)

print("\nProcedure features shape:", procedure_features.shape)
print(procedure_features.head())
print("\nProcedure count distribution:")
print(procedure_features["num_procedures"].describe())

#merge procedure features into main dataset
df = df.merge(procedure_features, on=["subject_id", "hadm_id"], how="left")
procedure_count_cols = ["num_procedures", "num_unique_procedure_codes", "had_procedure"]

for col in procedure_count_cols:
    df[col] = df[col].fillna(0).astype(int)

print("\nShape after merging procedure features:", df.shape)
print("\nMissing procedure feature values:")
print(df[procedure_count_cols].isna().sum())
print("\nProcedure feature preview:")
print(df[["subject_id", "hadm_id"] + procedure_count_cols].head())

Procedures shape: (859655, 6)
Procedures columns:
['subject_id', 'hadm_id', 'seq_num', 'chartdate', 'icd_code', 'icd_version']
   subject_id   hadm_id  seq_num   chartdate icd_code  icd_version
0    10000032  22595853        1  2180-05-07     5491            9
1    10000032  22841357        1  2180-06-27     5491            9
2    10000032  25742920        1  2180-08-06     5491            9
3    10000068  25022803        1  2160-03-03     8938            9
4    10000117  27988844        1  2183-09-19  0QS734Z           10

Procedures after restricting to cohort admissions: (343139, 6)

Procedure features shape: (109193, 5)
   subject_id   hadm_id  num_procedures  num_unique_procedure_codes  \
0    10000032  22595853               1                           1   
1    10000032  22841357               1                           1   
2    10000032  25742920               1                           1   
3    10000068  25022803               1                           1   
4    10000117

**Load and aggregate DRG severity and mortality features**

Diagnosis Related Group (DRG) records were used to extract admission-level severity and mortality indicators. APR-DRG records include severity and mortality subclasses, which provide structured measures of illness severity beyond psychiatric diagnosis codes alone.

In [ ]:
#load drg codes table
drgcodes = pd.read_csv(hosp_path / "drgcodes.csv.gz")
print("DRG codes shape:", drgcodes.shape)
print("DRG codes columns:")
print(drgcodes.columns.tolist())
print(drgcodes.head())

#restrict to cohort admissions
drgcodes = drgcodes.merge(df[["subject_id", "hadm_id"]], on=["subject_id", "hadm_id"], how="inner")
print("\nDRG codes after restricting to cohort admissions:", drgcodes.shape)

#apr-drg rows contain severity and mortality subclasses
apr_drg = drgcodes[drgcodes["drg_type"] == "APR"].copy()

#aggregate drg severity and mortality to admission level
drg_features = apr_drg.groupby(["subject_id", "hadm_id"]).agg(drg_severity=("drg_severity", "max"),
    drg_mortality=("drg_mortality", "max"), drg_code_count=("drg_code", "count")).reset_index()

print("\nDRG features shape:", drg_features.shape)
print(drg_features.head())

print("\nDRG severity distribution:")
print(drg_features["drg_severity"].value_counts(dropna=False).sort_index())

print("\nDRG mortality distribution:")
print(drg_features["drg_mortality"].value_counts(dropna=False).sort_index())

#merge drg features into main dataset
df = df.merge(drg_features, on=["subject_id", "hadm_id"], how="left")

#missing drg values indicate no apr-drg severity/mortality record was available
df["drg_code_count"] = df["drg_code_count"].fillna(0).astype(int)
df["drg_severity"] = df["drg_severity"].fillna(0).astype(int)
df["drg_mortality"] = df["drg_mortality"].fillna(0).astype(int)
print("\nShape after merging DRG features:", df.shape)
print("\nMissing DRG feature values:")
print(df[["drg_severity", "drg_mortality", "drg_code_count"]].isna().sum())
print("\nDRG feature preview:")
print(df[["subject_id", "hadm_id", "drg_severity", "drg_mortality", "drg_code_count"]].head())

DRG codes shape: (761856, 7)
DRG codes columns:
['subject_id', 'hadm_id', 'drg_type', 'drg_code', 'description', 'drg_severity', 'drg_mortality']
   subject_id   hadm_id drg_type  drg_code  \
0    10000032  22595853      APR       283   
1    10000032  22595853     HCFA       442   
2    10000032  22841357      APR       279   
3    10000032  22841357     HCFA       442   
4    10000032  25742920      APR       283   

                                         description  drg_severity  \
0                       OTHER DISORDERS OF THE LIVER           2.0   
1  DISORDERS OF LIVER EXCEPT MALIG,CIRR,ALC HEPA ...           NaN   
2  HEPATIC COMA AND OTHER MAJOR ACUTE LIVER DISOR...           3.0   
3  DISORDERS OF LIVER EXCEPT MALIG,CIRR,ALC HEPA ...           NaN   
4                       OTHER DISORDERS OF THE LIVER           3.0   

   drg_mortality  
0            2.0  
1            NaN  
2            2.0  
3            NaN  
4            2.0  

DRG codes after restricting to cohort adm

**Load and aggregate diagnosis-level features**

Diagnosis records were aggregated to the admission level to create features describing diagnostic burden and psychiatric diagnostic subtype. These features provide information about the type and complexity of conditions recorded during each admission.

In [17]:
#load diagnosis table
diagnoses = pd.read_csv(hosp_path / "diagnoses_icd.csv.gz")
print("Diagnoses shape:", diagnoses.shape)
print("Diagnoses columns:")
print(diagnoses.columns.tolist())
print(diagnoses.head())

#keep only admissions in current cohort
diagnoses = diagnoses.merge(df[["subject_id", "hadm_id"]], on=["subject_id", "hadm_id"], how="inner")
print("\nDiagnoses after restricting to cohort admissions:", diagnoses.shape)

#convert icd codes to string for prefix matching
diagnoses["icd_code"] = diagnoses["icd_code"].astype(str).str.upper().str.replace(".", "", regex=False)

#identify psychiatric diagnosis codes - extract first 3 characters of each icd code
diagnoses["icd3"] = diagnoses["icd_code"].str[:3]

#convert numeric icd prefixes where possible. non-numeric codes become nan
diagnoses["icd3_numeric"] = pd.to_numeric(diagnoses["icd3"], errors="coerce")

#identify psychiatric diagnosis codes - icd-9 psychiatric codes are 290-319, icd-10 psychiatric codes begin with f
diagnoses["is_psych_diagnosis"] = (((diagnoses["icd_version"] == 9) & (diagnoses["icd3_numeric"].between(290, 319)))|
    ((diagnoses["icd_version"] == 10) & (diagnoses["icd_code"].str.startswith("F")))).astype(int)

print("ICD version counts:")
print(diagnoses["icd_version"].value_counts())
print("\nPsychiatric diagnosis counts:")
print(diagnoses["is_psych_diagnosis"].value_counts())
print("\nExample psychiatric ICD codes:")
print(diagnoses.loc[diagnoses["is_psych_diagnosis"] == 1, ["icd_version", "icd_code", "icd3", "icd3_numeric"]].head(20))

#define broad psychiatric diagnosis groups
def classify_psych_diagnosis(row):
    code = row["icd_code"]
    version = row["icd_version"]
    if version == 9:
        if code.startswith(("2962", "2963", "311", "3004")):
            return "depression"
        elif code.startswith(("2960", "2961", "2964", "2965", "2966", "2967", "2968")):
            return "bipolar"
        elif code.startswith(("295", "297", "298")):
            return "psychotic_disorder"
        elif code.startswith(("3000", "3002", "3003", "30981")):
            return "anxiety_ptsd"
        elif code.startswith(("303", "304", "305")):
            return "substance_use"
        elif code.startswith(("290", "293", "294")):
            return "cognitive_delirium"
        elif code.startswith("301"):
            return "personality_disorder"
        else:
            return "other_psych"

    if version == 10:
        if code.startswith(("F32", "F33", "F341", "F4321", "F4323")):
            return "depression"
        elif code.startswith(("F30", "F31", "F39")):
            return "bipolar"
        elif code.startswith(("F20", "F21", "F22", "F23", "F24", "F25", "F28", "F29")):
            return "psychotic_disorder"
        elif code.startswith(("F40", "F41", "F42", "F431", "F4320", "F4322", "F4324", "F4325", "F4329", "F438", "F439")):
            return "anxiety_ptsd"
        elif code.startswith(("F10", "F11", "F12", "F13", "F14", "F15", "F16", "F17", "F18", "F19")):
            return "substance_use"
        elif code.startswith(("F00", "F01", "F02", "F03", "F04", "F05", "F06", "F07", "F09")):
            return "cognitive_delirium"
        elif code.startswith(("F60", "F61", "F62", "F63", "F64", "F65", "F66", "F68", "F69")):
            return "personality_disorder"
        elif code.startswith("F50"):
            return "eating_disorder"
        elif code.startswith(("F80", "F81", "F82", "F83", "F84", "F88", "F89", "F90", "F91", "F92", "F93", "F94", "F95", "F98")):
            return "neurodevelopmental_disorder"
        elif code.startswith(("F43", "F44", "F45", "F48", "F51", "F52", "F53", "F54", "F55", "F59", "F70", "F71", "F72", "F73", "F78", "F79")):
            return "other_psych"
        else:
            return "other_psych"

diagnoses["psych_diagnosis_group"] = diagnoses.apply(classify_psych_diagnosis, axis=1)
#only keep psychiatric group flags for rows identified as psychiatric diagnoses
diagnoses.loc[diagnoses["is_psych_diagnosis"] == 0, "psych_diagnosis_group"] = "non_psych"
print("\nPsychiatric diagnosis group counts:")
print(diagnoses["psych_diagnosis_group"].value_counts())

#aggregate diagnosis counts to admission level
diagnosis_features = diagnoses.groupby(["subject_id", "hadm_id"]).agg(num_total_diagnoses=("icd_code", "count"),
    num_psych_diagnoses=("is_psych_diagnosis", "sum")).reset_index()

diagnosis_features["num_nonpsych_diagnoses"] = (diagnosis_features["num_total_diagnoses"] - diagnosis_features["num_psych_diagnoses"])

#create binary flags for broad psychiatric diagnosis groups
diagnosis_group_flags = pd.crosstab([diagnoses["subject_id"], diagnoses["hadm_id"]], diagnoses["psych_diagnosis_group"]).reset_index()
diagnosis_group_flags.columns.name = None

#rename diagnosis group columns
rename_cols = {"depression": "has_depression", "bipolar": "has_bipolar_disorder", "psychotic_disorder": "has_psychotic_disorder",
    "anxiety_ptsd": "has_anxiety_ptsd", "substance_use": "has_substance_use", "cognitive_delirium": "has_cognitive_delirium",
    "personality_disorder": "has_personality_disorder", "eating_disorder": "has_eating_disorder",
    "neurodevelopmental_disorder": "has_neurodevelopmental_disorder", "other_psych": "has_other_psych_diagnosis"}

diagnosis_group_flags = diagnosis_group_flags.rename(columns=rename_cols)
#loop through each item in this feature block
for col in ["non_psych", "other"]:
    if col in diagnosis_group_flags.columns:
        diagnosis_group_flags = diagnosis_group_flags.drop(columns=col)
        
#convert counts to binary indicators
diagnosis_flag_cols = [col for col in diagnosis_group_flags.columns if col.startswith("has_")]
for col in diagnosis_flag_cols:
    diagnosis_group_flags[col] = (diagnosis_group_flags[col] > 0).astype(int)

#merge diagnosis count features with diagnosis group flags
diagnosis_features = diagnosis_features.merge(diagnosis_group_flags, on=["subject_id", "hadm_id"], how="left")
print("\nDiagnosis features shape:", diagnosis_features.shape)
print(diagnosis_features.head())

#merge diagnosis features into main dataset
df = df.merge(diagnosis_features, on=["subject_id", "hadm_id"], how="left")

#fill diagnosis feature missing values with 0
diagnosis_cols = [col for col in diagnosis_features.columns if col not in ["subject_id", "hadm_id"]]
df[diagnosis_cols] = df[diagnosis_cols].fillna(0)

#convert count and flag columns to integer
for col in diagnosis_cols:
    df[col] = df[col].astype(int)

print("\nShape after merging diagnosis features:", df.shape)
print("Missing diagnosis feature values:")
print(df[diagnosis_cols].isna().sum())
print("\nDiagnosis feature preview:")
print(df[["subject_id", "hadm_id"] + diagnosis_cols].head())

Diagnoses shape: (6364488, 5)
Diagnoses columns:
['subject_id', 'hadm_id', 'seq_num', 'icd_code', 'icd_version']
   subject_id   hadm_id  seq_num icd_code  icd_version
0    10000032  22595853        1     5723            9
1    10000032  22595853        2    78959            9
2    10000032  22595853        3     5715            9
3    10000032  22595853        4    07070            9
4    10000032  22595853        5      496            9

Diagnoses after restricting to cohort admissions: (3155015, 5)
ICD version counts:
icd_version
10    1800895
9     1354120
Name: count, dtype: int64

Psychiatric diagnosis counts:
is_psych_diagnosis
0    2731580
1     423435
Name: count, dtype: int64

Example psychiatric ICD codes:
     icd_version icd_code icd3  icd3_numeric
5              9    29680  296         296.0
6              9    30981  309         309.0
15             9     3051  305         305.0
24             9     3051  305         305.0
32             9     3051  305         305.0
37 

**Load and aggregate medication exposure features**

Medication records were used to create admission-level indicators of psychiatric medication exposure. Individual medication names were grouped into broader therapeutic classes to reduce sparsity and improve interpretability.

In [18]:
#load prescriptions table
prescriptions = pd.read_csv(hosp_path / "prescriptions.csv.gz", usecols=["subject_id", "hadm_id", "starttime", "stoptime", "drug"])
print("Prescriptions shape:", prescriptions.shape)
print("Prescriptions columns:")
print(prescriptions.columns.tolist())
print(prescriptions.head())

#restrict prescriptions to admissions in the cohort
prescriptions = prescriptions.merge(df[["subject_id", "hadm_id", "admittime", "dischtime"]], on=["subject_id", "hadm_id"], how="inner")
print("\nPrescriptions after restricting to cohort admissions:", prescriptions.shape)

#convert dates to datetime
prescriptions["starttime"] = pd.to_datetime(prescriptions["starttime"], errors="coerce")
prescriptions["stoptime"] = pd.to_datetime(prescriptions["stoptime"], errors="coerce")
prescriptions["admittime"] = pd.to_datetime(prescriptions["admittime"], errors="coerce")
prescriptions["dischtime"] = pd.to_datetime(prescriptions["dischtime"], errors="coerce")

#keep medication orders that started during the index admission
prescriptions = prescriptions[(prescriptions["starttime"].isna()) |((prescriptions["starttime"] >= prescriptions["admittime"]) &
     (prescriptions["starttime"] <= prescriptions["dischtime"]))].copy()
print("\nPrescriptions after admission-time filtering:", prescriptions.shape)

#standardise drug names and define med class keywords lists
prescriptions["drug_clean"] = prescriptions["drug"].astype(str).str.lower()
antidepressants = ["sertraline", "fluoxetine", "citalopram", "escitalopram", "paroxetine", "venlafaxine", 
                   "duloxetine", "mirtazapine", "trazodone", "bupropion", "amitriptyline", "nortriptyline",
                   "doxepin", "imipramine", "clomipramine", "desipramine", "phenelzine"]

antipsychotics = ["haloperidol", "risperidone", "olanzapine", "quetiapine", "clozapine", "aripiprazole", 
                  "ziprasidone", "chlorpromazine", "fluphenazine", "lurasidone", "paliperidone",
                  "perphenazine", "thioridazine", "trifluoperazine", "prochlorperazine"]
mood_stabilisers = ["lithium", "valproate", "valproic acid", "divalproex", "lamotrigine", "carbamazepine",
                    "oxcarbazepine"]
benzodiazepines = ["lorazepam", "diazepam", "clonazepam", "alprazolam", "midazolam", "temazepam", 
                   "chlordiazepoxide", "oxazepam", "triazolam"]
stimulants = ["methylphenidate", "amphetamine", "lisdexamfetamine", "modafinil", "dextroamphetamine"]
sedative_hypnotics = ["zolpidem", "zopiclone", "eszopiclone", "melatonin", "ramelteon", "suvorexant"]
long_acting_injectable_antipsychotics = ["decanoate", "paliperidone palmitate", "invega sustenna", "invega trinza", "invega hafyera", "risperdal consta",
    "risperidone microspheres", "risperidone long", "aripiprazole lauroxil", "abilify maintena",
    "haloperidol decanoate", "fluphenazine decanoate"]

#keyword matching function
def contains_any_keyword(series, keywords):
    pattern = "|".join([re.escape(keyword) for keyword in keywords])
    return series.str.contains(pattern, case=False, na=False).astype(int)

#create medication class indicators
prescriptions["had_antidepressant"] = contains_any_keyword(prescriptions["drug_clean"], antidepressants)
prescriptions["had_antipsychotic"] = contains_any_keyword(prescriptions["drug_clean"], antipsychotics)
prescriptions["had_mood_stabiliser"] = contains_any_keyword(prescriptions["drug_clean"], mood_stabilisers)
prescriptions["had_benzodiazepine"] = contains_any_keyword(prescriptions["drug_clean"], benzodiazepines)
prescriptions["had_stimulant"] = contains_any_keyword(prescriptions["drug_clean"], stimulants)
prescriptions["had_sedative_hypnotic"] = contains_any_keyword(prescriptions["drug_clean"], sedative_hypnotics)
prescriptions["had_long_acting_injectable_antipsychotic"] = contains_any_keyword(
    prescriptions["drug_clean"], long_acting_injectable_antipsychotics)
prescriptions["antipsychotic_drug_name"] = prescriptions["drug_clean"].where(prescriptions["had_antipsychotic"] == 1)

med_class_cols = ["had_antidepressant", "had_antipsychotic", "had_mood_stabiliser", 
                  "had_benzodiazepine", "had_stimulant", "had_sedative_hypnotic"]

print("\nMedication class exposure counts at prescription-row level:")
print(prescriptions[med_class_cols].sum())

#aggregate medication classes to admission level
medication_features = prescriptions.groupby(["subject_id", "hadm_id"]).agg(num_prescription_rows=("drug", "count"),
    num_unique_drugs=("drug_clean", "nunique"), had_antidepressant=("had_antidepressant", "max"),
    had_antipsychotic=("had_antipsychotic", "max"), had_mood_stabiliser=("had_mood_stabiliser", "max"),
    had_benzodiazepine=("had_benzodiazepine", "max"), had_stimulant=("had_stimulant", "max"),
    had_sedative_hypnotic=("had_sedative_hypnotic", "max"),
    antipsychotic_prescription_count=("had_antipsychotic", "sum"),
    num_unique_antipsychotics=("antipsychotic_drug_name", "nunique"),
    had_long_acting_injectable_antipsychotic=("had_long_acting_injectable_antipsychotic", "max")).reset_index()

#count number of psychiatric medication classes received during admission
medication_features["num_psych_med_classes"] = medication_features[med_class_cols].sum(axis=1)
medication_features["antipsychotic_polypharmacy_2plus"] = (medication_features["num_unique_antipsychotics"] >= 2).astype(int)
medication_features["antipsychotic_polypharmacy_3plus"] = (medication_features["num_unique_antipsychotics"] >= 3).astype(int)
medication_features["antipsychotic_plus_benzodiazepine"] = ((medication_features["had_antipsychotic"] == 1) &
    (medication_features["had_benzodiazepine"] == 1)).astype(int)
medication_features["antipsychotic_plus_mood_stabiliser"] = ((medication_features["had_antipsychotic"] == 1) &
    (medication_features["had_mood_stabiliser"] == 1)).astype(int)
print("\nMedication features shape:", medication_features.shape)
print(medication_features.head())

#remove previous medication features if this cell has already been run in the same session
medication_cols_to_drop = [col for col in df.columns if (col.startswith("num_prescription_rows") or
    col.startswith("num_unique_drugs") or col.startswith("had_antidepressant") or
    col.startswith("had_antipsychotic") or col.startswith("had_mood_stabiliser") or
    col.startswith("had_benzodiazepine") or col.startswith("had_stimulant") or
    col.startswith("had_sedative_hypnotic") or col.startswith("num_psych_med_classes") or
    col.startswith("antipsychotic_prescription_count") or col.startswith("num_unique_antipsychotics") or
    col.startswith("antipsychotic_polypharmacy") or col.startswith("antipsychotic_plus_") or
    col.startswith("had_long_acting_injectable_antipsychotic"))]
df = df.drop(columns=medication_cols_to_drop, errors="ignore")

#merge medication features into main dataset
df = df.merge(medication_features, on=["subject_id", "hadm_id"], how="left")
medication_cols = [col for col in medication_features.columns if col not in ["subject_id", "hadm_id"]]
df[medication_cols] = df[medication_cols].fillna(0)

#convert medication features to integer
for col in medication_cols:
    df[col] = df[col].astype(int)

print("\nShape after merging medication features:", df.shape)
print("Missing medication feature values:")
print(df[medication_cols].isna().sum())
print("\nMedication feature preview:")
print(df[["subject_id", "hadm_id"] + medication_cols].head())

Prescriptions shape: (20292611, 5)
Prescriptions columns:
['subject_id', 'hadm_id', 'starttime', 'stoptime', 'drug']
   subject_id   hadm_id            starttime             stoptime  \
0    10000032  22595853  2180-05-08 08:00:00  2180-05-07 22:00:00   
1    10000032  22595853  2180-05-07 02:00:00  2180-05-07 22:00:00   
2    10000032  22595853  2180-05-07 01:00:00  2180-05-07 09:00:00   
3    10000032  22595853  2180-05-07 01:00:00  2180-05-07 01:00:00   
4    10000032  22595853  2180-05-07 00:00:00  2180-05-07 22:00:00   

                          drug  
0                   Furosemide  
1      Ipratropium Bromide Neb  
2                   Furosemide  
3           Potassium Chloride  
4  Sodium Chloride 0.9%  Flush  

Prescriptions after restricting to cohort admissions: (9589549, 7)

Prescriptions after admission-time filtering: (9339354, 7)

Medication class exposure counts at prescription-row level:
had_antidepressant       174286
had_antipsychotic        184870
had_mood_stabilis

**Load and aggregate selected laboratory features**

Selected laboratory measurements were extracted to represent physiological status during the index admission. To keep the feature set interpretable, a small group of commonly measured laboratory tests was selected and aggregated at the admission level.

In [19]:
#load laboratory item dictionary
lab_items = pd.read_csv(hosp_path / "d_labitems.csv.gz")
print("Lab item dictionary shape:", lab_items.shape)
print(lab_items.head())

#select specific common blood-based laboratory measurements, exact matching is used to avoid pulling in related but different tests
selected_lab_rules = [{"label": "Sodium", "category": "Chemistry", "lab_short_name": "sodium"},
    {"label": "Potassium", "category": "Chemistry", "lab_short_name": "potassium"},
    {"label": "Creatinine", "category": "Chemistry", "lab_short_name": "creatinine"},
    {"label": "Urea Nitrogen", "category": "Chemistry", "lab_short_name": "urea_nitrogen"},
    {"label": "Glucose", "category": "Chemistry", "lab_short_name": "glucose"},
    {"label": "Hemoglobin", "category": "Hematology", "lab_short_name": "hemoglobin"},
    {"label": "White Blood Cells", "category": "Hematology", "lab_short_name": "wbc"},
    {"label": "Platelet Count", "category": "Hematology", "lab_short_name": "platelet"}]

selected_lab_itemids = []
for rule in selected_lab_rules:
    matches = lab_items[(lab_items["label"].str.lower() == rule["label"].lower()) &
        (lab_items["fluid"].str.lower() == "blood") &
        (lab_items["category"].str.lower() == rule["category"].lower())].copy()
    matches["lab_short_name"] = rule["lab_short_name"]
    selected_lab_itemids.append(matches)
selected_lab_itemids = pd.concat(selected_lab_itemids, ignore_index=True)
selected_lab_itemids = selected_lab_itemids.drop_duplicates(subset=["itemid"])

print("\nSelected lab itemids:")
print(selected_lab_itemids[["itemid", "label", "fluid", "category", "lab_short_name"]])
print("\nSelected lab counts by short name:")
print(selected_lab_itemids["lab_short_name"].value_counts())
itemid_to_lab = selected_lab_itemids.set_index("itemid")["lab_short_name"].to_dict()
selected_itemids = set(itemid_to_lab.keys())
print("\nNumber of selected itemids:", len(selected_itemids))

Lab item dictionary shape: (1650, 4)
   itemid                                label  fluid   category
0   50801           Alveolar-arterial Gradient  Blood  Blood Gas
1   50802                          Base Excess  Blood  Blood Gas
2   50803  Calculated Bicarbonate, Whole Blood  Blood  Blood Gas
3   50804                 Calculated Total CO2  Blood  Blood Gas
4   50805                    Carboxyhemoglobin  Blood  Blood Gas

Selected lab itemids:
    itemid              label  fluid    category lab_short_name
0    50983             Sodium  Blood   Chemistry         sodium
1    52623             Sodium  Blood   Chemistry         sodium
2    50971          Potassium  Blood   Chemistry      potassium
3    52610          Potassium  Blood   Chemistry      potassium
4    50912         Creatinine  Blood   Chemistry     creatinine
5    52546         Creatinine  Blood   Chemistry     creatinine
6    51006      Urea Nitrogen  Blood   Chemistry  urea_nitrogen
7    52647      Urea Nitrogen  Blood  

**Create admission-level laboratory features**

Laboratory results are recorded multiple times during a hospital admission. Since machine learning models require a single row per admission, laboratory measurements were aggregated to admission level.
For each selected laboratory test (sodium, creatinine, glucose, haemoglobin, white blood cell count, and platelet count), four features were generated:

- lab_first_value: first recorded value during the admission
- lab_mean_value: mean value across the admission
- lab_measured: indicator showing whether the laboratory test was measured
- lab_abnormal: indicator showing whether any recorded value was outside the reference range or flagged as abnormal

The resulting features were pivoted into a wide format and merged into the modelling dataset, producing one set of laboratory features per admission.

In [ ]:
#extract selected labevents in chunks because labevents is very large
cohort_ids = df[["subject_id", "hadm_id", "admittime", "dischtime"]].copy()
cohort_ids["admittime"] = pd.to_datetime(cohort_ids["admittime"])
cohort_ids["dischtime"] = pd.to_datetime(cohort_ids["dischtime"])
cohort_hadm_ids = set(cohort_ids["hadm_id"].dropna().astype(int))
lab_chunks = []
chunk_size = 2_000_000
usecols = ["subject_id", "hadm_id", "itemid", "charttime", "valuenum", "ref_range_lower", "ref_range_upper", "flag"]
print("Starting chunked labevents extraction...")

for chunk_number, chunk in enumerate(pd.read_csv(hosp_path / "labevents.csv.gz",
        usecols=usecols, chunksize=chunk_size)):

    #keep selected admissions and selected lab itemids only
    chunk = chunk[chunk["hadm_id"].isin(cohort_hadm_ids) & chunk["itemid"].isin(selected_itemids)].copy()
    if len(chunk) == 0:
        continue

    chunk["lab_name"] = chunk["itemid"].map(itemid_to_lab) #map lab names

    #merge admission times for leakage-safe filtering and keep labs recorded during index admission only
    chunk = chunk.merge(cohort_ids, on=["subject_id", "hadm_id"], how="left")
    chunk["charttime"] = pd.to_datetime(chunk["charttime"], errors="coerce")
    chunk = chunk[(chunk["charttime"] >= chunk["admittime"]) & (chunk["charttime"] <= chunk["dischtime"])].copy()

    if len(chunk) > 0:
        lab_chunks.append(chunk)

    print("Processed chunk:", chunk_number, "Rows kept:", len(chunk))

if len(lab_chunks) > 0:
    selected_labs = pd.concat(lab_chunks, ignore_index=True)
else:
    selected_labs = pd.DataFrame(columns=usecols + ["lab_name", "admittime", "dischtime"])

print("\nSelected labs shape:", selected_labs.shape)
print(selected_labs.head())

Starting chunked labevents extraction...
Processed chunk: 0 Rows kept: 126966
Processed chunk: 1 Rows kept: 114986
Processed chunk: 2 Rows kept: 112717
Processed chunk: 3 Rows kept: 118070
Processed chunk: 4 Rows kept: 127219
Processed chunk: 5 Rows kept: 115359
Processed chunk: 6 Rows kept: 134405
Processed chunk: 7 Rows kept: 114713
Processed chunk: 8 Rows kept: 119216
Processed chunk: 9 Rows kept: 124389
Processed chunk: 10 Rows kept: 132516
Processed chunk: 11 Rows kept: 122386
Processed chunk: 12 Rows kept: 122005
Processed chunk: 13 Rows kept: 112083
Processed chunk: 14 Rows kept: 108523
Processed chunk: 15 Rows kept: 122497
Processed chunk: 16 Rows kept: 128895
Processed chunk: 17 Rows kept: 118505
Processed chunk: 18 Rows kept: 111691
Processed chunk: 19 Rows kept: 123891
Processed chunk: 20 Rows kept: 117297
Processed chunk: 21 Rows kept: 110226
Processed chunk: 22 Rows kept: 120384
Processed chunk: 23 Rows kept: 118917
Processed chunk: 24 Rows kept: 125718
Processed chunk: 25

**Aggregate and merge laboratory features**

The extracted laboratory records are converted from repeated measurement-level rows into admission-level features. Each laboratory test may be measured multiple times during a hospital admission, so records are first sorted chronologically and then summarised for each subject_id and hadm_id. For each selected laboratory test, the first recorded value, mean value, measurement indicator, and abnormality indicator are created. The resulting long-format laboratory table is then pivoted into a wide-format dataset so that each admission has one row with separate laboratory feature columns. These table-derived columns are then merged into the main modelling dataset.

In [ ]:
#create abnormal and severe abnormal lab indicators
selected_labs = selected_labs.sort_values(["subject_id", "hadm_id", "lab_name", "charttime"])
selected_labs["lab_abnormal"] = 0
selected_labs["lab_severe_abnormal"] = 0

#use the flag column where available
selected_labs.loc[selected_labs["flag"].astype(str).str.lower().isin(["abnormal"]), "lab_abnormal"] = 1

#also use reference ranges where available
selected_labs.loc[selected_labs["valuenum"].notna() & selected_labs["ref_range_lower"].notna() &
    (selected_labs["valuenum"] < selected_labs["ref_range_lower"]), "lab_abnormal"] = 1

selected_labs.loc[selected_labs["valuenum"].notna() & selected_labs["ref_range_upper"].notna() &
    (selected_labs["valuenum"] > selected_labs["ref_range_upper"]), "lab_abnormal"] = 1

#mark extreme departures from the reference range as severe abnormality where reference ranges are available
selected_labs.loc[selected_labs["valuenum"].notna() & selected_labs["ref_range_lower"].notna() &
    (selected_labs["ref_range_lower"] > 0) &
    (selected_labs["valuenum"] < (0.5 * selected_labs["ref_range_lower"])), "lab_severe_abnormal"] = 1

selected_labs.loc[selected_labs["valuenum"].notna() & selected_labs["ref_range_upper"].notna() &
    (selected_labs["ref_range_upper"] > 0) &
    (selected_labs["valuenum"] > (1.5 * selected_labs["ref_range_upper"])), "lab_severe_abnormal"] = 1

#calculate time-window indicators for early and near-discharge biochemical summaries
selected_labs["hours_since_admission"] = ((selected_labs["charttime"] - selected_labs["admittime"]).dt.total_seconds() / 3600)
selected_labs["hours_to_discharge"] = ((selected_labs["dischtime"] - selected_labs["charttime"]).dt.total_seconds() / 3600)
selected_labs["lab_value_first_24h"] = selected_labs["valuenum"].where(selected_labs["hours_since_admission"].between(0, 24, inclusive="both"))
selected_labs["lab_value_first_72h"] = selected_labs["valuenum"].where(selected_labs["hours_since_admission"].between(0, 72, inclusive="both"))
selected_labs["lab_event_last_24h_before_discharge"] = selected_labs["hours_to_discharge"].between(0, 24, inclusive="both").astype(int)
selected_labs["lab_event_last_48h_before_discharge"] = selected_labs["hours_to_discharge"].between(0, 48, inclusive="both").astype(int)
selected_labs["abnormal_lab_last_48h"] = ((selected_labs["lab_abnormal"] == 1) &
    (selected_labs["lab_event_last_48h_before_discharge"] == 1)).astype(int)

#aggregate lab features to admission level
lab_value_features = selected_labs.groupby(["subject_id", "hadm_id", "lab_name"]).agg(
    lab_first_value=("valuenum", "first"),
    lab_last_value=("valuenum", "last"),
    lab_mean_value=("valuenum", "mean"),
    lab_min_value=("valuenum", "min"),
    lab_max_value=("valuenum", "max"),
    lab_std_value=("valuenum", "std"),
    lab_count=("valuenum", "count"),
    lab_first_24h_mean=("lab_value_first_24h", "mean"),
    lab_first_24h_min=("lab_value_first_24h", "min"),
    lab_first_24h_max=("lab_value_first_24h", "max"),
    lab_first_24h_count=("lab_value_first_24h", "count"),
    lab_first_72h_mean=("lab_value_first_72h", "mean"),
    lab_first_72h_min=("lab_value_first_72h", "min"),
    lab_first_72h_max=("lab_value_first_72h", "max"),
    lab_first_72h_count=("lab_value_first_72h", "count"),
    lab_measured=("valuenum", lambda x: int(x.notna().any())),
    lab_abnormal=("lab_abnormal", "max"),
    lab_severe_abnormal=("lab_severe_abnormal", "max")).reset_index()

#calculate simple biochemical change and spread across admission
lab_value_features["lab_value_change"] = (
    lab_value_features["lab_last_value"] - lab_value_features["lab_first_value"])
lab_value_features["lab_value_range"] = (
    lab_value_features["lab_max_value"] - lab_value_features["lab_min_value"])

#summarise row-level records to admission-level features
lab_activity_features = selected_labs.groupby(["subject_id", "hadm_id"], as_index=False).agg(
    num_lab_events_first_24h=("lab_value_first_24h", "count"),
    num_lab_events_first_72h=("lab_value_first_72h", "count"),
    num_lab_events_last_24h_before_discharge=("lab_event_last_24h_before_discharge", "sum"),
    num_lab_events_last_48h_before_discharge=("lab_event_last_48h_before_discharge", "sum"),
    num_abnormal_lab_flags=("lab_abnormal", "sum"),
    num_abnormal_lab_flags_last_48h=("abnormal_lab_last_48h", "sum"),
    num_critical_or_priority_labs=("lab_severe_abnormal", "sum"))
lab_activity_features["lab_activity_last_24h_flag"] = (lab_activity_features["num_lab_events_last_24h_before_discharge"] > 0).astype(int)
lab_activity_features["lab_activity_near_discharge_flag"] = (lab_activity_features["num_lab_events_last_48h_before_discharge"] > 0).astype(int)
lab_activity_features["abnormal_lab_burden_score"] = (lab_activity_features["num_abnormal_lab_flags"] +
    (2 * lab_activity_features["num_critical_or_priority_labs"]))

print("Long lab table-derived feature table shape:")
print(lab_value_features.shape)
print(lab_value_features.head())

#pivot lab features to wide format
lab_wide = lab_value_features.pivot_table(index=["subject_id", "hadm_id"], columns="lab_name",
    values=["lab_first_value", "lab_last_value", "lab_mean_value", "lab_min_value", "lab_max_value",
        "lab_std_value", "lab_count", "lab_value_change", "lab_value_range", "lab_first_24h_mean",
        "lab_first_24h_min", "lab_first_24h_max", "lab_first_24h_count", "lab_first_72h_mean",
        "lab_first_72h_min", "lab_first_72h_max", "lab_first_72h_count", "lab_measured",
        "lab_abnormal", "lab_severe_abnormal"], aggfunc="first")

#flatten multi-index columns
lab_wide.columns = [f"{lab}_{feature}" for feature, lab in lab_wide.columns]
lab_wide = lab_wide.reset_index()
print("\nWide lab features shape:")
print(lab_wide.shape)
print(lab_wide.head())

#remove previous laboratory features if this cell has already been run in the same session
lab_suffixes = ("_lab_first_value", "_lab_last_value", "_lab_mean_value", "_lab_min_value", "_lab_max_value",
    "_lab_std_value", "_lab_count", "_lab_value_change", "_lab_value_range", "_lab_first_24h_mean",
    "_lab_first_24h_min", "_lab_first_24h_max", "_lab_first_24h_count", "_lab_first_72h_mean",
    "_lab_first_72h_min", "_lab_first_72h_max", "_lab_first_72h_count", "_lab_measured",
    "_lab_abnormal", "_lab_severe_abnormal")
existing_lab_cols = [col for col in df.columns if col.endswith(lab_suffixes)]
df = df.drop(columns=existing_lab_cols, errors="ignore")

#merge lab features into main dataset
df = df.merge(lab_wide, on=["subject_id", "hadm_id"], how="left")
#join the derived features back to the admission table
df = df.merge(lab_activity_features, on=["subject_id", "hadm_id"], how="left")
lab_feature_cols = [col for col in lab_wide.columns if col not in ["subject_id", "hadm_id"]]
lab_activity_cols = [col for col in lab_activity_features.columns if col not in ["subject_id", "hadm_id"]]

#fill measured, count, and abnormal flags with 0
for col in lab_feature_cols:
    if col.endswith(("_lab_measured", "_lab_abnormal", "_lab_severe_abnormal", "_lab_count",
            "_lab_first_24h_count", "_lab_first_72h_count")):
        df[col] = df[col].fillna(0).astype(int)
#loop through each item in this feature block
for col in lab_activity_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    if col != "abnormal_lab_burden_score":
        df[col] = df[col].astype(int)

#leave raw lab values as missing for now; these can be imputed during preprocessing
print("\nShape after merging laboratory dynamic features:", df.shape)
print("\nMissing laboratory feature values:")
print(df[lab_feature_cols + lab_activity_cols].isna().sum().sort_values(ascending=False))
print("\nLaboratory feature preview:")
print(df[["subject_id", "hadm_id"] + lab_feature_cols + lab_activity_cols].head())

Long lab table-derived feature table shape:
(1390257, 23)
   subject_id     hadm_id    lab_name  lab_first_value  lab_last_value  \
0    10000032  22595853.0  creatinine              0.3             0.3   
1    10000032  22595853.0     glucose             99.0            99.0   
2    10000032  22595853.0  hemoglobin             12.7            12.7   
3    10000032  22595853.0    platelet             71.0            71.0   
4    10000032  22595853.0   potassium              4.5             4.5   

   lab_mean_value  lab_min_value  lab_max_value  lab_std_value  lab_count  \
0             0.3            0.3            0.3            NaN          1   
1            99.0           99.0           99.0            NaN          1   
2            12.7           12.7           12.7            NaN          1   
3            71.0           71.0           71.0            NaN          1   
4             4.5            4.5            4.5            NaN          1   

   ...  lab_first_24h_count  lab_f

**Load and aggregate selected ICU vital sign features**

Selected physiological measurements were extracted from the ICU `chartevents` table to capture patient state during the index admission. Heart rate, respiratory rate, oxygen saturation, systolic blood pressure, and mean blood pressure were selected because they describe acute physiological status and are commonly used in ICU deterioration and readmission prediction studies. Since `chartevents` is an ICU table, these table-derived columns are expected to be available mainly for admissions involving ICU charting.

In [22]:
#load icu item dictionary
d_items = pd.read_csv(icu_path / "d_items.csv.gz")

#selected icu vital sign itemids from mimic-iv d_items
selected_vital_items = {220045: "heart_rate", 220210: "respiratory_rate", 220277: "spo2",
    220050: "systolic_bp_arterial", 220179: "systolic_bp_noninvasive",
    220052: "mean_bp_arterial", 220181: "mean_bp_noninvasive"}

selected_vital_itemids = set(selected_vital_items.keys())

print("Selected vital sign itemids:")
print(d_items[d_items["itemid"].isin(selected_vital_itemids)]
[["itemid", "label", "category", "unitname"]])


Selected vital sign itemids:
    itemid                                 label             category  \
2   220045                            Heart Rate  Routine Vital Signs   
6   220050      Arterial Blood Pressure systolic  Routine Vital Signs   
8   220052          Arterial Blood Pressure mean  Routine Vital Signs   
24  220179  Non Invasive Blood Pressure systolic  Routine Vital Signs   
26  220181      Non Invasive Blood Pressure mean  Routine Vital Signs   
28  220210                      Respiratory Rate          Respiratory   
36  220277           O2 saturation pulseoxymetry          Respiratory   

    unitname  
2        bpm  
6       mmHg  
8       mmHg  
24      mmHg  
26      mmHg  
28  insp/min  
36         %  


In [ ]:
#extract selected vital signs in chunks because chartevents is very large
cohort_ids = df[["subject_id", "hadm_id", "admittime", "dischtime"]].copy()
cohort_ids["admittime"] = pd.to_datetime(cohort_ids["admittime"])
cohort_ids["dischtime"] = pd.to_datetime(cohort_ids["dischtime"])
cohort_hadm_ids = set(cohort_ids["hadm_id"].dropna().astype(int))
vital_chunks = []
chunk_size = 2_000_000

usecols = ["subject_id", "hadm_id", "stay_id", "charttime", "itemid", "valuenum"]
print("Starting chunked chartevents vital sign extraction...")

for chunk_number, chunk in enumerate(pd.read_csv(
icu_path / "chartevents.csv.gz", usecols=usecols, chunksize=chunk_size)):

    #keep only cohort admissions and selected vital sign itemids
    chunk = chunk[chunk["hadm_id"].isin(cohort_hadm_ids)
        & chunk["itemid"].isin(selected_vital_itemids)].copy()

    if len(chunk) == 0:
        continue

    #map itemids to readable vital sign names
    chunk["vital_name"] = chunk["itemid"].map(selected_vital_items)

    #keep numeric vital sign values only
    chunk["valuenum"] = pd.to_numeric(chunk["valuenum"], errors="coerce")
    chunk = chunk[chunk["valuenum"].notna()].copy()

    #merge admission times so only observations during the index admission are retained
    chunk = chunk.merge(cohort_ids, on=["subject_id", "hadm_id"], how="left")
    chunk["charttime"] = pd.to_datetime(chunk["charttime"], errors="coerce")

    chunk = chunk[(chunk["charttime"] >= chunk["admittime"])
        & (chunk["charttime"] <= chunk["dischtime"])].copy()

    if len(chunk) > 0:
        vital_chunks.append(chunk)

    print("Processed chunk:", chunk_number, "Rows kept:", len(chunk))

if len(vital_chunks) > 0:
    selected_vitals = pd.concat(vital_chunks, ignore_index=True)
else:
    selected_vitals = pd.DataFrame(columns=usecols + ["vital_name", "admittime", "dischtime"])

print("Selected vitals shape:", selected_vitals.shape)
print(selected_vitals.head())

#save filtered vital sign time series for possible later sequential modelling table or artefact for later review
vital_timeseries_file = output_path / "t2_1_selected_vital_sign_timeseries.csv"
selected_vitals.to_csv(vital_timeseries_file, index=False)
print("Saved selected vital sign time series to:")
print(vital_timeseries_file)

Starting chunked chartevents vital sign extraction...
Processed chunk: 0 Rows kept: 112602
Processed chunk: 1 Rows kept: 91792
Processed chunk: 2 Rows kept: 93426
Processed chunk: 3 Rows kept: 100375
Processed chunk: 4 Rows kept: 101132
Processed chunk: 5 Rows kept: 105389
Processed chunk: 6 Rows kept: 100748
Processed chunk: 7 Rows kept: 100655
Processed chunk: 8 Rows kept: 98444
Processed chunk: 9 Rows kept: 117107
Processed chunk: 10 Rows kept: 95857
Processed chunk: 11 Rows kept: 103925
Processed chunk: 12 Rows kept: 113749
Processed chunk: 13 Rows kept: 98931
Processed chunk: 14 Rows kept: 100404
Processed chunk: 15 Rows kept: 98678
Processed chunk: 16 Rows kept: 75087
Processed chunk: 17 Rows kept: 107770
Processed chunk: 18 Rows kept: 108710
Processed chunk: 19 Rows kept: 116097
Processed chunk: 20 Rows kept: 86461
Processed chunk: 21 Rows kept: 92509
Processed chunk: 22 Rows kept: 112102
Processed chunk: 23 Rows kept: 96194
Processed chunk: 24 Rows kept: 109450
Processed chunk:

**Add selected ICU chart-event acuity features**

This block extracts a deliberately small set of non-vital ICU chart concepts that can reflect acute medical or behavioural complexity. It looks for oxygen support, consciousness, agitation/sedation, pain, delirium, mobility/fall-risk, and restraint-related charting.

These features are kept in T2.1 because they come directly from raw ICU event rows and need timestamp/item-level filtering before they can safely become one admission-level row. The aim is not to add every charted variable, but to capture compact acuity markers that may help older or medically complex admissions.


In [ ]:
#extract selected non-vital icu chart-event domains without adding raw item ids
#these features target older/medical-complexity false negatives and icu admissions where standard vitals may be too limited.
print("Starting selected ICU chart-event acuity extraction...")

selected_chart_event_feature_cols = ["num_oxygen_support_chart_events", "max_oxygen_flow_rate", "mean_oxygen_flow_rate",
    "high_oxygen_flow_flag", "num_gcs_chart_events", "gcs_min_score", "low_gcs_flag",
    "num_orientation_chart_events", "impaired_orientation_flag", "num_pain_chart_events",
    "pain_max_score", "high_pain_flag", "num_rass_sedation_chart_events", "rass_extreme_flag",
    "num_cam_delirium_chart_events", "delirium_or_cam_positive_flag", "agitation_chart_event_flag",
    "num_restraint_chart_events", "had_restraint_chart_event", "num_fall_mobility_chart_events",
    "fall_or_mobility_risk_flag", "num_selected_neuro_behavioral_chart_events",
    "num_selected_resp_support_chart_events", "chart_acuity_event_density_per_day"]

chart_item_dictionary = d_items.copy()
chart_item_dictionary["label_lower"] = chart_item_dictionary["label"].fillna("").astype(str).str.lower()
chart_item_dictionary["category_lower"] = chart_item_dictionary["category"].fillna("").astype(str).str.lower()

oxygen_support_itemids = set(chart_item_dictionary.loc[
    chart_item_dictionary["category_lower"].eq("respiratory") &
    chart_item_dictionary["label_lower"].str.contains("o2 flow|inspired o2 fraction|ventilator mode|ventilator type|assistance device|flow rate", regex=True, na=False),
    "itemid"].astype(int))
gcs_itemids = set(chart_item_dictionary.loc[
    chart_item_dictionary["label_lower"].str.contains("gcs -|gcseye|gcsmotor|gcsverbal|gcsscore|glasgow", regex=True, na=False),
    "itemid"].astype(int))
orientation_itemids = set(chart_item_dictionary.loc[
    chart_item_dictionary["label_lower"].str.contains("orientation|level of consciousness|mental status", regex=True, na=False),
    "itemid"].astype(int))
pain_itemids = set(chart_item_dictionary.loc[
    chart_item_dictionary["category_lower"].eq("pain/sedation") &
    chart_item_dictionary["label_lower"].str.contains("pain level|pain present|pain assessment", regex=True, na=False),
    "itemid"].astype(int))
rass_sedation_itemids = set(chart_item_dictionary.loc[
    chart_item_dictionary["label_lower"].str.contains("richmond|rass|riker-sas|sedation", regex=True, na=False),
    "itemid"].astype(int))
cam_delirium_itemids = set(chart_item_dictionary.loc[
    chart_item_dictionary["label_lower"].str.contains("cam-icu|delirium", regex=True, na=False),
    "itemid"].astype(int))
agitation_itemids = set(chart_item_dictionary.loc[
    chart_item_dictionary["label_lower"].str.contains("agitation", regex=True, na=False),
    "itemid"].astype(int))
restraint_chart_itemids = set(chart_item_dictionary.loc[
    chart_item_dictionary["category_lower"].str.contains("restraint", regex=True, na=False) |
    chart_item_dictionary["label_lower"].str.contains("restraint|sitter|side rails", regex=True, na=False),
    "itemid"].astype(int))
fall_mobility_itemids = set(chart_item_dictionary.loc[
    chart_item_dictionary["label_lower"].str.contains("risk for falls|fall|mobility|ambulat|gait|transferring", regex=True, na=False),
    "itemid"].astype(int))

selected_chart_item_groups = {
    "oxygen_support": oxygen_support_itemids,
    "gcs": gcs_itemids,
    "orientation": orientation_itemids,
    "pain": pain_itemids,
    "rass_sedation": rass_sedation_itemids,
    "cam_delirium": cam_delirium_itemids,
    "agitation": agitation_itemids,
    "restraint": restraint_chart_itemids,
    "fall_mobility": fall_mobility_itemids}
selected_chart_itemids = set().union(*selected_chart_item_groups.values()) if selected_chart_item_groups else set()

print("Selected non-vital ICU chart item count:", len(selected_chart_itemids))
print(chart_item_dictionary.loc[chart_item_dictionary["itemid"].isin(selected_chart_itemids),
    ["itemid", "label", "category", "unitname"]].head(60))

selected_chart_id_cols = ["subject_id", "hadm_id"]
selected_chart_admission_context = df[selected_chart_id_cols + ["admittime", "dischtime"]].copy()
selected_chart_admission_context["admittime"] = pd.to_datetime(selected_chart_admission_context["admittime"], errors="coerce")
selected_chart_admission_context["dischtime"] = pd.to_datetime(selected_chart_admission_context["dischtime"], errors="coerce")
selected_chart_current_hadm_ids = set(selected_chart_admission_context["hadm_id"].dropna().astype(int))
selected_chart_features = df[selected_chart_id_cols].copy()
#loop through each item in this feature block
for col in selected_chart_event_feature_cols:
    selected_chart_features[col] = 0

chart_rows = []
if selected_chart_itemids:
    selected_chart_usecols = ["subject_id", "hadm_id", "stay_id", "charttime", "itemid", "value", "valuenum", "warning"]
    for chunk_number, chunk in enumerate(pd.read_csv(icu_path / "chartevents.csv.gz", usecols=selected_chart_usecols,
            chunksize=2_000_000, low_memory=False)):
        chunk = chunk[chunk["hadm_id"].isin(selected_chart_current_hadm_ids) & chunk["itemid"].isin(selected_chart_itemids)].copy()
        if chunk.empty:
            continue
        chunk["charttime"] = pd.to_datetime(chunk["charttime"], errors="coerce")
        chunk["valuenum"] = pd.to_numeric(chunk["valuenum"], errors="coerce")
        chunk["value_text"] = chunk["value"].fillna("").astype(str).str.lower()
        #join the derived features back to the admission table
        chunk = chunk.merge(selected_chart_admission_context, on=["subject_id", "hadm_id"], how="left")
        chunk = chunk[(chunk["charttime"].notna()) & (chunk["admittime"].notna()) &
            (chunk["charttime"] >= chunk["admittime"]) &
            ((chunk["dischtime"].isna()) | (chunk["charttime"] <= chunk["dischtime"]))].copy()
        if chunk.empty:
            continue

        chunk["oxygen_support_chart_event"] = chunk["itemid"].isin(oxygen_support_itemids).astype(int)
        chunk["gcs_chart_event"] = chunk["itemid"].isin(gcs_itemids).astype(int)
        chunk["orientation_chart_event"] = chunk["itemid"].isin(orientation_itemids).astype(int)
        chunk["pain_chart_event"] = chunk["itemid"].isin(pain_itemids).astype(int)
        chunk["rass_sedation_chart_event"] = chunk["itemid"].isin(rass_sedation_itemids).astype(int)
        chunk["cam_delirium_chart_event"] = chunk["itemid"].isin(cam_delirium_itemids).astype(int)
        chunk["agitation_chart_event"] = (chunk["itemid"].isin(agitation_itemids) |
            chunk["value_text"].str.contains("agitat|combative|restless", regex=True, na=False)).astype(int)
        chunk["restraint_chart_event"] = chunk["itemid"].isin(restraint_chart_itemids).astype(int)
        chunk["fall_mobility_chart_event"] = chunk["itemid"].isin(fall_mobility_itemids).astype(int)

        chunk["oxygen_flow_value"] = np.where(chunk["oxygen_support_chart_event"].eq(1), chunk["valuenum"], np.nan)
        chunk["gcs_value"] = np.where(chunk["gcs_chart_event"].eq(1), chunk["valuenum"], np.nan)
        chunk["pain_value"] = np.where(chunk["pain_chart_event"].eq(1), chunk["valuenum"], np.nan)
        chunk["rass_value"] = np.where(chunk["rass_sedation_chart_event"].eq(1), chunk["valuenum"], np.nan)

        chunk["impaired_orientation_event"] = ((chunk["orientation_chart_event"].eq(1)) &
            chunk["value_text"].str.contains("confused|disoriented|altered|drowsy|lethargic|unresponsive|not oriented", regex=True, na=False)).astype(int)
        chunk["delirium_or_cam_positive_event"] = ((chunk["cam_delirium_chart_event"].eq(1)) &
            chunk["value_text"].str.contains("positive|yes|present|delirium", regex=True, na=False) &
            ~chunk["value_text"].str.contains("negative|no", regex=True, na=False)).astype(int)
        chunk["fall_or_mobility_risk_event"] = ((chunk["fall_mobility_chart_event"].eq(1)) &
            (chunk["value_text"].str.contains("high|risk|impaired|assist|fall", regex=True, na=False) |
             chunk["valuenum"].fillna(0).gt(0))).astype(int)

        chunk["selected_neuro_behavioral_chart_event"] = chunk[["gcs_chart_event", "orientation_chart_event",
            "pain_chart_event", "rass_sedation_chart_event", "cam_delirium_chart_event", "agitation_chart_event",
            "restraint_chart_event", "fall_mobility_chart_event"]].max(axis=1)
        chunk["selected_resp_support_chart_event"] = chunk["oxygen_support_chart_event"]

        chunk["rass_extreme_event"] = ((chunk["rass_sedation_chart_event"].eq(1)) &
            chunk["rass_value"].notna() & chunk["rass_value"].abs().ge(3)).astype(int)
        chunk["oxygen_flow_sum"] = np.where(chunk["oxygen_flow_value"].notna(), chunk["oxygen_flow_value"], 0)
        chunk["oxygen_flow_count"] = chunk["oxygen_flow_value"].notna().astype(int)

        #summarise row-level records to admission-level features
        chart_rows.append(chunk.groupby(selected_chart_id_cols, as_index=False).agg(
            num_oxygen_support_chart_events=("oxygen_support_chart_event", "sum"),
            max_oxygen_flow_rate=("oxygen_flow_value", "max"),
            oxygen_flow_sum=("oxygen_flow_sum", "sum"),
            oxygen_flow_count=("oxygen_flow_count", "sum"),
            num_gcs_chart_events=("gcs_chart_event", "sum"),
            gcs_min_score=("gcs_value", "min"),
            num_orientation_chart_events=("orientation_chart_event", "sum"),
            impaired_orientation_flag=("impaired_orientation_event", "max"),
            num_pain_chart_events=("pain_chart_event", "sum"),
            pain_max_score=("pain_value", "max"),
            num_rass_sedation_chart_events=("rass_sedation_chart_event", "sum"),
            rass_extreme_flag=("rass_extreme_event", "max"),
            num_cam_delirium_chart_events=("cam_delirium_chart_event", "sum"),
            delirium_or_cam_positive_flag=("delirium_or_cam_positive_event", "max"),
            agitation_chart_event_flag=("agitation_chart_event", "max"),
            num_restraint_chart_events=("restraint_chart_event", "sum"),
            had_restraint_chart_event=("restraint_chart_event", "max"),
            num_fall_mobility_chart_events=("fall_mobility_chart_event", "sum"),
            fall_or_mobility_risk_flag=("fall_or_mobility_risk_event", "max"),
            num_selected_neuro_behavioral_chart_events=("selected_neuro_behavioral_chart_event", "sum"),
            num_selected_resp_support_chart_events=("selected_resp_support_chart_event", "sum")))
        print("Processed selected chart-event chunk:", chunk_number, "rows kept:", len(chunk))

if chart_rows:
    chart_combined = pd.concat(chart_rows, ignore_index=True)
    chart_agg_dict = {
        "num_oxygen_support_chart_events": "sum",
        "max_oxygen_flow_rate": "max",
        "oxygen_flow_sum": "sum",
        "oxygen_flow_count": "sum",
        "num_gcs_chart_events": "sum",
        "gcs_min_score": "min",
        "num_orientation_chart_events": "sum",
        "impaired_orientation_flag": "max",
        "num_pain_chart_events": "sum",
        "pain_max_score": "max",
        "num_rass_sedation_chart_events": "sum",
        "rass_extreme_flag": "max",
        "num_cam_delirium_chart_events": "sum",
        "delirium_or_cam_positive_flag": "max",
        "agitation_chart_event_flag": "max",
        "num_restraint_chart_events": "sum",
        "had_restraint_chart_event": "max",
        "num_fall_mobility_chart_events": "sum",
        "fall_or_mobility_risk_flag": "max",
        "num_selected_neuro_behavioral_chart_events": "sum",
        "num_selected_resp_support_chart_events": "sum"}
    #summarise row-level records to admission-level features
    chart_combined = chart_combined.groupby(selected_chart_id_cols, as_index=False).agg(chart_agg_dict)
    chart_combined["mean_oxygen_flow_rate"] = (chart_combined["oxygen_flow_sum"] /
        chart_combined["oxygen_flow_count"].replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)
    chart_combined = chart_combined.drop(columns=["oxygen_flow_sum", "oxygen_flow_count"], errors="ignore")
    selected_chart_features = selected_chart_features.drop(columns=selected_chart_event_feature_cols, errors="ignore").merge(
        chart_combined, on=selected_chart_id_cols, how="left")

for col in selected_chart_event_feature_cols:
    if col not in selected_chart_features.columns:
        selected_chart_features[col] = 0

selected_chart_features["high_oxygen_flow_flag"] = (pd.to_numeric(selected_chart_features["max_oxygen_flow_rate"], errors="coerce").fillna(0) >= 6).astype(int)
selected_chart_features["low_gcs_flag"] = (pd.to_numeric(selected_chart_features["gcs_min_score"], errors="coerce").fillna(15) < 15).astype(int)
selected_chart_features["high_pain_flag"] = (pd.to_numeric(selected_chart_features["pain_max_score"], errors="coerce").fillna(0) >= 7).astype(int)
chart_los_days = df[selected_chart_id_cols + ["hospital_los_days"]].copy() if "hospital_los_days" in df.columns else df[selected_chart_id_cols].assign(hospital_los_days=1)
#join the derived features back to the admission table
selected_chart_features = selected_chart_features.merge(chart_los_days, on=selected_chart_id_cols, how="left")
selected_chart_features["chart_acuity_event_density_per_day"] = (selected_chart_features["num_selected_neuro_behavioral_chart_events"].fillna(0) +
    selected_chart_features["num_selected_resp_support_chart_events"].fillna(0)) / selected_chart_features["hospital_los_days"].fillna(1).clip(lower=1)
selected_chart_features = selected_chart_features.drop(columns=["hospital_los_days"], errors="ignore")

for col in selected_chart_event_feature_cols:
    selected_chart_features[col] = pd.to_numeric(selected_chart_features[col], errors="coerce").fillna(0)
    if col not in ["max_oxygen_flow_rate", "mean_oxygen_flow_rate", "gcs_min_score", "pain_max_score", "chart_acuity_event_density_per_day"]:
        selected_chart_features[col] = selected_chart_features[col].astype(int)

df = df.drop(columns=[col for col in selected_chart_event_feature_cols if col in df.columns], errors="ignore")
df = df.merge(selected_chart_features[selected_chart_id_cols + selected_chart_event_feature_cols], on=selected_chart_id_cols, how="left")
for col in selected_chart_event_feature_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

print("Selected ICU chart-event acuity extraction complete.")
print("Selected chart-event feature count:", len(selected_chart_event_feature_cols))
print("Dataset shape after selected chart-event features:", df.shape)

Starting selected ICU chart-event acuity extraction...
Selected non-vital ICU chart item count: 176
      itemid                                             label  \
159   220739                                 GCS - Eye Opening   
332   223753                                   Riker-SAS Scale   
333   223754                                    Risk for Falls   
352   223781                                      Pain Present   
356   223791                                        Pain Level   
358   223794                             Pain Level Acceptable   
359   223795                            Pain Assessment Method   
371   223817                                         Agitation   
383   223834                                           O2 Flow   
384   223835                              Inspired O2 Fraction   
390   223848                                   Ventilator Type   
391   223849                                   Ventilator Mode   
406   223898                              

**Aggregate and merge vital sign features**

Vital signs are repeated measurements, so they were aggregated to admission level. For each selected vital sign, the first, last, mean, minimum, and maximum values were calculated. A simple change feature was also created by subtracting the first value from the last value. This provides static summaries while preserving some information about physiological change during admission. The filtered time-stamped vital sign table was also saved separately so that time-window or sequential models could be developed later.

In [ ]:
#sort vital signs so first and last values are chronological
selected_vitals = selected_vitals.sort_values(["subject_id", "hadm_id", "vital_name", "charttime"])

#aggregate repeated vital sign measurements to one row per admission and vital sign
vital_features_long = selected_vitals.groupby(["subject_id", "hadm_id", "vital_name"]).agg(
    vital_first_value=("valuenum", "first"), vital_last_value=("valuenum", "last"),
    vital_mean_value=("valuenum", "mean"), vital_min_value=("valuenum", "min"),
    vital_max_value=("valuenum", "max"),
    vital_measured=("valuenum", lambda x: int(x.notna().any()))).reset_index()

#calculate simple trend/change during admission
vital_features_long["vital_value_change"] = (vital_features_long["vital_last_value"]
    - vital_features_long["vital_first_value"])

print("Long vital table-derived feature table shape:")
print(vital_features_long.shape)
print(vital_features_long.head())

#pivot vital features to wide format
vital_wide = vital_features_long.pivot_table(index=["subject_id", "hadm_id"], columns="vital_name",
    values=["vital_first_value", "vital_last_value", "vital_mean_value", "vital_min_value",
        "vital_max_value", "vital_value_change", "vital_measured"], aggfunc="first")

#flatten multi-index columns
vital_wide.columns = [f"{vital}_{feature}" for feature, vital in vital_wide.columns]
vital_wide = vital_wide.reset_index()
print("\nWide vital features shape:")
print(vital_wide.shape)
print(vital_wide.head())

#remove previous vital features if this cell has already been run
existing_vital_cols = [col for col in df.columns if "_vital_" in col]
df = df.drop(columns=existing_vital_cols, errors="ignore")

#merge vital features into main dataset
df = df.merge(vital_wide, on=["subject_id", "hadm_id"], how="left")

vital_feature_cols = [col for col in vital_wide.columns if col not in ["subject_id", "hadm_id"]]

#fill measured indicators with 0 for admissions without icu vital charting
for col in vital_feature_cols:
    if col.endswith("_vital_measured"):
        df[col] = df[col].fillna(0).astype(int)

print("\nShape after merging vital sign features:", df.shape)
print("\nMissing vital sign feature values:")
print(df[vital_feature_cols].isna().sum().sort_values(ascending=False))
print("\nVital sign feature preview:")
print(df[["subject_id", "hadm_id"] + vital_feature_cols].head())

Long vital table-derived feature table shape:
(230788, 10)
   subject_id   hadm_id               vital_name  vital_first_value  \
0    10000032  29079034               heart_rate               91.0   
1    10000032  29079034      mean_bp_noninvasive               56.0   
2    10000032  29079034         respiratory_rate               24.0   
3    10000032  29079034                     spo2               98.0   
4    10000032  29079034  systolic_bp_noninvasive               84.0   

   vital_last_value  vital_mean_value  vital_min_value  vital_max_value  \
0              94.0              96.5             91.0            105.0   
1              62.0              62.3             56.0             67.0   
2              20.0              20.7             16.0             24.0   
3              95.0              96.3             94.0             99.0   
4              85.0              88.9             82.0             95.0   

   vital_measured  vital_value_change  
0               1      

#### **T2.1.2 - Additional admission-level raw-table extraction features**

These table-derived feature blocks remain in T2.1 because they extract or aggregate information directly from raw MIMIC-IV tables before cleaning and downstream feature engineering. Each block creates admission-level columns that can flow cleanly into T2.2 and T2.3.

**Prepare shared admission context for additional extraction features**

Admission identifiers, raw admission metadata, and diagnosis-code tables are prepared once so the following extraction blocks can reuse them.

In [ ]:
#prepare shared admission context for additional raw-table extraction features
id_cols = ["subject_id", "hadm_id"]

#standardise the current admission identifiers and timestamps for all later joins
current_admissions = df[id_cols + ["admittime", "dischtime"]].copy()
current_admissions["admittime"] = pd.to_datetime(current_admissions["admittime"], errors="coerce")
current_admissions["dischtime"] = pd.to_datetime(current_admissions["dischtime"], errors="coerce")
current_hadm_ids = set(current_admissions["hadm_id"].dropna().astype(int))
current_subject_ids = set(current_admissions["subject_id"].dropna().astype(int))

print("Preparing shared inputs for additional T2.1 extraction features...")
print("Current admission rows:", current_admissions.shape[0])
print("Current unique patients:", len(current_subject_ids))
print("Current unique admissions:", len(current_hadm_ids))

#load raw admissions metadata for prior utilisation windows and ed timing
raw_admissions_cols = ["subject_id", "hadm_id", "admittime", "dischtime", "edregtime", "edouttime", "discharge_location"]
raw_admissions = pd.read_csv(hosp_path / "admissions.csv.gz", usecols=raw_admissions_cols)
#loop through each item in this feature block
for col in ["admittime", "dischtime", "edregtime", "edouttime"]:
    raw_admissions[col] = pd.to_datetime(raw_admissions[col], errors="coerce")
raw_admissions = raw_admissions[raw_admissions["subject_id"].isin(current_subject_ids)].copy()

#load diagnosis codes once because they support psychiatric subgroups and physical comorbidity flags
diagnoses = pd.read_csv(hosp_path / "diagnoses_icd.csv.gz", usecols=["subject_id", "hadm_id", "icd_code", "icd_version"])
diagnoses = diagnoses[diagnoses["subject_id"].isin(current_subject_ids)].copy()
diagnoses["icd_code_clean"] = diagnoses["icd_code"].astype(str).str.upper().str.replace(".", "", regex=False)
diagnoses["icd3"] = diagnoses["icd_code_clean"].str[:3]
diagnoses["icd4"] = diagnoses["icd_code_clean"].str[:4]
diagnoses["icd3_numeric"] = pd.to_numeric(diagnoses["icd_code_clean"].str.extract(r"^(\d{3})")[0], errors="coerce")

#identify whether each raw admission had any psychiatric diagnosis for prior-history summaries
diagnoses["is_psych_diagnosis"] = (
    ((diagnoses["icd_version"] == 9) & diagnoses["icd3_numeric"].between(290, 319)) |
    ((diagnoses["icd_version"] == 10) & diagnoses["icd_code_clean"].str.startswith(("F", "R45851")))
).astype(int)
#summarise row-level records to admission-level features
admission_psych_flag = diagnoses.groupby(id_cols, as_index=False)["is_psych_diagnosis"].max()
admission_psych_flag = admission_psych_flag.rename(columns={"is_psych_diagnosis": "admission_has_psych_diagnosis"})
#join the derived features back to the admission table
raw_admissions = raw_admissions.merge(admission_psych_flag, on=id_cols, how="left")
raw_admissions["admission_has_psych_diagnosis"] = raw_admissions["admission_has_psych_diagnosis"].fillna(0).astype(int)
raw_admissions["discharge_location_clean"] = raw_admissions["discharge_location"].fillna("").astype(str).str.upper()
raw_admissions["prior_discharge_against_advice_source"] = raw_admissions["discharge_location_clean"].str.contains("AGAINST ADVICE|AMA", regex=True).astype(int)
raw_admissions["prior_discharge_to_psych_facility_source"] = raw_admissions["discharge_location_clean"].str.contains("PSYCH", regex=True).astype(int)
raw_admissions["prior_discharge_to_facility_source"] = raw_admissions["discharge_location_clean"].str.contains("SKILLED|REHAB|LONG TERM|CHRONIC|FACILITY|NURSING", regex=True).astype(int)

print("Raw admissions available for current patients:", raw_admissions.shape)
print("Diagnosis rows available for current patients:", diagnoses.shape)

Preparing shared inputs for additional T2.1 extraction features...
Current admission rows: 238565
Current unique patients: 107956
Current unique admissions: 238565
Raw admissions available for current patients: (338595, 12)
Diagnosis rows available for current patients: (4333977, 9)


**Create prior utilisation window features**

This block counts previous hospital, psychiatric, and non-psychiatric admissions before the index admission across several windows. It also creates recency, acceleration, and utilisation-pattern features so the model can distinguish long-term history from recent instability.

The features are derived only from admissions before the index admission. This matters because prior utilisation is one of the strongest available structured signals, but it has to be constructed chronologically to avoid using future admissions.


In [ ]:
#prior utilisation windows are calculated using previous discharge time, avoiding future admissions
#sections in this cell:
#1. sort admissions chronologically per patient.
#2. count prior hospital, psychiatric, non-psychiatric, icu, and ed contacts across lookback windows.
#3. derive prior discharge-pathway and admission-gap features using only admissions before the index stay.
window_days = [7, 14, 30, 60, 90, 180, 365]
prior_rows = []
#summarise row-level records to admission-level features
raw_admission_groups = {sid: group.sort_values("dischtime") for sid, group in raw_admissions.groupby("subject_id")}

#loop through each item in this feature block
for subject_id, current_group in current_admissions.groupby("subject_id", sort=False):
    history = raw_admission_groups.get(subject_id)
    if history is None or history.empty:
        #loop through each item in this feature block
        for _, row in current_group.iterrows():
            prior_rows.append({"subject_id": row["subject_id"], "hadm_id": row["hadm_id"]})
        continue

    history = history.dropna(subset=["dischtime"]).sort_values("dischtime")
    discharge_times = history["dischtime"].to_numpy(dtype="datetime64[ns]")
    admission_times = history["admittime"].to_numpy(dtype="datetime64[ns]")
    psych_flags = history["admission_has_psych_diagnosis"].to_numpy(dtype=int)
    ed_visit_flags = history["edregtime"].notna().to_numpy(dtype=int) if "edregtime" in history.columns else np.zeros(len(history), dtype=int)
    prior_discharge_against_advice_flags = history["prior_discharge_against_advice_source"].to_numpy(dtype=int)
    prior_discharge_psych_facility_flags = history["prior_discharge_to_psych_facility_source"].to_numpy(dtype=int)
    prior_discharge_facility_flags = history["prior_discharge_to_facility_source"].to_numpy(dtype=int)
    los_days = ((history["dischtime"] - history["admittime"]).dt.total_seconds() / 86400).to_numpy()

    #loop through each item in this feature block
    for _, row in current_group.iterrows():
        admission_time = row["admittime"]
        result = {"subject_id": row["subject_id"], "hadm_id": row["hadm_id"]}

        if pd.isna(admission_time) or len(history) == 0:
            #loop through each item in this feature block
            for days in window_days:
                result[f"previous_total_admissions_{days}d"] = 0
                result[f"previous_psych_admissions_{days}d"] = 0
                result[f"previous_nonpsych_admissions_{days}d"] = 0
                result[f"previous_ed_visits_{days}d"] = 0
            result["days_since_previous_hospital_admission"] = np.nan
            result["prior_near_30d_readmission_flag"] = 0
            result["previous_admission_within_27_33d_flag"] = 0
            result["days_since_previous_admission_near_30d_flag"] = 0
            result["previous_discharge_against_advice_flag"] = 0
            result["previous_discharge_to_psych_facility_flag"] = 0
            result["previous_discharge_to_facility_flag"] = 0
            result["previous_short_los_flag"] = 0
            result["previous_long_los_flag"] = 0
            result["previous_admission_los_days"] = np.nan
            result["mean_previous_los_days"] = np.nan
            result["max_previous_los_days"] = np.nan
            result["time_between_last_two_admissions_days"] = np.nan
            result["mean_previous_admission_gap_days"] = np.nan
            result["min_previous_admission_gap_days"] = np.nan
            result["std_previous_admission_gap_days"] = np.nan
            prior_rows.append(result)
            continue

        admission_time64 = np.datetime64(admission_time)
        previous_end = np.searchsorted(discharge_times, admission_time64, side="left")
        previous_discharge_times = discharge_times[:previous_end]
        previous_psych_flags = psych_flags[:previous_end]
        previous_ed_visit_flags = ed_visit_flags[:previous_end]
        previous_admission_times = admission_times[:previous_end]
        previous_prior_discharge_against_advice_flags = prior_discharge_against_advice_flags[:previous_end]
        previous_prior_discharge_psych_facility_flags = prior_discharge_psych_facility_flags[:previous_end]
        previous_prior_discharge_facility_flags = prior_discharge_facility_flags[:previous_end]
        previous_los_days = los_days[:previous_end]

        for days in window_days:
            window_start = np.datetime64(admission_time - pd.Timedelta(days=days))
            window_start_index = np.searchsorted(previous_discharge_times, window_start, side="left")
            window_psych_flags = previous_psych_flags[window_start_index:]
            result[f"previous_total_admissions_{days}d"] = int(len(window_psych_flags))
            result[f"previous_psych_admissions_{days}d"] = int(window_psych_flags.sum())
            result[f"previous_nonpsych_admissions_{days}d"] = int(len(window_psych_flags) - window_psych_flags.sum())
            result[f"previous_ed_visits_{days}d"] = int(previous_ed_visit_flags[window_start_index:].sum())

        if len(previous_discharge_times) > 0:
            result["days_since_previous_hospital_admission"] = (
                admission_time64 - previous_discharge_times[-1]) / np.timedelta64(1, "D")
            result["previous_admission_within_27_33d_flag"] = int(27 <= result["days_since_previous_hospital_admission"] <= 33)
            result["days_since_previous_admission_near_30d_flag"] = result["previous_admission_within_27_33d_flag"]
            result["prior_near_30d_readmission_flag"] = result["previous_admission_within_27_33d_flag"]
            result["previous_discharge_against_advice_flag"] = int(previous_prior_discharge_against_advice_flags[-1] == 1)
            result["previous_discharge_to_psych_facility_flag"] = int(previous_prior_discharge_psych_facility_flags[-1] == 1)
            result["previous_discharge_to_facility_flag"] = int(previous_prior_discharge_facility_flags[-1] == 1)
            result["previous_short_los_flag"] = int(pd.notna(previous_los_days[-1]) and previous_los_days[-1] < 2)
            result["previous_long_los_flag"] = int(pd.notna(previous_los_days[-1]) and previous_los_days[-1] >= 7)
            result["previous_admission_los_days"] = previous_los_days[-1]
            result["mean_previous_los_days"] = np.nanmean(previous_los_days)
            result["max_previous_los_days"] = np.nanmax(previous_los_days)
            if len(previous_discharge_times) > 1:
                previous_gap_days = ((previous_admission_times[1:] - previous_discharge_times[:-1]) / np.timedelta64(1, "D")).astype(float)
                previous_gap_days = previous_gap_days[np.isfinite(previous_gap_days) & (previous_gap_days >= 0)]
                if len(previous_gap_days) > 0:
                    result["time_between_last_two_admissions_days"] = previous_gap_days[-1]
                    result["mean_previous_admission_gap_days"] = np.nanmean(previous_gap_days)
                    result["min_previous_admission_gap_days"] = np.nanmin(previous_gap_days)
                    result["std_previous_admission_gap_days"] = np.nanstd(previous_gap_days)
                else:
                    result["time_between_last_two_admissions_days"] = np.nan
                    result["mean_previous_admission_gap_days"] = np.nan
                    result["min_previous_admission_gap_days"] = np.nan
                    result["std_previous_admission_gap_days"] = np.nan
            else:
                result["time_between_last_two_admissions_days"] = np.nan
                result["mean_previous_admission_gap_days"] = np.nan
                result["min_previous_admission_gap_days"] = np.nan
                result["std_previous_admission_gap_days"] = np.nan
        else:
            result["days_since_previous_hospital_admission"] = np.nan
            result["prior_near_30d_readmission_flag"] = 0
            result["previous_admission_within_27_33d_flag"] = 0
            result["days_since_previous_admission_near_30d_flag"] = 0
            result["previous_discharge_against_advice_flag"] = 0
            result["previous_discharge_to_psych_facility_flag"] = 0
            result["previous_discharge_to_facility_flag"] = 0
            result["previous_short_los_flag"] = 0
            result["previous_long_los_flag"] = 0
            result["previous_admission_los_days"] = np.nan
            result["mean_previous_los_days"] = np.nan
            result["max_previous_los_days"] = np.nan
            result["time_between_last_two_admissions_days"] = np.nan
            result["mean_previous_admission_gap_days"] = np.nan
            result["min_previous_admission_gap_days"] = np.nan
            result["std_previous_admission_gap_days"] = np.nan
        prior_rows.append(result)

prior_utilisation_features = pd.DataFrame(prior_rows)
df = df.drop(columns=[col for col in prior_utilisation_features.columns if col not in id_cols], errors="ignore")
#join the derived features back to the admission table
df = df.merge(prior_utilisation_features, on=id_cols, how="left")

for days in window_days:
    for prefix in ["previous_total_admissions", "previous_psych_admissions", "previous_nonpsych_admissions", "previous_ed_visits"]:
        col = f"{prefix}_{days}d"
        df[col] = df[col].fillna(0).astype(int)

df["admissions_per_30d"] = df["previous_total_admissions_30d"] / 30
df["admissions_per_90d"] = df["previous_total_admissions_90d"] / 90
df["admissions_per_365d"] = df["previous_total_admissions_365d"] / 365
df["admission_frequency_acceleration"] = df["admissions_per_30d"] - df["admissions_per_365d"]
df["increasing_admission_frequency_flag"] = (df["admission_frequency_acceleration"] > 0).astype(int)
df["decreasing_admission_frequency_flag"] = (df["admission_frequency_acceleration"] < 0).astype(int)
df["frequent_ed_user_flag"] = (df["previous_ed_visits_365d"] >= 2).astype(int)

for col in ["prior_near_30d_readmission_flag", "previous_admission_within_27_33d_flag",
        "days_since_previous_admission_near_30d_flag", "previous_discharge_against_advice_flag",
        "previous_discharge_to_psych_facility_flag", "previous_discharge_to_facility_flag",
        "previous_short_los_flag", "previous_long_los_flag", "frequent_ed_user_flag",
        "increasing_admission_frequency_flag", "decreasing_admission_frequency_flag"]:
    df[col] = df[col].fillna(0).astype(int)

for col in ["previous_admission_los_days", "mean_previous_los_days", "max_previous_los_days",
        "time_between_last_two_admissions_days", "mean_previous_admission_gap_days",
        "min_previous_admission_gap_days", "std_previous_admission_gap_days"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["previous_admission_los_days_filled"] = df["previous_admission_los_days"].fillna(0)
df["mean_previous_los_days_filled"] = df["mean_previous_los_days"].fillna(0)
df["max_previous_los_days_filled"] = df["max_previous_los_days"].fillna(0)
df["time_between_last_two_admissions_days_filled"] = df["time_between_last_two_admissions_days"].fillna(9999)
df["mean_previous_admission_gap_days_filled"] = df["mean_previous_admission_gap_days"].fillna(9999)
df["min_previous_admission_gap_days_filled"] = df["min_previous_admission_gap_days"].fillna(9999)
df["std_previous_admission_gap_days_filled"] = df["std_previous_admission_gap_days"].fillna(0)

df["days_since_previous_hospital_admission_filled"] = df["days_since_previous_hospital_admission"].fillna(9999)
df["high_utiliser_previous_year"] = (df["previous_total_admissions_365d"] >= 3).astype(int)
df["previous_psych_to_total_admission_ratio"] = np.where(
    df["previous_total_admissions"] > 0,
    df["previous_psych_admissions_from_all_hosp"] / df["previous_total_admissions"], 0)
df["previous_2plus_psych_admissions"] = (df["previous_psych_admissions_from_all_hosp"] >= 2).astype(int)
df["previous_3plus_psych_admissions"] = (df["previous_psych_admissions_from_all_hosp"] >= 3).astype(int)
df["previous_2plus_total_admissions_365d"] = (df["previous_total_admissions_365d"] >= 2).astype(int)
df["previous_3plus_total_admissions_365d"] = (df["previous_total_admissions_365d"] >= 3).astype(int)
df["frequent_psych_admitter_flag"] = ((df["previous_psych_admissions_365d"] >= 2) |
    (df["previous_psych_admissions_from_all_hosp"] >= 3)).astype(int)

frequent_admission_cols = ["previous_2plus_psych_admissions", "previous_3plus_psych_admissions",
    "previous_2plus_total_admissions_365d", "previous_3plus_total_admissions_365d",
    "frequent_psych_admitter_flag", "prior_near_30d_readmission_flag", "previous_admission_within_27_33d_flag",
    "previous_discharge_against_advice_flag", "previous_discharge_to_psych_facility_flag",
    "previous_discharge_to_facility_flag", "previous_short_los_flag", "previous_long_los_flag", "frequent_ed_user_flag",
    "increasing_admission_frequency_flag", "decreasing_admission_frequency_flag"]
print("Prior utilisation window and frequent-admitter features created:")
print(df[frequent_admission_cols].sum().sort_values(ascending=False))

Prior utilisation window and frequent-admitter features created:
previous_2plus_psych_admissions              88614
frequent_psych_admitter_flag                 76742
previous_2plus_total_admissions_365d         68824
previous_3plus_psych_admissions              66736
previous_short_los_flag                      61466
decreasing_admission_frequency_flag          61429
frequent_ed_user_flag                        55169
increasing_admission_frequency_flag          53532
previous_3plus_total_admissions_365d         45735
previous_long_los_flag                       27561
previous_discharge_to_facility_flag          24265
previous_admission_within_27_33d_flag         4811
prior_near_30d_readmission_flag               4811
previous_discharge_to_psych_facility_flag     1822
previous_discharge_against_advice_flag        1731
dtype: int64


/tmp/ipykernel_52805/4069560834.py:142: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["admissions_per_30d"] = df["previous_total_admissions_30d"] / 30
/tmp/ipykernel_52805/4069560834.py:143: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["admissions_per_90d"] = df["previous_total_admissions_90d"] / 90
/tmp/ipykernel_52805/4069560834.py:144: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once us

**Create ED timing features**

Emergency department registration and departure timestamps are converted into ED length of stay and ED-to-admission timing features. These columns describe the admission pathway before the inpatient stay begins.

In [ ]:
#ed timing from raw admissions, available in the admissions table when the stay passed through ed registration
ed_features = raw_admissions[raw_admissions["hadm_id"].isin(current_hadm_ids)][id_cols + ["edregtime", "edouttime", "admittime"]].copy()
ed_features["has_ed_timing"] = ed_features[["edregtime", "edouttime"]].notna().any(axis=1).astype(int)
ed_features["ed_length_of_stay_hours"] = ((ed_features["edouttime"] - ed_features["edregtime"]).dt.total_seconds() / 3600)
ed_features["ed_to_admission_hours"] = ((ed_features["admittime"] - ed_features["edregtime"]).dt.total_seconds() / 3600)
ed_features["ed_to_ward_delay_hours"] = ((ed_features["admittime"] - ed_features["edouttime"]).dt.total_seconds() / 3600)
for col in ["ed_length_of_stay_hours", "ed_to_admission_hours", "ed_to_ward_delay_hours"]:
    ed_features.loc[ed_features[col] < 0, col] = np.nan
ed_features = ed_features.drop(columns=["edregtime", "edouttime", "admittime"])
df = df.drop(columns=[col for col in ed_features.columns if col not in id_cols], errors="ignore")
df = df.merge(ed_features, on=id_cols, how="left")
df["has_ed_timing"] = df["has_ed_timing"].fillna(0).astype(int)

**Create prior ICU exposure features**

Previous ICU admissions and time since prior ICU discharge are calculated using ICU stays before the index admission. These features capture prior critical-care exposure and longer-term medical complexity.

In [ ]:
#prior icu exposure using previous icu outtimes
#very recent icu windows capture acute medical cycling before the index admission
icu_window_days = [7, 14, 30, 60, 90, 180, 365]
icu_stays = pd.read_csv(icu_path / "icustays.csv.gz", usecols=["subject_id", "hadm_id", "intime", "outtime", "first_careunit", "last_careunit"])
icu_stays = icu_stays[icu_stays["subject_id"].isin(current_subject_ids)].copy()
icu_stays["intime"] = pd.to_datetime(icu_stays["intime"], errors="coerce")
icu_stays["outtime"] = pd.to_datetime(icu_stays["outtime"], errors="coerce")
icu_groups = {sid: group.dropna(subset=["outtime"]).sort_values("outtime") for sid, group in icu_stays.groupby("subject_id")}
prior_icu_rows = []

for subject_id, current_group in current_admissions.groupby("subject_id", sort=False):
    history = icu_groups.get(subject_id)
    outtimes = np.array([], dtype="datetime64[ns]") if history is None else history["outtime"].to_numpy(dtype="datetime64[ns]")
    for _, row in current_group.iterrows():
        admission_time = row["admittime"]
        result = {"subject_id": row["subject_id"], "hadm_id": row["hadm_id"],
            "previous_icu_admissions": 0, "has_previous_icu_admission": 0,
            "days_since_previous_icu_admission": np.nan}
        for days in icu_window_days:
            result[f"previous_icu_admissions_{days}d"] = 0
            result[f"recent_icu_admission_{days}d_flag"] = 0
        if pd.notna(admission_time) and len(outtimes) > 0:
            admission_time64 = np.datetime64(admission_time)
            previous_end = np.searchsorted(outtimes, admission_time64, side="left")
            previous_outtimes = outtimes[:previous_end]
            result["previous_icu_admissions"] = int(previous_end)
            result["has_previous_icu_admission"] = int(previous_end > 0)
            if previous_end > 0:
                result["days_since_previous_icu_admission"] = ((admission_time64 - previous_outtimes[-1]) / np.timedelta64(1, "D"))
            for days in icu_window_days:
                window_start = np.datetime64(admission_time - pd.Timedelta(days=days))
                window_start_index = np.searchsorted(previous_outtimes, window_start, side="left")
                window_count = int(len(previous_outtimes[window_start_index:]))
                result[f"previous_icu_admissions_{days}d"] = window_count
                result[f"recent_icu_admission_{days}d_flag"] = int(window_count > 0)
        prior_icu_rows.append(result)

prior_icu_features = pd.DataFrame(prior_icu_rows)
df = df.drop(columns=[col for col in prior_icu_features.columns if col not in id_cols], errors="ignore")
df = df.merge(prior_icu_features, on=id_cols, how="left")
df["previous_icu_admissions"] = df["previous_icu_admissions"].fillna(0).astype(int)
df["has_previous_icu_admission"] = df["has_previous_icu_admission"].fillna(0).astype(int)
for days in icu_window_days:
    df[f"previous_icu_admissions_{days}d"] = df[f"previous_icu_admissions_{days}d"].fillna(0).astype(int)
    df[f"recent_icu_admission_{days}d_flag"] = df[f"recent_icu_admission_{days}d_flag"].fillna(0).astype(int)
df["days_since_previous_icu_admission_filled"] = df["days_since_previous_icu_admission"].fillna(9999)

print("Prior ICU window features created:")
print(df[[f"previous_icu_admissions_{days}d" for days in icu_window_days]].sum().sort_values(ascending=False))

Prior ICU window features created:
previous_icu_admissions_365d    43856
previous_icu_admissions_180d    29047
previous_icu_admissions_90d     18666
previous_icu_admissions_60d     14016
previous_icu_admissions_30d      8145
previous_icu_admissions_14d      3934
previous_icu_admissions_7d       1707
dtype: int64


/tmp/ipykernel_52805/3764671361.py:46: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["days_since_previous_icu_admission_filled"] = df["days_since_previous_icu_admission"].fillna(9999)


**Create ICD psychiatric and physical comorbidity features**

Diagnosis codes are used to add specific psychiatric subgroup flags and simplified physical comorbidity flags. These columns separate broad diagnosis burden from clinically interpretable mental-health and physical-health conditions.

In [ ]:
#icd-derived psychiatric subgroups and simplified physical comorbidity flags for the index admission
index_dx = diagnoses[diagnoses["hadm_id"].isin(current_hadm_ids)].copy()
index_dx["has_self_harm_or_suicidal_ideation"] = (index_dx["icd_code_clean"].str.startswith(("R45851", "T1491")) |
    index_dx["icd_code_clean"].str.startswith(("X7", "X80", "X81", "X82", "X83", "E95", "V6284"))).astype(int)
index_dx["has_alcohol_related_disorder"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith("F10")) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith(("303", "3050")))).astype(int)
index_dx["has_opioid_related_disorder"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith("F11")) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith(("3040", "3055")))).astype(int)
index_dx["has_stimulant_related_disorder"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("F14", "F15"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith(("3042", "3056", "3057")))).astype(int)
index_dx["has_diabetes"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("E10", "E11", "E12", "E13", "E14"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith("250"))).astype(int)
index_dx["has_chronic_kidney_disease"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith("N18")) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith("585"))).astype(int)
index_dx["has_copd"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("J40", "J41", "J42", "J43", "J44"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd3_numeric"].between(490, 496))).astype(int)
index_dx["has_heart_failure"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith("I50")) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith("428"))).astype(int)
index_dx["has_liver_disease"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("K70", "K71", "K72", "K73", "K74", "K75", "K76", "K77"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd3_numeric"].between(570, 573))).astype(int)
index_dx["has_cancer"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith("C")) |
    ((index_dx["icd_version"] == 9) & index_dx["icd3_numeric"].between(140, 209))).astype(int)
index_dx["has_dementia"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("F00", "F01", "F02", "F03", "G30"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith(("290", "3310")))).astype(int)

icd_feature_cols = ["has_self_harm_or_suicidal_ideation", "has_alcohol_related_disorder",
    "has_opioid_related_disorder", "has_stimulant_related_disorder", "has_diabetes",
    "has_chronic_kidney_disease", "has_copd", "has_heart_failure", "has_liver_disease",
    "has_cancer", "has_dementia"]
#summarise row-level records to admission-level features
icd_features = index_dx.groupby(id_cols, as_index=False)[icd_feature_cols].max()
physical_flag_cols = ["has_diabetes", "has_chronic_kidney_disease", "has_copd", "has_heart_failure",
    "has_liver_disease", "has_cancer", "has_dementia"]
icd_features["physical_comorbidity_count"] = icd_features[physical_flag_cols].sum(axis=1)

#simplified charlson-style score; this is not a fully validated claims algorithm, but gives an interpretable physical comorbidity burden measure
charlson_weights = {"has_heart_failure": 1, "has_copd": 1, "has_diabetes": 1, "has_dementia": 1,
    "has_chronic_kidney_disease": 2, "has_liver_disease": 3, "has_cancer": 2}
icd_features["charlson_comorbidity_index_simplified"] = sum(icd_features[col] * weight for col, weight in charlson_weights.items())
icd_features["elixhauser_comorbidity_group_count_simplified"] = icd_features[physical_flag_cols].sum(axis=1)

df = df.drop(columns=[col for col in icd_features.columns if col not in id_cols], errors="ignore")
#join the derived features back to the admission table
df = df.merge(icd_features, on=id_cols, how="left")
#loop through each item in this feature block
for col in icd_feature_cols + ["physical_comorbidity_count", "charlson_comorbidity_index_simplified", "elixhauser_comorbidity_group_count_simplified"]:
    df[col] = df[col].fillna(0).astype(int)

#prior diagnosis persistence features compare current admission diagnoses with diagnoses recorded before the index admission.
diagnosis_persistence_cols = ["psychosis_on_current_and_previous_admission_flag", "depression_on_current_and_previous_admission_flag",
    "substance_use_on_current_and_previous_admission_flag", "self_harm_recurrent_flag", "num_recurrent_psych_diagnosis_groups"]
all_dx_for_persistence = diagnoses.copy()
all_dx_for_persistence["dx_psychosis_flag"] = (((all_dx_for_persistence["icd_version"] == 10) & all_dx_for_persistence["icd_code_clean"].str.startswith(("F20", "F21", "F22", "F23", "F24", "F25", "F28", "F29"))) |
    ((all_dx_for_persistence["icd_version"] == 9) & all_dx_for_persistence["icd_code_clean"].str.startswith(("295", "297", "298")))).astype(int)
all_dx_for_persistence["dx_depression_flag"] = (((all_dx_for_persistence["icd_version"] == 10) & all_dx_for_persistence["icd_code_clean"].str.startswith(("F32", "F33"))) |
    ((all_dx_for_persistence["icd_version"] == 9) & all_dx_for_persistence["icd_code_clean"].str.startswith(("2962", "2963", "311")))).astype(int)
all_dx_for_persistence["dx_substance_use_flag"] = (((all_dx_for_persistence["icd_version"] == 10) & all_dx_for_persistence["icd_code_clean"].str.startswith(("F10", "F11", "F12", "F13", "F14", "F15", "F16", "F18", "F19"))) |
    ((all_dx_for_persistence["icd_version"] == 9) & all_dx_for_persistence["icd_code_clean"].str.startswith(("303", "304", "305")))).astype(int)
all_dx_for_persistence["dx_self_harm_flag"] = (all_dx_for_persistence["icd_code_clean"].str.startswith(("R45851", "T1491", "X7", "X80", "X81", "X82", "X83", "E95", "V6284"))).astype(int)

dx_persistence_admission_flags = all_dx_for_persistence.groupby(id_cols, as_index=False)[
    ["dx_psychosis_flag", "dx_depression_flag", "dx_substance_use_flag", "dx_self_harm_flag"]].max()
dx_persistence_history = raw_admissions[["subject_id", "hadm_id", "admittime", "dischtime"]].merge(
    dx_persistence_admission_flags, on=id_cols, how="left")
for col in ["dx_psychosis_flag", "dx_depression_flag", "dx_substance_use_flag", "dx_self_harm_flag"]:
    dx_persistence_history[col] = dx_persistence_history[col].fillna(0).astype(int)

dx_persistence_rows = []
dx_history_groups = {sid: group.dropna(subset=["dischtime"]).sort_values("dischtime")
    #summarise row-level records to admission-level features
    for sid, group in dx_persistence_history.groupby("subject_id")}
for _, row in current_admissions.iterrows():
    history = dx_history_groups.get(row["subject_id"])
    result = {"subject_id": row["subject_id"], "hadm_id": row["hadm_id"]}
    if history is None or pd.isna(row["admittime"]):
        prior_flags = pd.Series({"dx_psychosis_flag": 0, "dx_depression_flag": 0, "dx_substance_use_flag": 0, "dx_self_harm_flag": 0})
    else:
        prior_history = history[history["dischtime"] < row["admittime"]]
        prior_flags = prior_history[["dx_psychosis_flag", "dx_depression_flag", "dx_substance_use_flag", "dx_self_harm_flag"]].max()
        prior_flags = prior_flags.fillna(0) if len(prior_history) > 0 else pd.Series({"dx_psychosis_flag": 0, "dx_depression_flag": 0, "dx_substance_use_flag": 0, "dx_self_harm_flag": 0})
    current_flags = dx_persistence_admission_flags[(dx_persistence_admission_flags["subject_id"] == row["subject_id"]) &
        (dx_persistence_admission_flags["hadm_id"] == row["hadm_id"])]
    if current_flags.empty:
        current_flags = pd.Series({"dx_psychosis_flag": 0, "dx_depression_flag": 0, "dx_substance_use_flag": 0, "dx_self_harm_flag": 0})
    else:
        current_flags = current_flags.iloc[0]
    result["psychosis_on_current_and_previous_admission_flag"] = int((current_flags["dx_psychosis_flag"] == 1) and (prior_flags["dx_psychosis_flag"] == 1))
    result["depression_on_current_and_previous_admission_flag"] = int((current_flags["dx_depression_flag"] == 1) and (prior_flags["dx_depression_flag"] == 1))
    result["substance_use_on_current_and_previous_admission_flag"] = int((current_flags["dx_substance_use_flag"] == 1) and (prior_flags["dx_substance_use_flag"] == 1))
    result["self_harm_recurrent_flag"] = int((current_flags["dx_self_harm_flag"] == 1) and (prior_flags["dx_self_harm_flag"] == 1))
    result["num_recurrent_psych_diagnosis_groups"] = sum(result[col] for col in diagnosis_persistence_cols[:-1])
    dx_persistence_rows.append(result)

diagnosis_persistence_features = pd.DataFrame(dx_persistence_rows)
df = df.drop(columns=[col for col in diagnosis_persistence_cols if col in df.columns], errors="ignore")
df = df.merge(diagnosis_persistence_features, on=id_cols, how="left")
for col in diagnosis_persistence_cols:
    df[col] = df[col].fillna(0).astype(int)

**Create prescription exposure features**

Prescription orders are summarised into antibiotic, opioid, steroid, IV order, psychiatric medication, antipsychotic polypharmacy, and long-acting injectable antipsychotic features. These are medication-order proxies, not proof that a dose was administered.

In [ ]:
#prescription exposure features, kept as medication-order proxies rather than full emar administration features
prescription_feature_rows = []
antipsychotic_name_rows = []
admission_times_for_rx = current_admissions[["hadm_id", "admittime"]].copy()
rx_usecols = ["subject_id", "hadm_id", "starttime", "drug", "route"]
rx_chunk_size = 2_000_000
#loop through each item in this feature block
for chunk_number, rx_chunk in enumerate(pd.read_csv(hosp_path / "prescriptions.csv.gz", usecols=rx_usecols, chunksize=rx_chunk_size)):
    rx_chunk = rx_chunk[rx_chunk["hadm_id"].isin(current_hadm_ids)].copy()
    if rx_chunk.empty:
        continue
    rx_chunk["starttime"] = pd.to_datetime(rx_chunk["starttime"], errors="coerce")
    #join the derived features back to the admission table
    rx_chunk = rx_chunk.merge(admission_times_for_rx, on="hadm_id", how="left")
    drug_text = rx_chunk["drug"].astype(str).str.lower()
    route_text = rx_chunk["route"].astype(str).str.lower()
    rx_chunk["had_antibiotic_exposure"] = drug_text.str.contains("vancomycin|cef|penicillin|piperacillin|tazobactam|meropenem|azithromycin|levofloxacin|ciprofloxacin|metronidazole|doxycycline|gentamicin|trimethoprim|sulfamethoxazole", regex=True).astype(int)
    rx_chunk["had_opioid_exposure"] = drug_text.str.contains("morphine|oxycodone|hydromorphone|fentanyl|methadone|tramadol|codeine|hydrocodone", regex=True).astype(int)
    rx_chunk["had_steroid_exposure"] = drug_text.str.contains("prednisone|prednisolone|methylprednisolone|hydrocortisone|dexamethasone", regex=True).astype(int)
    rx_chunk["had_antipsychotic_order"] = drug_text.str.contains("haloperidol|risperidone|olanzapine|quetiapine|clozapine|aripiprazole|ziprasidone|chlorpromazine|fluphenazine|lurasidone|paliperidone|perphenazine|thioridazine|trifluoperazine|prochlorperazine", regex=True).astype(int)
    rx_chunk["had_benzodiazepine_order"] = drug_text.str.contains("lorazepam|diazepam|clonazepam|alprazolam|midazolam|temazepam|chlordiazepoxide|oxazepam|triazolam", regex=True).astype(int)
    rx_chunk["had_mood_stabiliser_order"] = drug_text.str.contains("lithium|valproate|valproic acid|divalproex|lamotrigine|carbamazepine|oxcarbazepine", regex=True).astype(int)
    rx_chunk["had_antidepressant_order"] = drug_text.str.contains("sertraline|fluoxetine|citalopram|escitalopram|paroxetine|venlafaxine|duloxetine|mirtazapine|trazodone|bupropion|amitriptyline|nortriptyline", regex=True).astype(int)
    rx_chunk["had_long_acting_injectable_antipsychotic"] = drug_text.str.contains("decanoate|paliperidone palmitate|invega sustenna|invega trinza|invega hafyera|risperdal consta|risperidone microspheres|risperidone long|aripiprazole lauroxil|abilify maintena|haloperidol decanoate|fluphenazine decanoate", regex=True).astype(int)
    rx_chunk["antipsychotic_drug_name"] = rx_chunk["drug"].astype(str).str.lower().where(rx_chunk["had_antipsychotic_order"] == 1)
    rx_chunk["had_iv_prescription"] = route_text.str.contains("iv|intravenous|ivpb|iv drip", regex=True).astype(int)
    rx_chunk["medication_order_first_24h"] = rx_chunk["starttime"].between(rx_chunk["admittime"], rx_chunk["admittime"] + pd.Timedelta(hours=24), inclusive="both").astype(int)
    rx_chunk["medication_order_first_72h"] = rx_chunk["starttime"].between(rx_chunk["admittime"], rx_chunk["admittime"] + pd.Timedelta(hours=72), inclusive="both").astype(int)
    rx_chunk["psychotropic_order_started_after_72h"] = ((rx_chunk["starttime"] > rx_chunk["admittime"] + pd.Timedelta(hours=72)) &
        ((rx_chunk["had_antipsychotic_order"] == 1) | (rx_chunk["had_benzodiazepine_order"] == 1) |
         (rx_chunk["had_mood_stabiliser_order"] == 1) | (rx_chunk["had_antidepressant_order"] == 1))).astype(int)
    #summarise row-level records to admission-level features
    prescription_feature_rows.append(rx_chunk.groupby(id_cols, as_index=False).agg(
        num_antibiotic_prescriptions=("had_antibiotic_exposure", "sum"),
        had_antibiotic_exposure=("had_antibiotic_exposure", "max"),
        num_opioid_prescriptions=("had_opioid_exposure", "sum"),
        had_opioid_exposure=("had_opioid_exposure", "max"),
        num_steroid_prescriptions=("had_steroid_exposure", "sum"),
        had_steroid_exposure=("had_steroid_exposure", "max"),
        antipsychotic_prescription_count=("had_antipsychotic_order", "sum"),
        had_long_acting_injectable_antipsychotic=("had_long_acting_injectable_antipsychotic", "max"),
        new_antipsychotic_started_flag=("had_antipsychotic_order", "max"),
        new_antidepressant_started_flag=("had_antidepressant_order", "max"),
        had_benzodiazepine_order=("had_benzodiazepine_order", "max"),
        new_benzodiazepine_started_flag=("had_benzodiazepine_order", "max"),
        had_mood_stabiliser_order=("had_mood_stabiliser_order", "max"),
        new_mood_stabiliser_started_flag=("had_mood_stabiliser_order", "max"),
        psychotropic_medication_change_count=("psychotropic_order_started_after_72h", "sum"),
        psychotropic_started_after_72h_flag=("psychotropic_order_started_after_72h", "max"),
        late_psychotropic_change_flag=("psychotropic_order_started_after_72h", "max"),
        num_iv_prescription_rows=("had_iv_prescription", "sum"),
        had_iv_prescription=("had_iv_prescription", "max"),
        medication_order_count_first_24h=("medication_order_first_24h", "sum"),
        medication_order_count_first_72h=("medication_order_first_72h", "sum")))
    antipsychotic_name_rows.append(rx_chunk.loc[rx_chunk["had_antipsychotic_order"] == 1,
        id_cols + ["antipsychotic_drug_name"]].dropna().drop_duplicates())
    print("Processed prescription chunk:", chunk_number, "rows kept:", len(rx_chunk))

if prescription_feature_rows:
    prescription_features = pd.concat(prescription_feature_rows, ignore_index=True).groupby(id_cols, as_index=False).sum()
    if len(antipsychotic_name_rows) > 0:
        antipsychotic_unique_counts = pd.concat(antipsychotic_name_rows, ignore_index=True).drop_duplicates().groupby(
            id_cols, as_index=False)["antipsychotic_drug_name"].nunique()
        antipsychotic_unique_counts = antipsychotic_unique_counts.rename(columns={"antipsychotic_drug_name": "num_unique_antipsychotics"})
        #join the derived features back to the admission table
        prescription_features = prescription_features.merge(antipsychotic_unique_counts, on=id_cols, how="left")
    else:
        prescription_features["num_unique_antipsychotics"] = 0
    prescription_features["num_unique_antipsychotics"] = prescription_features["num_unique_antipsychotics"].fillna(0).astype(int)
    for col in ["had_antibiotic_exposure", "had_opioid_exposure", "had_steroid_exposure", "had_iv_prescription",
            "had_long_acting_injectable_antipsychotic", "had_benzodiazepine_order", "had_mood_stabiliser_order",
            "new_antipsychotic_started_flag", "new_antidepressant_started_flag", "new_benzodiazepine_started_flag",
            "new_mood_stabiliser_started_flag", "psychotropic_started_after_72h_flag", "late_psychotropic_change_flag"]:
        prescription_features[col] = (prescription_features[col] > 0).astype(int)
    prescription_features["antipsychotic_polypharmacy_2plus"] = (prescription_features["num_unique_antipsychotics"] >= 2).astype(int)
    prescription_features["antipsychotic_polypharmacy_3plus"] = (prescription_features["num_unique_antipsychotics"] >= 3).astype(int)
    prescription_features["antipsychotic_plus_benzodiazepine"] = ((prescription_features["antipsychotic_prescription_count"] > 0) &
        (prescription_features["had_benzodiazepine_order"] == 1)).astype(int)
    prescription_features["antipsychotic_plus_mood_stabiliser"] = ((prescription_features["antipsychotic_prescription_count"] > 0) &
        (prescription_features["had_mood_stabiliser_order"] == 1)).astype(int)
else:
    prescription_features = pd.DataFrame(columns=id_cols + ["num_unique_antipsychotics", "antipsychotic_prescription_count",
        "had_long_acting_injectable_antipsychotic", "had_benzodiazepine_order", "had_mood_stabiliser_order",
        "new_antipsychotic_started_flag", "new_antidepressant_started_flag", "new_mood_stabiliser_started_flag",
        "new_benzodiazepine_started_flag", "psychotropic_medication_change_count", "psychotropic_started_after_72h_flag",
        "late_psychotropic_change_flag", "antipsychotic_polypharmacy_2plus", "antipsychotic_polypharmacy_3plus", "antipsychotic_plus_benzodiazepine",
        "antipsychotic_plus_mood_stabiliser"])

df = df.drop(columns=[col for col in prescription_features.columns if col not in id_cols], errors="ignore")
#join the derived features back to the admission table
df = df.merge(prescription_features, on=id_cols, how="left")
prescription_engineered_cols = [col for col in prescription_features.columns if col not in id_cols]
for col in prescription_engineered_cols:
    df[col] = df[col].fillna(0).astype(int)

Processed prescription chunk: 0 rows kept: 955387
Processed prescription chunk: 1 rows kept: 948987
Processed prescription chunk: 2 rows kept: 945184
Processed prescription chunk: 3 rows kept: 971249
Processed prescription chunk: 4 rows kept: 929093
Processed prescription chunk: 5 rows kept: 927793
Processed prescription chunk: 6 rows kept: 946102
Processed prescription chunk: 7 rows kept: 941712
Processed prescription chunk: 8 rows kept: 943894
Processed prescription chunk: 9 rows kept: 939210
Processed prescription chunk: 10 rows kept: 140938


**Create POE order acuity, discharge-readiness, and aftercare features**

This block uses POE order timing, order type/subtype, transaction status, and broad text patterns to describe the care pathway during the index admission. The aim is not to add raw order IDs or provider IDs, but to convert them into interpretable signals: how active care was near discharge, whether late activity looked like medication/lab/imaging/consult work, whether safety observation or suicide/elopement precautions appeared, and whether discharge planning/follow-up/placement support was documented.

These table-derived columns are especially important because the rapid-trial SHAP results repeatedly showed that late order activity and care-team/order pathway features separate high-risk from low-risk admissions. Counts and flags are derived only from events occurring during the current admission up to discharge.

In [ ]:
#poe feature setup
#define expected poe feature names, input paths, chunk buffers, and admission timestamps.
#this cell scans poe.csv.gz in chunks and creates admission-level order counts, timing windows,
#late-order categories, and discharge-readiness proxies without retaining raw order ids.
print("Starting POE order acuity and timing feature extraction...")

poe_engineered_cols = ["num_orders_first_6h", "num_orders_first_12h", "num_orders_first_24h", "num_orders_first_72h",
    "orders_24_to_72h", "orders_after_72h_count", "order_activity_slope_24_to_72h",
    "first_order_hours_from_admission", "last_order_hours_from_admission", "num_total_poe_orders",
    "num_unique_order_types", "num_unique_order_subtypes", "num_unique_poe_order_categories",
    "num_unique_order_providers", "num_unique_order_providers_first_24h",
    "num_unique_order_providers_first_72h", "num_discontinued_orders", "num_provider_order_changes",
    "num_psych_related_poe_orders", "num_consult_orders", "num_social_work_case_management_orders",
    "num_observation_safety_orders", "num_discharge_planning_orders", "num_followup_referral_orders",
    "had_sitter_or_constant_observation_order", "had_suicide_precaution_order", "had_elopement_precaution_order",
    "had_restraint_order", "had_psych_consult_order", "had_social_work_case_management_order",
    "had_followup_or_outpatient_referral_order", "had_discharge_planning_order", "first_discharge_planning_order_hours",
    "last_discharge_planning_order_hours", "discharge_planning_first_72h_flag",
    "late_discharge_planning_after_72h_flag", "had_psych_followup_order", "had_outpatient_followup_order",
    "had_clinic_referral_order", "had_case_management_order", "num_followup_related_orders",
    "followup_order_first_72h_flag", "late_followup_order_flag", "had_sitter_order",
    "had_constant_observation_order", "had_behavioral_observation_order", "orders_last_24h_before_discharge",
    "orders_last_48h_before_discharge", "orders_last_12h_before_discharge", "orders_last_6h_before_discharge",
    "new_orders_last_24h_before_discharge_flag", "new_orders_last_12h_before_discharge_flag",
    "order_activity_last_24h_to_total_ratio", "order_activity_last_48h_to_total_ratio",
    "order_activity_last_12h_to_total_ratio", "order_activity_last_6h_to_total_ratio",
    "last_order_within_6h_of_discharge_flag", "last_order_within_12h_of_discharge_flag",
    "last_order_within_24h_of_discharge_flag", "last_order_close_to_discharge_hours", "order_activity_duration_hours",
    "safety_orders_first_24h", "safety_orders_first_72h", "safety_orders_after_72h",
    "safety_orders_last_24h_before_discharge", "safety_order_density_per_day", "late_safety_order_flag",
    "safety_order_near_discharge_flag", "self_harm_and_late_safety_order_flag",
    "care_team_touchpoints_after_72h", "care_team_touchpoints_last_24h_before_discharge",
    "care_team_touchpoint_density_per_day", "late_care_team_activity_flag",
    "high_care_team_touchpoints_near_discharge_flag", "had_ciWA_protocol_order", "had_alcohol_withdrawal_protocol_order",
    "late_medication_orders_last_12h", "late_lab_orders_last_12h", "late_imaging_orders_last_12h",
    "late_consult_orders_last_12h", "late_discharge_admin_orders_last_12h",
    "late_routine_care_orders_last_12h", "late_psych_safety_orders_last_12h",
    "late_social_work_or_case_management_orders_last_12h", "late_followup_or_aftercare_orders_last_12h",
    "late_medication_orders_last_6h", "late_lab_orders_last_6h", "late_imaging_orders_last_6h",
    "late_consult_orders_last_6h", "late_discharge_admin_orders_last_6h",
    "late_routine_care_orders_last_6h", "late_psych_safety_orders_last_6h",
    "late_social_work_or_case_management_orders_last_6h", "late_followup_or_aftercare_orders_last_6h",
    "discontinued_orders_last_24h_before_discharge", "discontinued_orders_last_12h_before_discharge",
    "cancelled_orders_last_24h_before_discharge", "late_discontinued_medication_orders",
    "late_discontinued_lab_orders", "late_discontinued_safety_orders", "placement_related_order_flag",
    "placement_related_orders_last_48h", "awaiting_bed_or_placement_flag", "delayed_discharge_planning_flag",
    "transport_delay_order_flag", "social_barrier_order_flag", "discharge_medication_order_flag",
    "meds_to_beds_order_flag", "had_specific_psych_followup_order", "had_outpatient_psychiatry_appointment_order",
    "had_therapy_or_counselling_followup_order", "had_primary_care_followup_order",
    "had_substance_use_followup_order", "had_home_health_or_vna_order",
    "had_social_work_discharge_plan_order", "had_snf_or_rehab_placement_order",
    "had_transport_arranged_order", "followup_order_last_48h_before_discharge",
    "followup_order_last_24h_before_discharge", "safety_orders_last_12h_before_discharge",
    "safety_orders_last_6h_before_discharge", "new_safety_order_last_24h_before_discharge_flag",
    "new_safety_order_last_12h_before_discharge_flag", "first_safety_order_hours_from_admission",
    "last_safety_order_hours_from_admission", "safety_order_activity_duration_hours"]

poe_path = hosp_path / "poe.csv.gz"
poe_detail_path = hosp_path / "poe_detail.csv.gz"
poe_feature_rows = []
poe_type_rows = []
poe_subtype_rows = []
poe_provider_rows = []
poe_provider_sequence_rows = []
poe_category_rows = []
poe_key_rows = []
admission_times_for_orders = current_admissions[["subject_id", "hadm_id", "admittime", "dischtime"]].copy()

Starting POE order acuity and timing feature extraction...


**Scan POE orders in chunks**

The POE table is too large to load casually, so this section reads it in chunks and keeps only orders from the index admission window. It derives timing features, late-order counts, safety/order-status flags, follow-up patterns, cancellation/discontinuation markers, and broad order meaning groups.

This stays in T2.1 because POE features require raw order timestamps and order text/type fields. The main reason for adding them is that late activity close to discharge can reflect unresolved care, discharge coordination, safety management, or routine administrative work.


In [ ]:
#poe chunk scan
#read poe.csv.gz in chunks, restrict to current admissions, and derive row-level order timing/type flags
#before collapsing each chunk to one row per admission.
if poe_path.exists():
    poe_usecols = ["poe_id", "poe_seq", "subject_id", "hadm_id", "ordertime", "order_type", "order_subtype",
        "transaction_type", "discontinue_of_poe_id", "discontinued_by_poe_id", "order_provider_id", "order_status"]
    #loop through each item in this feature block
    for chunk_number, poe_chunk in enumerate(pd.read_csv(poe_path, usecols=poe_usecols, chunksize=1000000, low_memory=False)):
        poe_chunk = poe_chunk[poe_chunk["hadm_id"].isin(current_hadm_ids)].copy()
        if poe_chunk.empty:
            continue
        poe_chunk["ordertime"] = pd.to_datetime(poe_chunk["ordertime"], errors="coerce")
        #join the derived features back to the admission table
        poe_chunk = poe_chunk.merge(admission_times_for_orders, on=["subject_id", "hadm_id"], how="left")
        poe_chunk = poe_chunk[(poe_chunk["ordertime"].notna()) & (poe_chunk["admittime"].notna()) &
            (poe_chunk["ordertime"] >= poe_chunk["admittime"]) &
            ((poe_chunk["dischtime"].isna()) | (poe_chunk["ordertime"] <= poe_chunk["dischtime"]))].copy()
        if poe_chunk.empty:
            continue


        #timing windows: early admission, after 72h, and close to discharge
        hours_from_admission = (poe_chunk["ordertime"] - poe_chunk["admittime"]).dt.total_seconds() / 3600
        hours_to_discharge = (poe_chunk["dischtime"] - poe_chunk["ordertime"]).dt.total_seconds() / 3600
        order_text = (poe_chunk["order_type"].fillna("") + " " + poe_chunk["order_subtype"].fillna("") + " " +
            poe_chunk["transaction_type"].fillna("") + " " + poe_chunk["order_status"].fillna("")).str.lower()
        poe_chunk["order_first_6h"] = ((hours_from_admission >= 0) & (hours_from_admission <= 6)).astype(int)
        poe_chunk["order_first_12h"] = ((hours_from_admission >= 0) & (hours_from_admission <= 12)).astype(int)
        poe_chunk["order_first_24h"] = ((hours_from_admission >= 0) & (hours_from_admission <= 24)).astype(int)
        poe_chunk["order_first_72h"] = ((hours_from_admission >= 0) & (hours_from_admission <= 72)).astype(int)
        poe_chunk["order_24_to_72h"] = ((hours_from_admission > 24) & (hours_from_admission <= 72)).astype(int)
        poe_chunk["order_after_72h"] = (hours_from_admission > 72).astype(int)
        poe_chunk["order_hours_from_admission"] = hours_from_admission
        poe_chunk["order_last_24h_before_discharge"] = ((hours_to_discharge >= 0) & (hours_to_discharge <= 24)).astype(int)
        poe_chunk["order_last_48h_before_discharge"] = ((hours_to_discharge >= 0) & (hours_to_discharge <= 48)).astype(int)
        poe_chunk["order_last_12h_before_discharge"] = ((hours_to_discharge >= 0) & (hours_to_discharge <= 12)).astype(int)
        poe_chunk["order_last_6h_before_discharge"] = ((hours_to_discharge >= 0) & (hours_to_discharge <= 6)).astype(int)
        poe_chunk["order_hours_to_discharge"] = hours_to_discharge
        poe_chunk["order_category_text"] = (poe_chunk["order_type"].fillna("Unknown").astype(str).str.strip() + " / " +
            poe_chunk["order_subtype"].fillna("Unknown").astype(str).str.strip())

        #interpretable order categories from order type/subtype/status text
        poe_chunk["discontinued_order"] = (order_text.str.contains("discontinue|discontinued|cancel|inactive|void", regex=True) |
            poe_chunk["discontinue_of_poe_id"].notna() | poe_chunk["discontinued_by_poe_id"].notna()).astype(int)
        poe_chunk["psych_related_poe_order"] = order_text.str.contains("psych|psychiat|behavioral health|behavioural health|suicid|self harm|self-harm", regex=True).astype(int)
        poe_chunk["consult_order"] = order_text.str.contains("consult|consultation|referral", regex=True).astype(int)
        poe_chunk["social_work_case_management_order"] = order_text.str.contains("social work|case management|care coordination", regex=True).astype(int)
        poe_chunk["sitter_order"] = order_text.str.contains("sitter|patient observer|safety watch", regex=True).astype(int)
        poe_chunk["constant_observation_order"] = order_text.str.contains("constant observation|1:1|one to one|special observation", regex=True).astype(int)
        poe_chunk["suicide_precaution_order"] = order_text.str.contains("suicide precaution|suicidal|suicide|self harm|self-harm", regex=True).astype(int)
        poe_chunk["elopement_precaution_order"] = order_text.str.contains("elopement|elope|flight risk|wandering", regex=True).astype(int)
        poe_chunk["behavioral_observation_order"] = order_text.str.contains("behavioral observation|behavioural observation|agitation|violent|assaultive", regex=True).astype(int)
        poe_chunk["observation_safety_order"] = ((poe_chunk["sitter_order"] == 1) | (poe_chunk["constant_observation_order"] == 1) |
            (poe_chunk["suicide_precaution_order"] == 1) | (poe_chunk["elopement_precaution_order"] == 1) |
            (poe_chunk["behavioral_observation_order"] == 1) | order_text.str.contains("restraint|seclusion|security|combative|escape risk|si precautions", regex=True)).astype(int)
        poe_chunk["safety_order_first_24h"] = ((poe_chunk["observation_safety_order"] == 1) & (hours_from_admission <= 24)).astype(int)
        poe_chunk["safety_order_first_72h"] = ((poe_chunk["observation_safety_order"] == 1) & (hours_from_admission <= 72)).astype(int)
        poe_chunk["safety_order_after_72h"] = ((poe_chunk["observation_safety_order"] == 1) & (hours_from_admission > 72)).astype(int)
        poe_chunk["safety_order_last_24h_before_discharge"] = ((poe_chunk["observation_safety_order"] == 1) &
            (poe_chunk["order_last_24h_before_discharge"] == 1)).astype(int)
        poe_chunk["discharge_planning_order"] = order_text.str.contains("discharge planning|discharge plan|discharge instruction|aftercare|placement|home health|rehab|skilled nursing|home care", regex=True).astype(int)
        poe_chunk["psych_followup_order"] = order_text.str.contains("psych.*follow|psychiat.*follow|behavioral health.*follow|behavioural health.*follow", regex=True).astype(int)
        poe_chunk["outpatient_followup_order"] = order_text.str.contains("outpatient|ambulatory|follow up|follow-up|followup|aftercare", regex=True).astype(int)
        poe_chunk["clinic_referral_order"] = order_text.str.contains("clinic|referral|appointment", regex=True).astype(int)
        poe_chunk["case_management_order"] = order_text.str.contains("case management|social work|care coordination", regex=True).astype(int)
        poe_chunk["followup_referral_order"] = ((poe_chunk["psych_followup_order"] == 1) | (poe_chunk["outpatient_followup_order"] == 1) |
            (poe_chunk["clinic_referral_order"] == 1) | (poe_chunk["case_management_order"] == 1)).astype(int)
        poe_chunk["ciwa_protocol_order"] = order_text.str.contains("ciwa|withdrawal protocol|alcohol withdrawal", regex=True).astype(int)
        poe_chunk["alcohol_withdrawal_protocol_order"] = order_text.str.contains("alcohol withdrawal|ciwa", regex=True).astype(int)
        #discharge friction, aftercare, and medication-readiness text patterns
        poe_chunk["placement_related_order"] = order_text.str.contains(
            "placement|awaiting bed|bed search|skilled nursing|snf|rehab|residential|group home|shelter|housing", regex=True, na=False).astype(int)
        poe_chunk["awaiting_bed_or_placement_order"] = order_text.str.contains(
            "awaiting.*bed|bed.*awaiting|bed search|placement pending|awaiting placement|pending placement", regex=True, na=False).astype(int)
        poe_chunk["delayed_discharge_planning_order"] = order_text.str.contains(
            "delayed discharge|discharge delay|barrier to discharge|discharge barrier|insurance auth|authorization|authorisation", regex=True, na=False).astype(int)
        poe_chunk["transport_delay_order"] = order_text.str.contains(
            "transport|ambulance|chair car|taxi|ride|escort", regex=True, na=False).astype(int)
        poe_chunk["social_barrier_order"] = order_text.str.contains(
            "housing|homeless|shelter|social barrier|financial|insurance|guardian|conservator|family meeting", regex=True, na=False).astype(int)
        poe_chunk["discharge_medication_order"] = order_text.str.contains(
            "discharge.*med|med.*discharge|prescription.*discharge|meds to beds|meds-to-beds|bedside delivery", regex=True, na=False).astype(int)
        poe_chunk["meds_to_beds_order"] = order_text.str.contains("meds to beds|meds-to-beds|bedside delivery", regex=True, na=False).astype(int)
        poe_chunk["specific_psych_followup_order"] = order_text.str.contains(
            "psychiatry.*appointment|psychiatric.*appointment|psych.*clinic|behavioral health.*appointment|behavioural health.*appointment", regex=True, na=False).astype(int)
        poe_chunk["outpatient_psychiatry_appointment_order"] = order_text.str.contains(
            "outpatient psychiatry|psychiatry outpatient|psych outpatient|psychiatry clinic|psych clinic", regex=True, na=False).astype(int)
        poe_chunk["therapy_or_counselling_followup_order"] = order_text.str.contains(
            "therapy|therapist|counseling|counselling|counsellor|counselor", regex=True, na=False).astype(int)
        poe_chunk["primary_care_followup_order"] = order_text.str.contains("primary care|pcp|gp follow|general practitioner", regex=True, na=False).astype(int)
        poe_chunk["substance_use_followup_order"] = order_text.str.contains(
            "substance use|addiction|detox|methadone clinic|buprenorphine clinic|alcohol clinic", regex=True, na=False).astype(int)
        poe_chunk["home_health_or_vna_order"] = order_text.str.contains("home health|vna|visiting nurse|home nursing", regex=True, na=False).astype(int)
        poe_chunk["social_work_discharge_plan_order"] = ((poe_chunk["social_work_case_management_order"] == 1) &
            (poe_chunk["discharge_planning_order"] == 1)).astype(int)
        poe_chunk["snf_or_rehab_placement_order"] = order_text.str.contains("skilled nursing|snf|rehab|rehabilitation", regex=True, na=False).astype(int)
        poe_chunk["transport_arranged_order"] = order_text.str.contains("transport arranged|ambulance booked|chair car|ride arranged", regex=True, na=False).astype(int)
        poe_chunk["cancelled_order"] = order_text.str.contains("cancel|cancelled|canceled|void", regex=True, na=False).astype(int)

        poe_chunk["had_psych_consult_order"] = poe_chunk["psych_related_poe_order"]
        poe_chunk["had_social_work_case_management_order"] = poe_chunk["social_work_case_management_order"]
        poe_chunk["had_followup_or_outpatient_referral_order"] = poe_chunk["followup_referral_order"]
        poe_chunk["had_discharge_planning_order"] = poe_chunk["discharge_planning_order"]
        poe_chunk["had_psych_followup_order"] = poe_chunk["psych_followup_order"]
        poe_chunk["had_outpatient_followup_order"] = poe_chunk["outpatient_followup_order"]
        poe_chunk["had_clinic_referral_order"] = poe_chunk["clinic_referral_order"]
        poe_chunk["had_case_management_order"] = poe_chunk["case_management_order"]
        poe_chunk["had_sitter_order"] = poe_chunk["sitter_order"]
        poe_chunk["had_constant_observation_order"] = poe_chunk["constant_observation_order"]
        poe_chunk["had_behavioral_observation_order"] = poe_chunk["behavioral_observation_order"]
        poe_chunk["had_ciWA_protocol_order"] = poe_chunk["ciwa_protocol_order"]
        poe_chunk["had_alcohol_withdrawal_protocol_order"] = poe_chunk["alcohol_withdrawal_protocol_order"]

        poe_chunk["discontinued_order_last_24h_before_discharge"] = ((poe_chunk["discontinued_order"] == 1) &
            (poe_chunk["order_last_24h_before_discharge"] == 1)).astype(int)
        poe_chunk["discontinued_order_last_12h_before_discharge"] = ((poe_chunk["discontinued_order"] == 1) &
            (poe_chunk["order_last_12h_before_discharge"] == 1)).astype(int)
        poe_chunk["cancelled_order_last_24h_before_discharge"] = ((poe_chunk["cancelled_order"] == 1) &
            (poe_chunk["order_last_24h_before_discharge"] == 1)).astype(int)
        poe_chunk["late_discontinued_medication_order"] = ((poe_chunk["discontinued_order"] == 1) &
            (poe_chunk["order_last_24h_before_discharge"] == 1) &
            order_text.str.contains("medication|pharmacy|drug|dose|antipsychotic|benzodiazepine|antidepressant|mood stabil", regex=True, na=False)).astype(int)
        poe_chunk["late_discontinued_lab_order"] = ((poe_chunk["discontinued_order"] == 1) &
            (poe_chunk["order_last_24h_before_discharge"] == 1) &
            order_text.str.contains("lab|laboratory|blood|cbc|chemistry|culture|specimen|urine", regex=True, na=False)).astype(int)
        poe_chunk["late_discontinued_safety_order"] = ((poe_chunk["discontinued_order"] == 1) &
            (poe_chunk["order_last_24h_before_discharge"] == 1) &
            (poe_chunk["observation_safety_order"] == 1)).astype(int)
        poe_chunk["placement_related_order_last_48h"] = ((poe_chunk["placement_related_order"] == 1) &
            (poe_chunk["order_last_48h_before_discharge"] == 1)).astype(int)
        poe_chunk["followup_order_last_48h_before_discharge"] = ((poe_chunk["followup_referral_order"] == 1) &
            (poe_chunk["order_last_48h_before_discharge"] == 1)).astype(int)
        poe_chunk["followup_order_last_24h_before_discharge"] = ((poe_chunk["followup_referral_order"] == 1) &
            (poe_chunk["order_last_24h_before_discharge"] == 1)).astype(int)
        poe_chunk["safety_order_last_12h_before_discharge"] = ((poe_chunk["observation_safety_order"] == 1) &
            (poe_chunk["order_last_12h_before_discharge"] == 1)).astype(int)
        poe_chunk["safety_order_last_6h_before_discharge"] = ((poe_chunk["observation_safety_order"] == 1) &
            (poe_chunk["order_last_6h_before_discharge"] == 1)).astype(int)

        #split very late poe activity into broad order types so true risk and routine discharge activity are not treated identically.
        late_order_type_patterns = {
            "medication": "medication|pharmacy|drug|dose|administer|antipsychotic|benzodiazepine|antidepressant|mood stabil|opioid|naloxone|methadone|buprenorphine",
            "lab": "lab|laboratory|blood|cbc|chemistry|metabolic|electrolyte|culture|specimen|urine|wbc|hemoglobin|platelet|sodium|potassium|creatinine|glucose",
            "imaging": "xray|x-ray|radiology|ct |ct scan|mri|ultrasound|echo|imaging|cxr",
            "consult": "consult|consultation|referral|psychiat|psych consult|social work|case management|care coordination",
            "discharge_admin": "discharge|aftercare|follow up|follow-up|followup|appointment|outpatient|clinic|home health|vna|placement|rehab|skilled nursing|snf",
            "routine_care": "diet|activity|nursing|vital|vitals|monitor|telemetry|precaution|fall|turn|ambulate|wound|line care",
            "psych_safety": "sitter|1:1|one to one|constant observation|special observation|suicide|suicidal|self harm|self-harm|elopement|elope|safety watch|patient observer|behavioral|behavioural|agitation|restraint|seclusion|security|combative",
            "social_work_or_case_management": "social work|case management|care coordination",
            "followup_or_aftercare": "follow up|follow-up|followup|aftercare|appointment|outpatient|clinic|referral|home health|vna|visiting nurse"}
        
        #loop through each item in this feature block
        for order_group, pattern in late_order_type_patterns.items():
            order_group_match = order_text.str.contains(pattern, regex=True, na=False)
            poe_chunk[f"late_{order_group}_orders_last_12h"] = ((poe_chunk["order_last_12h_before_discharge"] == 1) & order_group_match).astype(int)
            poe_chunk[f"late_{order_group}_orders_last_6h"] = ((poe_chunk["order_last_6h_before_discharge"] == 1) & order_group_match).astype(int)

        #collapse this chunk to admission-level counts and flags
        poe_feature_rows.append(poe_chunk.groupby(id_cols, as_index=False).agg(
            num_total_poe_orders=("poe_id", "count"),
            num_orders_first_6h=("order_first_6h", "sum"),
            num_orders_first_12h=("order_first_12h", "sum"),
            num_orders_first_24h=("order_first_24h", "sum"),
            num_orders_first_72h=("order_first_72h", "sum"),
            orders_24_to_72h=("order_24_to_72h", "sum"),
            orders_after_72h_count=("order_after_72h", "sum"),
            first_order_hours_from_admission=("order_hours_from_admission", "min"),
            last_order_hours_from_admission=("order_hours_from_admission", "max"),
            orders_last_24h_before_discharge=("order_last_24h_before_discharge", "sum"),
            orders_last_48h_before_discharge=("order_last_48h_before_discharge", "sum"),
            orders_last_12h_before_discharge=("order_last_12h_before_discharge", "sum"),
            orders_last_6h_before_discharge=("order_last_6h_before_discharge", "sum"),
            last_order_close_to_discharge_hours=("order_hours_to_discharge", "min"),
            num_discontinued_orders=("discontinued_order", "sum"),
            discontinued_orders_last_24h_before_discharge=("discontinued_order_last_24h_before_discharge", "sum"),
            discontinued_orders_last_12h_before_discharge=("discontinued_order_last_12h_before_discharge", "sum"),
            cancelled_orders_last_24h_before_discharge=("cancelled_order_last_24h_before_discharge", "sum"),
            late_discontinued_medication_orders=("late_discontinued_medication_order", "sum"),
            late_discontinued_lab_orders=("late_discontinued_lab_order", "sum"),
            late_discontinued_safety_orders=("late_discontinued_safety_order", "sum"),
            num_psych_related_poe_orders=("psych_related_poe_order", "sum"),
            num_consult_orders=("consult_order", "sum"),
            num_social_work_case_management_orders=("social_work_case_management_order", "sum"),
            num_observation_safety_orders=("observation_safety_order", "sum"),
            safety_orders_first_24h=("safety_order_first_24h", "sum"),
            safety_orders_first_72h=("safety_order_first_72h", "sum"),
            safety_orders_after_72h=("safety_order_after_72h", "sum"),
            safety_orders_last_24h_before_discharge=("safety_order_last_24h_before_discharge", "sum"),
            num_discharge_planning_orders=("discharge_planning_order", "sum"),
            num_followup_referral_orders=("followup_referral_order", "sum"),
            num_followup_related_orders=("followup_referral_order", "sum"),
            first_discharge_planning_order_hours=("order_hours_from_admission", lambda x: x[poe_chunk.loc[x.index, "discharge_planning_order"] == 1].min()),
            last_discharge_planning_order_hours=("order_hours_from_admission", lambda x: x[poe_chunk.loc[x.index, "discharge_planning_order"] == 1].max()),
            discharge_planning_first_72h_flag=("discharge_planning_order", lambda x: int(((x == 1) & (hours_from_admission.loc[x.index] <= 72)).any())),
            late_discharge_planning_after_72h_flag=("discharge_planning_order", lambda x: int(((x == 1) & (hours_from_admission.loc[x.index] > 72)).any())),
            followup_order_first_72h_flag=("followup_referral_order", lambda x: int(((x == 1) & (hours_from_admission.loc[x.index] <= 72)).any())),
            late_followup_order_flag=("followup_referral_order", lambda x: int(((x == 1) & (hours_from_admission.loc[x.index] > 72)).any())),
            had_psych_followup_order=("had_psych_followup_order", "max"),
            had_outpatient_followup_order=("had_outpatient_followup_order", "max"),
            had_clinic_referral_order=("had_clinic_referral_order", "max"),
            had_case_management_order=("had_case_management_order", "max"),
            had_sitter_order=("had_sitter_order", "max"),
            had_constant_observation_order=("had_constant_observation_order", "max"),
            had_behavioral_observation_order=("had_behavioral_observation_order", "max"),
            had_ciWA_protocol_order=("had_ciWA_protocol_order", "max"),
            had_alcohol_withdrawal_protocol_order=("had_alcohol_withdrawal_protocol_order", "max"),
            placement_related_order_flag=("placement_related_order", "max"),
            placement_related_orders_last_48h=("placement_related_order_last_48h", "sum"),
            awaiting_bed_or_placement_flag=("awaiting_bed_or_placement_order", "max"),
            delayed_discharge_planning_flag=("delayed_discharge_planning_order", "max"),
            transport_delay_order_flag=("transport_delay_order", "max"),
            social_barrier_order_flag=("social_barrier_order", "max"),
            discharge_medication_order_flag=("discharge_medication_order", "max"),
            meds_to_beds_order_flag=("meds_to_beds_order", "max"),
            had_specific_psych_followup_order=("specific_psych_followup_order", "max"),
            had_outpatient_psychiatry_appointment_order=("outpatient_psychiatry_appointment_order", "max"),
            had_therapy_or_counselling_followup_order=("therapy_or_counselling_followup_order", "max"),
            had_primary_care_followup_order=("primary_care_followup_order", "max"),
            had_substance_use_followup_order=("substance_use_followup_order", "max"),
            had_home_health_or_vna_order=("home_health_or_vna_order", "max"),
            had_social_work_discharge_plan_order=("social_work_discharge_plan_order", "max"),
            had_snf_or_rehab_placement_order=("snf_or_rehab_placement_order", "max"),
            had_transport_arranged_order=("transport_arranged_order", "max"),
            followup_order_last_48h_before_discharge=("followup_order_last_48h_before_discharge", "sum"),
            followup_order_last_24h_before_discharge=("followup_order_last_24h_before_discharge", "sum"),
            safety_orders_last_12h_before_discharge=("safety_order_last_12h_before_discharge", "sum"),
            safety_orders_last_6h_before_discharge=("safety_order_last_6h_before_discharge", "sum"),
            first_safety_order_hours_from_admission=("order_hours_from_admission", lambda x: x[poe_chunk.loc[x.index, "observation_safety_order"] == 1].min()),
            last_safety_order_hours_from_admission=("order_hours_from_admission", lambda x: x[poe_chunk.loc[x.index, "observation_safety_order"] == 1].max()),
            had_psych_consult_order=("had_psych_consult_order", "max"),
            had_social_work_case_management_order=("had_social_work_case_management_order", "max"),
            had_followup_or_outpatient_referral_order=("had_followup_or_outpatient_referral_order", "max"),
            had_discharge_planning_order=("had_discharge_planning_order", "max"),
            late_medication_orders_last_12h=("late_medication_orders_last_12h", "sum"),
            late_lab_orders_last_12h=("late_lab_orders_last_12h", "sum"),
            late_imaging_orders_last_12h=("late_imaging_orders_last_12h", "sum"),
            late_consult_orders_last_12h=("late_consult_orders_last_12h", "sum"),
            late_discharge_admin_orders_last_12h=("late_discharge_admin_orders_last_12h", "sum"),
            late_routine_care_orders_last_12h=("late_routine_care_orders_last_12h", "sum"),
            late_psych_safety_orders_last_12h=("late_psych_safety_orders_last_12h", "sum"),
            late_social_work_or_case_management_orders_last_12h=("late_social_work_or_case_management_orders_last_12h", "sum"),
            late_followup_or_aftercare_orders_last_12h=("late_followup_or_aftercare_orders_last_12h", "sum"),
            late_medication_orders_last_6h=("late_medication_orders_last_6h", "sum"),
            late_lab_orders_last_6h=("late_lab_orders_last_6h", "sum"),
            late_imaging_orders_last_6h=("late_imaging_orders_last_6h", "sum"),
            late_consult_orders_last_6h=("late_consult_orders_last_6h", "sum"),
            late_discharge_admin_orders_last_6h=("late_discharge_admin_orders_last_6h", "sum"),
            late_routine_care_orders_last_6h=("late_routine_care_orders_last_6h", "sum"),
            late_psych_safety_orders_last_6h=("late_psych_safety_orders_last_6h", "sum"),
            late_social_work_or_case_management_orders_last_6h=("late_social_work_or_case_management_orders_last_6h", "sum"),
            late_followup_or_aftercare_orders_last_6h=("late_followup_or_aftercare_orders_last_6h", "sum")))
        poe_type_rows.append(poe_chunk[id_cols + ["order_type"]].dropna(subset=["order_type"]).drop_duplicates())
        poe_subtype_rows.append(poe_chunk[id_cols + ["order_subtype"]].dropna(subset=["order_subtype"]).drop_duplicates())
        poe_category_rows.append(poe_chunk[id_cols + ["order_category_text"]].dropna(subset=["order_category_text"]).drop_duplicates())
        poe_provider_rows.append(poe_chunk[id_cols + ["order_provider_id"]].dropna(subset=["order_provider_id"]).drop_duplicates())
        poe_provider_sequence_rows.append(poe_chunk[id_cols + ["ordertime", "order_provider_id"]].dropna(subset=["order_provider_id"]).copy())
        poe_key_rows.append(poe_chunk[["subject_id", "hadm_id", "poe_id", "poe_seq"]].drop_duplicates())
        print("Processed POE chunk:", chunk_number, "rows kept:", len(poe_chunk))

Processed POE chunk: 0 rows kept: 442514
Processed POE chunk: 1 rows kept: 410840
Processed POE chunk: 2 rows kept: 439985
Processed POE chunk: 3 rows kept: 427702
Processed POE chunk: 4 rows kept: 455040
Processed POE chunk: 5 rows kept: 417298
Processed POE chunk: 6 rows kept: 456165
Processed POE chunk: 7 rows kept: 433968
Processed POE chunk: 8 rows kept: 423652
Processed POE chunk: 9 rows kept: 411410
Processed POE chunk: 10 rows kept: 443605
Processed POE chunk: 11 rows kept: 425826
Processed POE chunk: 12 rows kept: 427172
Processed POE chunk: 13 rows kept: 424038
Processed POE chunk: 14 rows kept: 421769
Processed POE chunk: 15 rows kept: 430159
Processed POE chunk: 16 rows kept: 445250
Processed POE chunk: 17 rows kept: 457836
Processed POE chunk: 18 rows kept: 430068
Processed POE chunk: 19 rows kept: 426262
Processed POE chunk: 20 rows kept: 441122
Processed POE chunk: 21 rows kept: 408579
Processed POE chunk: 22 rows kept: 423762
Processed POE chunk: 23 rows kept: 428327
Pr

**Aggregate POE orders to one row per admission**

The chunk-level POE outputs are combined into one row per admission. This creates total order burden, early-versus-late activity, order-type diversity, discharge planning and aftercare indicators, and timing summaries such as the last order before discharge.

The purpose is to turn repeated order rows into stable admission-level predictors. Counts, shares, and flags are all kept because a single late order and a broad late-order pattern may carry different meaning.


In [ ]:
#poe admission-level aggregation
#combine chunk-level summaries into final admission-level poe features, then add unique order/provider counts and derived timing/rate variables.
if poe_feature_rows:
    #summarise row-level records to admission-level features
    poe_features = pd.concat(poe_feature_rows, ignore_index=True).groupby(id_cols, as_index=False).agg(
        num_total_poe_orders=("num_total_poe_orders", "sum"),
        num_orders_first_6h=("num_orders_first_6h", "sum"),
        num_orders_first_12h=("num_orders_first_12h", "sum"),
        num_orders_first_24h=("num_orders_first_24h", "sum"),
        num_orders_first_72h=("num_orders_first_72h", "sum"),
        orders_24_to_72h=("orders_24_to_72h", "sum"),
        orders_after_72h_count=("orders_after_72h_count", "sum"),
        first_order_hours_from_admission=("first_order_hours_from_admission", "min"),
        last_order_hours_from_admission=("last_order_hours_from_admission", "max"),
        orders_last_24h_before_discharge=("orders_last_24h_before_discharge", "sum"),
        orders_last_48h_before_discharge=("orders_last_48h_before_discharge", "sum"),
        orders_last_12h_before_discharge=("orders_last_12h_before_discharge", "sum"),
        orders_last_6h_before_discharge=("orders_last_6h_before_discharge", "sum"),
        last_order_close_to_discharge_hours=("last_order_close_to_discharge_hours", "min"),
        num_discontinued_orders=("num_discontinued_orders", "sum"),
        discontinued_orders_last_24h_before_discharge=("discontinued_orders_last_24h_before_discharge", "sum"),
        discontinued_orders_last_12h_before_discharge=("discontinued_orders_last_12h_before_discharge", "sum"),
        cancelled_orders_last_24h_before_discharge=("cancelled_orders_last_24h_before_discharge", "sum"),
        late_discontinued_medication_orders=("late_discontinued_medication_orders", "sum"),
        late_discontinued_lab_orders=("late_discontinued_lab_orders", "sum"),
        late_discontinued_safety_orders=("late_discontinued_safety_orders", "sum"),
        num_psych_related_poe_orders=("num_psych_related_poe_orders", "sum"),
        num_consult_orders=("num_consult_orders", "sum"),
        num_social_work_case_management_orders=("num_social_work_case_management_orders", "sum"),
        num_observation_safety_orders=("num_observation_safety_orders", "sum"),
        safety_orders_first_24h=("safety_orders_first_24h", "sum"),
        safety_orders_first_72h=("safety_orders_first_72h", "sum"),
        safety_orders_after_72h=("safety_orders_after_72h", "sum"),
        safety_orders_last_24h_before_discharge=("safety_orders_last_24h_before_discharge", "sum"),
        num_discharge_planning_orders=("num_discharge_planning_orders", "sum"),
        num_followup_referral_orders=("num_followup_referral_orders", "sum"),
        num_followup_related_orders=("num_followup_related_orders", "sum"),
        first_discharge_planning_order_hours=("first_discharge_planning_order_hours", "min"),
        last_discharge_planning_order_hours=("last_discharge_planning_order_hours", "max"),
        discharge_planning_first_72h_flag=("discharge_planning_first_72h_flag", "max"),
        late_discharge_planning_after_72h_flag=("late_discharge_planning_after_72h_flag", "max"),
        followup_order_first_72h_flag=("followup_order_first_72h_flag", "max"),
        late_followup_order_flag=("late_followup_order_flag", "max"),
        had_psych_followup_order=("had_psych_followup_order", "max"),
        had_outpatient_followup_order=("had_outpatient_followup_order", "max"),
        had_clinic_referral_order=("had_clinic_referral_order", "max"),
        had_case_management_order=("had_case_management_order", "max"),
        had_sitter_order=("had_sitter_order", "max"),
        had_constant_observation_order=("had_constant_observation_order", "max"),
        had_behavioral_observation_order=("had_behavioral_observation_order", "max"),
        had_ciWA_protocol_order=("had_ciWA_protocol_order", "max"),
        had_alcohol_withdrawal_protocol_order=("had_alcohol_withdrawal_protocol_order", "max"),
        placement_related_order_flag=("placement_related_order_flag", "max"),
        placement_related_orders_last_48h=("placement_related_orders_last_48h", "sum"),
        awaiting_bed_or_placement_flag=("awaiting_bed_or_placement_flag", "max"),
        delayed_discharge_planning_flag=("delayed_discharge_planning_flag", "max"),
        transport_delay_order_flag=("transport_delay_order_flag", "max"),
        social_barrier_order_flag=("social_barrier_order_flag", "max"),
        discharge_medication_order_flag=("discharge_medication_order_flag", "max"),
        meds_to_beds_order_flag=("meds_to_beds_order_flag", "max"),
        had_specific_psych_followup_order=("had_specific_psych_followup_order", "max"),
        had_outpatient_psychiatry_appointment_order=("had_outpatient_psychiatry_appointment_order", "max"),
        had_therapy_or_counselling_followup_order=("had_therapy_or_counselling_followup_order", "max"),
        had_primary_care_followup_order=("had_primary_care_followup_order", "max"),
        had_substance_use_followup_order=("had_substance_use_followup_order", "max"),
        had_home_health_or_vna_order=("had_home_health_or_vna_order", "max"),
        had_social_work_discharge_plan_order=("had_social_work_discharge_plan_order", "max"),
        had_snf_or_rehab_placement_order=("had_snf_or_rehab_placement_order", "max"),
        had_transport_arranged_order=("had_transport_arranged_order", "max"),
        followup_order_last_48h_before_discharge=("followup_order_last_48h_before_discharge", "sum"),
        followup_order_last_24h_before_discharge=("followup_order_last_24h_before_discharge", "sum"),
        safety_orders_last_12h_before_discharge=("safety_orders_last_12h_before_discharge", "sum"),
        safety_orders_last_6h_before_discharge=("safety_orders_last_6h_before_discharge", "sum"),
        first_safety_order_hours_from_admission=("first_safety_order_hours_from_admission", "min"),
        last_safety_order_hours_from_admission=("last_safety_order_hours_from_admission", "max"),
        had_psych_consult_order=("had_psych_consult_order", "max"),
        had_social_work_case_management_order=("had_social_work_case_management_order", "max"),
        had_followup_or_outpatient_referral_order=("had_followup_or_outpatient_referral_order", "max"),
        had_discharge_planning_order=("had_discharge_planning_order", "max"),
        late_medication_orders_last_12h=("late_medication_orders_last_12h", "sum"),
        late_lab_orders_last_12h=("late_lab_orders_last_12h", "sum"),
        late_imaging_orders_last_12h=("late_imaging_orders_last_12h", "sum"),
        late_consult_orders_last_12h=("late_consult_orders_last_12h", "sum"),
        late_discharge_admin_orders_last_12h=("late_discharge_admin_orders_last_12h", "sum"),
        late_routine_care_orders_last_12h=("late_routine_care_orders_last_12h", "sum"),
        late_psych_safety_orders_last_12h=("late_psych_safety_orders_last_12h", "sum"),
        late_social_work_or_case_management_orders_last_12h=("late_social_work_or_case_management_orders_last_12h", "sum"),
        late_followup_or_aftercare_orders_last_12h=("late_followup_or_aftercare_orders_last_12h", "sum"),
        late_medication_orders_last_6h=("late_medication_orders_last_6h", "sum"),
        late_lab_orders_last_6h=("late_lab_orders_last_6h", "sum"),
        late_imaging_orders_last_6h=("late_imaging_orders_last_6h", "sum"),
        late_consult_orders_last_6h=("late_consult_orders_last_6h", "sum"),
        late_discharge_admin_orders_last_6h=("late_discharge_admin_orders_last_6h", "sum"),
        late_routine_care_orders_last_6h=("late_routine_care_orders_last_6h", "sum"),
        late_psych_safety_orders_last_6h=("late_psych_safety_orders_last_6h", "sum"),
        late_social_work_or_case_management_orders_last_6h=("late_social_work_or_case_management_orders_last_6h", "sum"),
        late_followup_or_aftercare_orders_last_6h=("late_followup_or_aftercare_orders_last_6h", "sum"))

    #derived order trajectory and close-to-discharge activity metrics
    poe_features["order_activity_slope_24_to_72h"] = poe_features["orders_24_to_72h"] - poe_features["num_orders_first_24h"]
    poe_features["order_activity_duration_hours"] = (poe_features["last_order_hours_from_admission"] -
        poe_features["first_order_hours_from_admission"]).clip(lower=0)
    poe_features["new_orders_last_24h_before_discharge_flag"] = (poe_features["orders_last_24h_before_discharge"] > 0).astype(int)
    poe_features["new_orders_last_12h_before_discharge_flag"] = (poe_features["orders_last_12h_before_discharge"] > 0).astype(int)
    #loop through each item in this feature block
    for window_hours in [24, 48, 12, 6]:
        source_col = f"orders_last_{window_hours}h_before_discharge"
        ratio_col = f"order_activity_last_{window_hours}h_to_total_ratio"
        poe_features[ratio_col] = np.where(poe_features["num_total_poe_orders"] > 0,
            poe_features[source_col] / poe_features["num_total_poe_orders"], 0)
    poe_features["last_order_within_6h_of_discharge_flag"] = (poe_features["last_order_close_to_discharge_hours"] <= 6).fillna(False).astype(int)
    poe_features["last_order_within_12h_of_discharge_flag"] = (poe_features["last_order_close_to_discharge_hours"] <= 12).fillna(False).astype(int)
    poe_features["last_order_within_24h_of_discharge_flag"] = (poe_features["last_order_close_to_discharge_hours"] <= 24).fillna(False).astype(int)
    admission_los_days_for_poe = ((poe_features["last_order_hours_from_admission"].fillna(0) +
        poe_features["last_order_close_to_discharge_hours"].fillna(0)) / 24).clip(lower=1)
    poe_features["safety_order_density_per_day"] = (poe_features["num_observation_safety_orders"] / admission_los_days_for_poe).replace([np.inf, -np.inf], 0).fillna(0)
    poe_features["late_safety_order_flag"] = (poe_features["safety_orders_after_72h"] > 0).astype(int)
    poe_features["safety_order_near_discharge_flag"] = (poe_features["safety_orders_last_24h_before_discharge"] > 0).astype(int)
    poe_features["new_safety_order_last_24h_before_discharge_flag"] = (poe_features["safety_orders_last_24h_before_discharge"] > 0).astype(int)
    poe_features["new_safety_order_last_12h_before_discharge_flag"] = (poe_features["safety_orders_last_12h_before_discharge"] > 0).astype(int)
    poe_features["safety_order_activity_duration_hours"] = (poe_features["last_safety_order_hours_from_admission"] -
        poe_features["first_safety_order_hours_from_admission"]).clip(lower=0).fillna(0)
    poe_features["late_order_discontinuation_share"] = np.where(poe_features["orders_last_24h_before_discharge"] > 0,
        poe_features["discontinued_orders_last_24h_before_discharge"] / poe_features["orders_last_24h_before_discharge"], 0)

    #diversity counts: order types, subtypes, categories, and ordering providers
    if poe_type_rows:
        #summarise row-level records to admission-level features
        unique_order_types = pd.concat(poe_type_rows, ignore_index=True).drop_duplicates().groupby(id_cols, as_index=False)["order_type"].nunique()
        unique_order_types = unique_order_types.rename(columns={"order_type": "num_unique_order_types"})
        #join the derived features back to the admission table
        poe_features = poe_features.merge(unique_order_types, on=id_cols, how="left")
    if poe_subtype_rows:
        unique_order_subtypes = pd.concat(poe_subtype_rows, ignore_index=True).drop_duplicates().groupby(id_cols, as_index=False)["order_subtype"].nunique()
        unique_order_subtypes = unique_order_subtypes.rename(columns={"order_subtype": "num_unique_order_subtypes"})
        poe_features = poe_features.merge(unique_order_subtypes, on=id_cols, how="left")
    if poe_category_rows:
        unique_order_categories = pd.concat(poe_category_rows, ignore_index=True).drop_duplicates().groupby(id_cols, as_index=False)["order_category_text"].nunique()
        unique_order_categories = unique_order_categories.rename(columns={"order_category_text": "num_unique_poe_order_categories"})
        poe_features = poe_features.merge(unique_order_categories, on=id_cols, how="left")
        
    if poe_provider_rows:
        unique_order_providers = pd.concat(poe_provider_rows, ignore_index=True).drop_duplicates().groupby(id_cols, as_index=False)["order_provider_id"].nunique()
        unique_order_providers = unique_order_providers.rename(columns={"order_provider_id": "num_unique_order_providers"})
        poe_features = poe_features.merge(unique_order_providers, on=id_cols, how="left")

    #provider turnover and late care-team activity proxies
    if poe_provider_sequence_rows:
        poe_provider_timing = pd.concat(poe_provider_sequence_rows, ignore_index=True)
        poe_provider_timing = poe_provider_timing.merge(current_admissions[id_cols + ["admittime"]], on=id_cols, how="left")
        poe_provider_timing["ordertime"] = pd.to_datetime(poe_provider_timing["ordertime"], errors="coerce")
        poe_provider_timing["admittime"] = pd.to_datetime(poe_provider_timing["admittime"], errors="coerce")
        poe_provider_timing["order_hours_from_admission"] = ((poe_provider_timing["ordertime"] -
            poe_provider_timing["admittime"]).dt.total_seconds() / 3600).clip(lower=0)
        poe_provider_first_24h = poe_provider_timing.loc[poe_provider_timing["order_hours_from_admission"] <= 24,
            #summarise row-level records to admission-level features
            id_cols + ["order_provider_id"]].drop_duplicates().groupby(id_cols, as_index=False)["order_provider_id"].nunique()
        poe_provider_first_24h = poe_provider_first_24h.rename(columns={"order_provider_id": "num_unique_order_providers_first_24h"})
        poe_provider_first_72h = poe_provider_timing.loc[poe_provider_timing["order_hours_from_admission"] <= 72,
            id_cols + ["order_provider_id"]].drop_duplicates().groupby(id_cols, as_index=False)["order_provider_id"].nunique()
        poe_provider_first_72h = poe_provider_first_72h.rename(columns={"order_provider_id": "num_unique_order_providers_first_72h"})
        poe_features = poe_features.merge(poe_provider_first_24h, on=id_cols, how="left")
        poe_features = poe_features.merge(poe_provider_first_72h, on=id_cols, how="left")

        provider_sequence = poe_provider_timing.sort_values(id_cols + ["ordertime"])
        provider_sequence["previous_order_provider_id"] = provider_sequence.groupby(id_cols)["order_provider_id"].shift(1)
        provider_sequence["provider_changed"] = ((provider_sequence["previous_order_provider_id"].notna()) &
            (provider_sequence["order_provider_id"] != provider_sequence["previous_order_provider_id"])).astype(int)
        provider_changes = provider_sequence.groupby(id_cols, as_index=False)["provider_changed"].sum()
        provider_changes = provider_changes.rename(columns={"provider_changed": "num_provider_order_changes"})
        poe_features = poe_features.merge(provider_changes, on=id_cols, how="left")
        late_provider_timing = poe_provider_timing.merge(current_admissions[id_cols + ["dischtime"]], on=id_cols, how="left")
        late_provider_timing["hours_to_discharge"] = ((late_provider_timing["dischtime"] -
            late_provider_timing["ordertime"]).dt.total_seconds() / 3600)
        care_team_after_72h = late_provider_timing.loc[late_provider_timing["order_hours_from_admission"] > 72,
            id_cols + ["order_provider_id"]].drop_duplicates().groupby(id_cols, as_index=False)["order_provider_id"].nunique()
        care_team_after_72h = care_team_after_72h.rename(columns={"order_provider_id": "care_team_touchpoints_after_72h"})
        care_team_last_24h = late_provider_timing.loc[late_provider_timing["hours_to_discharge"].between(0, 24, inclusive="both"),
            id_cols + ["order_provider_id"]].drop_duplicates().groupby(id_cols, as_index=False)["order_provider_id"].nunique()
        care_team_last_24h = care_team_last_24h.rename(columns={"order_provider_id": "care_team_touchpoints_last_24h_before_discharge"})
        poe_features = poe_features.merge(care_team_after_72h, on=id_cols, how="left")
        poe_features = poe_features.merge(care_team_last_24h, on=id_cols, how="left")
else:
    poe_features = df[id_cols].copy()

/tmp/ipykernel_52805/2398243615.py:109: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  poe_features[ratio_col] = np.where(poe_features["num_total_poe_orders"] > 0,
/tmp/ipykernel_52805/2398243615.py:109: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  poe_features[ratio_col] = np.where(poe_features["num_total_poe_orders"] > 0,
/tmp/ipykernel_52805/2398243615.py:111: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all 

**Add POE-detail safety and aftercare flags**

The main POE table captures order timing and broad type/subtype labels. The detail table can contain more explicit wording for observation, suicide/elopement precautions, follow-up, and discharge planning, so this small follow-on block uses those terms as additional safety and aftercare proxies.

In [ ]:
#poe detail text features
#this cell uses poe_detail field names/values to catch safety, follow-up, discharge-planning,
#and exact administrative order-meaning terms that may not appear cleanly in poe.order_type/order_subtype.
#the features stay compact: flags for whether a term appeared and counts for repeated detail rows.
poe_detail_flag_cols = ["had_sitter_or_constant_observation_order", "had_suicide_precaution_order",
    "had_elopement_precaution_order", "had_restraint_order", "had_psych_consult_order",
    "had_social_work_case_management_order", "had_followup_or_outpatient_referral_order",
    "had_discharge_planning_order", "had_psych_followup_order", "had_outpatient_followup_order",
    "had_clinic_referral_order", "had_case_management_order", "had_sitter_order",
    "had_constant_observation_order", "had_behavioral_observation_order", "had_ciWA_protocol_order",
    "had_alcohol_withdrawal_protocol_order", "had_code_status_change_order", "had_discharge_now_order",
    "urgent_order_flag", "routine_order_flag"]
poe_detail_count_cols = ["num_code_status_orders", "num_discharge_when_orders",
    "num_transfer_to_orders", "num_level_of_urgency_orders"]
poe_detail_features = df[id_cols].copy()
#loop through each item in this feature block
for col in poe_detail_flag_cols + poe_detail_count_cols:
    poe_detail_features[col] = 0

if poe_key_rows and poe_detail_path.exists():
    poe_keys = pd.concat(poe_key_rows, ignore_index=True).drop_duplicates()
    poe_detail_rows = []
    poe_detail_usecols = ["poe_id", "poe_seq", "subject_id", "field_name", "field_value"]
    #loop through each item in this feature block
    for chunk_number, detail_chunk in enumerate(pd.read_csv(poe_detail_path, usecols=poe_detail_usecols, chunksize=1000000, low_memory=False)):
        #join the derived features back to the admission table
        detail_chunk = detail_chunk.merge(poe_keys, on=["subject_id", "poe_id", "poe_seq"], how="inner")
        if detail_chunk.empty:
            continue
        detail_text = (detail_chunk["field_name"].fillna("") + " " + detail_chunk["field_value"].fillna("")).str.lower()
        detail_chunk["had_sitter_or_constant_observation_order"] = detail_text.str.contains("sitter|constant observation|1:1|one to one|special observation|safety watch|observer", regex=True).astype(int)
        detail_chunk["had_suicide_precaution_order"] = detail_text.str.contains("suicide|suicidal|self harm|self-harm", regex=True).astype(int)
        detail_chunk["had_elopement_precaution_order"] = detail_text.str.contains("elopement|elope|flight risk", regex=True).astype(int)
        detail_chunk["had_restraint_order"] = detail_text.str.contains("restraint|seclusion", regex=True).astype(int)
        detail_chunk["had_psych_consult_order"] = detail_text.str.contains("psych|psychiat|behavioral health|behavioural health", regex=True).astype(int)
        detail_chunk["had_social_work_case_management_order"] = detail_text.str.contains("social work|case management|care coordination", regex=True).astype(int)
        detail_chunk["had_followup_or_outpatient_referral_order"] = detail_text.str.contains("follow up|follow-up|followup|appointment|outpatient|referral|clinic|aftercare", regex=True).astype(int)
        detail_chunk["had_discharge_planning_order"] = detail_text.str.contains("discharge planning|discharge plan|discharge instruction|aftercare|placement|home health|rehab|skilled nursing|home care", regex=True).astype(int)
        detail_chunk["had_psych_followup_order"] = detail_text.str.contains("psych.*follow|psychiat.*follow|behavioral health.*follow|behavioural health.*follow", regex=True).astype(int)
        detail_chunk["had_outpatient_followup_order"] = detail_text.str.contains("outpatient|ambulatory|follow up|follow-up|followup|aftercare", regex=True).astype(int)
        detail_chunk["had_clinic_referral_order"] = detail_text.str.contains("clinic|referral|appointment", regex=True).astype(int)
        detail_chunk["had_case_management_order"] = detail_text.str.contains("case management|social work|care coordination", regex=True).astype(int)
        detail_chunk["had_sitter_order"] = detail_text.str.contains("sitter|patient observer|safety watch", regex=True).astype(int)
        detail_chunk["had_constant_observation_order"] = detail_text.str.contains("constant observation|1:1|one to one|special observation", regex=True).astype(int)
        detail_chunk["had_behavioral_observation_order"] = detail_text.str.contains("behavioral observation|behavioural observation|agitation|violent|assaultive", regex=True).astype(int)
        field_name_text = detail_chunk["field_name"].fillna("").astype(str).str.lower()
        field_value_text = detail_chunk["field_value"].fillna("").astype(str).str.lower()
        detail_chunk["had_ciWA_protocol_order"] = detail_text.str.contains("ciwa|withdrawal protocol|alcohol withdrawal", regex=True).astype(int)
        detail_chunk["had_alcohol_withdrawal_protocol_order"] = detail_text.str.contains("alcohol withdrawal|ciwa", regex=True).astype(int)

        #exact poe-detail fields help separate routine discharge/admin activity from escalation or movement.
        detail_chunk["code_status_detail"] = field_name_text.str.contains("code status", regex=False).astype(int)
        detail_chunk["discharge_when_detail"] = field_name_text.str.contains("discharge when", regex=False).astype(int)
        detail_chunk["transfer_to_detail"] = field_name_text.str.contains("transfer to", regex=False).astype(int)
        detail_chunk["level_of_urgency_detail"] = field_name_text.str.contains("level of urgency|urgency", regex=True).astype(int)
        detail_chunk["had_code_status_change_order"] = ((detail_chunk["code_status_detail"] == 1) &
            field_value_text.str.contains("change|modified|full|dnr|dni|comfort|cmo", regex=True)).astype(int)
        detail_chunk["had_discharge_now_order"] = ((detail_chunk["discharge_when_detail"] == 1) &
            field_value_text.str.contains("now|today|immediate", regex=True)).astype(int)
        detail_chunk["urgent_order_flag"] = ((detail_chunk["level_of_urgency_detail"] == 1) &
            field_value_text.str.contains("urgent|stat|emergent|asap", regex=True)).astype(int)
        detail_chunk["routine_order_flag"] = ((detail_chunk["level_of_urgency_detail"] == 1) &
            field_value_text.str.contains("routine|standard|normal", regex=True)).astype(int)
        detail_chunk["num_code_status_orders"] = detail_chunk["code_status_detail"]
        detail_chunk["num_discharge_when_orders"] = detail_chunk["discharge_when_detail"]
        detail_chunk["num_transfer_to_orders"] = detail_chunk["transfer_to_detail"]
        detail_chunk["num_level_of_urgency_orders"] = detail_chunk["level_of_urgency_detail"]

        detail_agg = {col: (col, "max") for col in poe_detail_flag_cols}
        detail_agg.update({col: (col, "sum") for col in poe_detail_count_cols})
        #summarise row-level records to admission-level features
        poe_detail_rows.append(detail_chunk.groupby(id_cols, as_index=False).agg(**detail_agg))
        print("Processed POE detail chunk:", chunk_number, "rows kept:", len(detail_chunk))
    if poe_detail_rows:
        poe_detail_agg = {col: (col, "max") for col in poe_detail_flag_cols}
        poe_detail_agg.update({col: (col, "sum") for col in poe_detail_count_cols})
        poe_detail_features = pd.concat(poe_detail_rows, ignore_index=True).groupby(id_cols, as_index=False).agg(**poe_detail_agg)

#join the derived features back to the admission table
poe_features = poe_features.merge(poe_detail_features, on=id_cols, how="left", suffixes=("", "_detail"))
for count_col in poe_detail_count_cols:
    if count_col not in poe_features.columns:
        poe_features[count_col] = 0
    detail_col = count_col + "_detail"
    if detail_col in poe_features.columns:
        poe_features[count_col] = (pd.to_numeric(poe_features[count_col], errors="coerce").fillna(0) +
            pd.to_numeric(poe_features[detail_col], errors="coerce").fillna(0)).astype(int)
        poe_features = poe_features.drop(columns=[detail_col])
for flag_col in poe_detail_flag_cols:
    if flag_col not in poe_features.columns:
        poe_features[flag_col] = 0
    detail_col = flag_col + "_detail"
    if detail_col in poe_features.columns:
        poe_features[flag_col] = np.maximum(
            pd.to_numeric(poe_features[flag_col], errors="coerce").fillna(0),
            pd.to_numeric(poe_features[detail_col], errors="coerce").fillna(0)).astype(int)
        poe_features = poe_features.drop(columns=[detail_col])

if "num_unique_order_providers" in poe_features.columns:
    admission_los_days_for_care_team = ((poe_features["last_order_hours_from_admission"].fillna(0) +
        poe_features["last_order_close_to_discharge_hours"].fillna(0)) / 24).clip(lower=1)
    poe_features["care_team_touchpoint_density_per_day"] = (poe_features["num_unique_order_providers"].fillna(0) /
        admission_los_days_for_care_team).replace([np.inf, -np.inf], 0).fillna(0)
    if "care_team_touchpoints_after_72h" not in poe_features.columns:
        poe_features["care_team_touchpoints_after_72h"] = 0
    if "care_team_touchpoints_last_24h_before_discharge" not in poe_features.columns:
        poe_features["care_team_touchpoints_last_24h_before_discharge"] = 0
    poe_features["late_care_team_activity_flag"] = (pd.to_numeric(
        poe_features["care_team_touchpoints_after_72h"], errors="coerce").fillna(0) > 0).astype(int)
    poe_features["high_care_team_touchpoints_near_discharge_flag"] = (pd.to_numeric(
        poe_features["care_team_touchpoints_last_24h_before_discharge"], errors="coerce").fillna(0) >= 3).astype(int)

Processed POE detail chunk: 0 rows kept: 464871
Processed POE detail chunk: 1 rows kept: 459461
Processed POE detail chunk: 2 rows kept: 465758
Processed POE detail chunk: 3 rows kept: 458345
Processed POE detail chunk: 4 rows kept: 458085
Processed POE detail chunk: 5 rows kept: 458635
Processed POE detail chunk: 6 rows kept: 456935
Processed POE detail chunk: 7 rows kept: 456190
Processed POE detail chunk: 8 rows kept: 230119


/tmp/ipykernel_52805/4151162619.py:102: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  poe_features["care_team_touchpoint_density_per_day"] = (poe_features["num_unique_order_providers"].fillna(0) /
/tmp/ipykernel_52805/4151162619.py:108: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  poe_features["late_care_team_activity_flag"] = (pd.to_numeric(
/tmp/ipykernel_52805/4151162619.py:110: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  C

**Merge POE features into the modelling cohort**

This final POE step standardises missing columns and data types before merging the order-pathway table-derived feature block into the main index-admission dataset. Timing and ratio columns remain numeric; binary/count indicators are filled to zero.

In [ ]:
#finalise and merge poe table-derived feature table
#this cell guarantees all expected poe columns exist, preserves true numeric timing/ratio fields,
#casts flag/count columns consistently, and merges the poe table-derived feature block into the main cohort.
for col in poe_engineered_cols:
    if col not in poe_features.columns:
        poe_features[col] = 0

df = df.drop(columns=[col for col in poe_engineered_cols if col in df.columns], errors="ignore")
df = df.merge(poe_features[id_cols + poe_engineered_cols], on=id_cols, how="left")
for col in poe_engineered_cols:
    if col in ["first_order_hours_from_admission", "last_order_hours_from_admission", "order_activity_slope_24_to_72h",
            "first_discharge_planning_order_hours", "last_discharge_planning_order_hours",
            "last_order_close_to_discharge_hours", "order_activity_duration_hours", "first_safety_order_hours_from_admission",
            "last_safety_order_hours_from_admission", "safety_order_activity_duration_hours",
            "order_activity_last_24h_to_total_ratio", "order_activity_last_48h_to_total_ratio",
            "order_activity_last_12h_to_total_ratio", "order_activity_last_6h_to_total_ratio",
            "late_order_discontinuation_share", "safety_order_density_per_day", "care_team_touchpoint_density_per_day"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    else:
        df[col] = df[col].fillna(0).astype(int)

if "has_self_harm_or_suicidal_ideation" in df.columns:
    df["self_harm_and_late_safety_order_flag"] = ((df["has_self_harm_or_suicidal_ideation"] == 1) &
        (df["late_safety_order_flag"] == 1)).astype(int)
else:
    df["self_harm_and_late_safety_order_flag"] = 0

print("POE order table-derived feature extraction complete.")
print("POE table-derived feature count:", len(poe_engineered_cols))
print("Dataset shape after POE features:", df.shape)

POE order table-derived feature extraction complete.
POE table-derived feature count: 126
Dataset shape after POE features: (238565, 592)


/tmp/ipykernel_52805/1151353876.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  poe_features[col] = 0


**Summarise previous-admission instability patterns**

This block mirrors selected order-pathway signals from earlier admissions only. It captures whether previous admissions had late orders, safety orders, follow-up or discharge planning activity, home discharge without follow-up, and other instability-like patterns.

The reason for this block is that the number of prior admissions alone is quite broad. Describing what those previous admissions looked like gives the model a more specific history profile without using anything after the index admission.


In [ ]:
#previous-admission order-instability features
#sections:
#1. link each index stay to prior stays for the same patient.
#2. reuse prior-stay poe summaries to describe previous late orders, safety orders, and discharge planning.
#3. keep only past information so the variables are valid predictors rather than future outcome markers.
print("Starting previous-admission order-instability feature extraction...")

previous_order_instability_cols = ["previous_admission_had_late_orders", "previous_admission_had_safety_order",
    "previous_admission_had_discharge_planning_order", "previous_admission_order_activity_count",
    "previous_admission_had_late_safety_orders", "previous_admission_had_late_lab_orders",
    "previous_admission_had_late_medication_orders", "previous_admission_had_followup_order",
    "previous_admission_had_home_discharge_no_followup", "previous_admission_had_psych_facility_discharge",
    "previous_admission_had_facility_discharge", "previous_admission_late_order_count",
    "previous_admission_late_order_share", "prior_discharge_instability_score", "prior_instability_score"]
prior_order_hadm_features = pd.DataFrame(columns=id_cols + previous_order_instability_cols[:-1])

if poe_path.exists():
    prior_poe_rows = []
    prior_order_admissions = raw_admissions[["subject_id", "hadm_id", "admittime", "dischtime"]].copy()
    prior_order_hadm_ids = set(prior_order_admissions["hadm_id"].dropna().astype(int))
    prior_poe_usecols = ["subject_id", "hadm_id", "poe_id", "ordertime", "order_type", "order_subtype", "transaction_type", "order_status"]

    #loop through each item in this feature block
    for chunk_number, prior_poe_chunk in enumerate(pd.read_csv(poe_path, usecols=prior_poe_usecols, chunksize=1000000, low_memory=False)):
        prior_poe_chunk = prior_poe_chunk[prior_poe_chunk["subject_id"].isin(current_subject_ids) &
            prior_poe_chunk["hadm_id"].isin(prior_order_hadm_ids)].copy()
        if prior_poe_chunk.empty:
            continue
        prior_poe_chunk["ordertime"] = pd.to_datetime(prior_poe_chunk["ordertime"], errors="coerce")
        #join the derived features back to the admission table
        prior_poe_chunk = prior_poe_chunk.merge(prior_order_admissions, on=["subject_id", "hadm_id"], how="left")
        prior_poe_chunk = prior_poe_chunk[(prior_poe_chunk["ordertime"].notna()) & (prior_poe_chunk["admittime"].notna()) &
            (prior_poe_chunk["ordertime"] >= prior_poe_chunk["admittime"]) &
            ((prior_poe_chunk["dischtime"].isna()) | (prior_poe_chunk["ordertime"] <= prior_poe_chunk["dischtime"]))].copy()
        if prior_poe_chunk.empty:
            continue
        prior_hours_to_discharge = (prior_poe_chunk["dischtime"] - prior_poe_chunk["ordertime"]).dt.total_seconds() / 3600
        prior_order_text = (prior_poe_chunk["order_type"].fillna("") + " " + prior_poe_chunk["order_subtype"].fillna("") + " " +
            prior_poe_chunk["transaction_type"].fillna("") + " " + prior_poe_chunk["order_status"].fillna("")).str.lower()
        prior_poe_chunk["previous_late_order"] = prior_hours_to_discharge.between(0, 24, inclusive="both").astype(int)
        prior_poe_chunk["previous_late_order_12h"] = prior_hours_to_discharge.between(0, 12, inclusive="both").astype(int)
        prior_poe_chunk["previous_safety_order"] = prior_order_text.str.contains(
            "sitter|constant observation|1:1|one to one|special observation|safety watch|observer|suicide|suicidal|self harm|self-harm|elopement|elope|escape risk|restraint|seclusion|security|agitation|combative",
            regex=True).astype(int)
        prior_poe_chunk["previous_discharge_planning_order"] = prior_order_text.str.contains(
            "discharge planning|discharge plan|discharge instruction|aftercare|placement|home health|rehab|skilled nursing|home care|follow up|follow-up|followup|appointment|outpatient|referral|clinic|case management|social work",
            regex=True).astype(int)
        prior_poe_chunk["previous_followup_order"] = prior_order_text.str.contains(
            "follow up|follow-up|followup|f/u|appointment|outpatient|referral|clinic|aftercare",
            regex=True).astype(int)
        prior_poe_chunk["previous_lab_order"] = prior_order_text.str.contains(
            "lab|laboratory|blood draw|blood test|chemistry|hematology|microbiology|culture",
            regex=True).astype(int)
        prior_poe_chunk["previous_medication_order"] = prior_order_text.str.contains(
            "medication|pharmacy|drug|rx|prescription|dose|infusion|iv|po|oral|prn",
            regex=True).astype(int)
        prior_poe_chunk["previous_late_safety_order"] = ((prior_poe_chunk["previous_late_order"] == 1) &
            (prior_poe_chunk["previous_safety_order"] == 1)).astype(int)
        prior_poe_chunk["previous_late_lab_order"] = ((prior_poe_chunk["previous_late_order"] == 1) &
            (prior_poe_chunk["previous_lab_order"] == 1)).astype(int)
        prior_poe_chunk["previous_late_medication_order"] = ((prior_poe_chunk["previous_late_order"] == 1) &
            (prior_poe_chunk["previous_medication_order"] == 1)).astype(int)
        #summarise row-level records to admission-level features
        prior_poe_rows.append(prior_poe_chunk.groupby(id_cols, as_index=False).agg(
            previous_admission_order_activity_count=("poe_id", "count"),
            previous_admission_had_late_orders=("previous_late_order", "max"),
            previous_admission_had_safety_order=("previous_safety_order", "max"),
            previous_admission_had_discharge_planning_order=("previous_discharge_planning_order", "max"),
            previous_admission_had_late_safety_orders=("previous_late_safety_order", "max"),
            previous_admission_had_late_lab_orders=("previous_late_lab_order", "max"),
            previous_admission_had_late_medication_orders=("previous_late_medication_order", "max"),
            previous_admission_had_followup_order=("previous_followup_order", "max"),
            previous_admission_late_order_count=("previous_late_order", "sum")))
        print("Processed prior POE chunk:", chunk_number, "rows kept:", len(prior_poe_chunk))

    if prior_poe_rows:
        prior_order_hadm_features = pd.concat(prior_poe_rows, ignore_index=True).groupby(id_cols, as_index=False).agg(
            previous_admission_order_activity_count=("previous_admission_order_activity_count", "sum"),
            previous_admission_had_late_orders=("previous_admission_had_late_orders", "max"),
            previous_admission_had_safety_order=("previous_admission_had_safety_order", "max"),
            previous_admission_had_discharge_planning_order=("previous_admission_had_discharge_planning_order", "max"),
            previous_admission_had_late_safety_orders=("previous_admission_had_late_safety_orders", "max"),
            previous_admission_had_late_lab_orders=("previous_admission_had_late_lab_orders", "max"),
            previous_admission_had_late_medication_orders=("previous_admission_had_late_medication_orders", "max"),
            previous_admission_had_followup_order=("previous_admission_had_followup_order", "max"),
            previous_admission_late_order_count=("previous_admission_late_order_count", "sum"))
        prior_order_hadm_features["previous_admission_late_order_share"] = (
            prior_order_hadm_features["previous_admission_late_order_count"] /
            prior_order_hadm_features["previous_admission_order_activity_count"].replace(0, np.nan)).replace([np.inf, -np.inf], 0).fillna(0)

prior_discharge_lookup = pd.DataFrame()
if "discharge_location" in raw_admissions.columns:
    prior_discharge_lookup = raw_admissions[["hadm_id", "discharge_location"]].copy()
    prior_discharge_text = prior_discharge_lookup["discharge_location"].fillna("").astype(str).str.upper()
    prior_discharge_lookup["previous_admission_had_psych_facility_discharge"] = prior_discharge_text.str.contains(
        "PSYCH|PSYCHIATRIC", regex=True).astype(int)
    prior_discharge_lookup["previous_admission_had_facility_discharge"] = prior_discharge_text.str.contains(
        "FACILITY|SNF|REHAB|NURSING|SKILLED|CHRONIC|INTERMEDIATE|ASSISTED", regex=True).astype(int)
    prior_discharge_lookup["previous_admission_had_home_discharge"] = prior_discharge_text.str.contains(
        "HOME", regex=True).astype(int)
    prior_discharge_lookup = prior_discharge_lookup.set_index("hadm_id")

prior_order_lookup = prior_order_hadm_features.set_index("hadm_id") if not prior_order_hadm_features.empty else pd.DataFrame()
prior_order_rows = []
raw_admission_groups_for_prior_orders = {sid: group.dropna(subset=["dischtime"]).sort_values("dischtime")
    #summarise row-level records to admission-level features
    for sid, group in raw_admissions.groupby("subject_id")}

#loop through each item in this feature block
for subject_id, current_group in current_admissions.groupby("subject_id", sort=False):
    history = raw_admission_groups_for_prior_orders.get(subject_id)
    if history is None or history.empty:
        #loop through each item in this feature block
        for _, row in current_group.iterrows():
            prior_order_rows.append({"subject_id": row["subject_id"], "hadm_id": row["hadm_id"]})
        continue
    history_discharge_times = history["dischtime"].to_numpy(dtype="datetime64[ns]")
    history_hadm_ids = history["hadm_id"].to_numpy()
    for _, row in current_group.iterrows():
        result = {"subject_id": row["subject_id"], "hadm_id": row["hadm_id"]}
        if pd.notna(row["admittime"]):
            previous_end = np.searchsorted(history_discharge_times, np.datetime64(row["admittime"]), side="left")
            if previous_end > 0:
                previous_hadm_id = history_hadm_ids[previous_end - 1]
                previous_home_discharge = 0
                if not prior_discharge_lookup.empty and previous_hadm_id in prior_discharge_lookup.index:
                    prior_discharge_values = prior_discharge_lookup.loc[previous_hadm_id]
                    result["previous_admission_had_psych_facility_discharge"] = prior_discharge_values["previous_admission_had_psych_facility_discharge"]
                    result["previous_admission_had_facility_discharge"] = prior_discharge_values["previous_admission_had_facility_discharge"]
                    previous_home_discharge = prior_discharge_values["previous_admission_had_home_discharge"]
                if not prior_order_lookup.empty and previous_hadm_id in prior_order_lookup.index:
                    prior_values = prior_order_lookup.loc[previous_hadm_id]
                    #loop through each item in this feature block
                    for col in [col for col in previous_order_instability_cols[:-1] if col in prior_order_lookup.columns]:
                        result[col] = prior_values[col]
                result["previous_admission_had_home_discharge_no_followup"] = int(
                    previous_home_discharge == 1 and result.get("previous_admission_had_followup_order", 0) == 0)
        prior_order_rows.append(result)

prior_order_features = pd.DataFrame(prior_order_rows)
for col in previous_order_instability_cols[:-1]:
    if col not in prior_order_features.columns:
        prior_order_features[col] = 0
    if col == "previous_admission_late_order_share":
        prior_order_features[col] = pd.to_numeric(prior_order_features[col], errors="coerce").fillna(0)
    else:
        prior_order_features[col] = pd.to_numeric(prior_order_features[col], errors="coerce").fillna(0).astype(int)

df = df.drop(columns=[col for col in previous_order_instability_cols if col in df.columns], errors="ignore")
#join the derived features back to the admission table
df = df.merge(prior_order_features[id_cols + previous_order_instability_cols[:-1]], on=id_cols, how="left")
for col in previous_order_instability_cols[:-1]:
    if col == "previous_admission_late_order_share":
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    else:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

prior_instability_components = ["previous_admission_had_late_orders", "previous_admission_had_safety_order",
    "previous_admission_had_discharge_planning_order", "previous_admission_had_late_safety_orders",
    "previous_admission_had_late_lab_orders", "previous_admission_had_late_medication_orders",
    "previous_admission_had_home_discharge_no_followup", "previous_discharge_against_advice_flag",
    "previous_discharge_to_psych_facility_flag", "previous_discharge_to_facility_flag", "previous_short_los_flag",
    "previous_long_los_flag"]
df["prior_instability_score"] = df[[col for col in prior_instability_components if col in df.columns]].sum(axis=1).astype(int)
df["prior_discharge_instability_score"] = df[[col for col in [
    "previous_admission_had_late_orders", "previous_admission_had_late_safety_orders",
    "previous_admission_had_late_lab_orders", "previous_admission_had_late_medication_orders",
    "previous_admission_had_home_discharge_no_followup", "previous_admission_had_psych_facility_discharge",
    "previous_admission_had_facility_discharge", "previous_discharge_against_advice_flag"] if col in df.columns]].sum(axis=1).astype(int)

print("Previous-admission order-instability table-derived feature extraction complete.")
print("Previous-order table-derived feature count:", len(previous_order_instability_cols))
print("Dataset shape after prior-order instability features:", df.shape)

Starting previous-admission order-instability feature extraction...
Processed prior POE chunk: 0 rows kept: 591379
Processed prior POE chunk: 1 rows kept: 583292
Processed prior POE chunk: 2 rows kept: 585416
Processed prior POE chunk: 3 rows kept: 597348
Processed prior POE chunk: 4 rows kept: 593494
Processed prior POE chunk: 5 rows kept: 570053
Processed prior POE chunk: 6 rows kept: 610587
Processed prior POE chunk: 7 rows kept: 579518
Processed prior POE chunk: 8 rows kept: 576871
Processed prior POE chunk: 9 rows kept: 578942
Processed prior POE chunk: 10 rows kept: 599966
Processed prior POE chunk: 11 rows kept: 596916
Processed prior POE chunk: 12 rows kept: 596578
Processed prior POE chunk: 13 rows kept: 586804
Processed prior POE chunk: 14 rows kept: 596621
Processed prior POE chunk: 15 rows kept: 584502
Processed prior POE chunk: 16 rows kept: 627642
Processed prior POE chunk: 17 rows kept: 607408
Processed prior POE chunk: 18 rows kept: 603599
Processed prior POE chunk: 19 

/tmp/ipykernel_52805/22434153.py:165: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["prior_instability_score"] = df[[col for col in prior_instability_components if col in df.columns]].sum(axis=1).astype(int)


**Create pharmacy medication-intensity features**

Pharmacy records add route, schedule, infusion, duration, verification, dispensation, and process-status signals. This block summarises medication intensity and medication-logistics complexity at admission level.

These features complement prescription orders and EMAR administration records. They help separate ordinary medication exposure from more complex medication management, such as infusions, sliding scales, dose changes, or discharge-medication preparation.


In [ ]:
#pharmacy medication-intensity features
#sections in this cell:
#1. read pharmacy records in chunks for the current cohort.
#2. summarise medication intensity, route/frequency, verification, prn, infusion, and psychotropic timing.
#3. convert detailed medication-process fields into compact counts/flags rather than raw medication ids.
print("Starting pharmacy medication-intensity feature extraction...")

pharmacy_engineered_cols = ["num_pharmacy_records", "num_unique_pharmacy_medications", "num_iv_pharmacy_orders",
    "num_infusion_pharmacy_orders", "num_prn_pharmacy_orders", "num_scheduled_pharmacy_orders",
    "num_high_frequency_med_orders", "num_med_orders_with_duration", "num_discontinued_pharmacy_orders",
    "num_verified_pharmacy_orders", "num_oral_pharmacy_orders", "num_iv_route_pharmacy_orders",
    "num_intramuscular_pharmacy_orders", "num_subcutaneous_pharmacy_orders", "num_prn_psychotropic_orders",
    "num_scheduled_psychotropic_orders", "num_medication_route_types", "num_medication_frequency_types",
    "psychotropic_order_density_per_day", "had_prn_antipsychotic", "had_prn_benzodiazepine",
    "num_prn_benzodiazepine_orders", "num_prn_antipsychotic_orders", "had_naloxone_order",
    "had_methadone_or_buprenorphine", "had_withdrawal_treatment_order", "had_lockout_interval_order",
    "had_basal_rate_order", "had_sliding_scale_order", "had_one_hr_max_order",
    "num_unverified_pharmacy_orders", "num_floor_stock_pharmacy_orders",
    "num_patient_may_take_own_med_orders", "num_dosing_by_pharmacy_orders",
    "num_discharge_med_proc_type_orders", "pharmacy_logistics_complexity_score",
    "pharmacy_infusion_complexity_score",
    "psychotropic_orders_last_24h_before_discharge", "psychotropic_orders_last_48h_before_discharge",
    "prn_psychotropic_orders_last_24h_before_discharge"]

pharmacy_path = hosp_path / "pharmacy.csv.gz"
pharmacy_feature_rows = []
pharmacy_medication_rows = []
pharmacy_route_rows = []
pharmacy_frequency_rows = []
admission_times_for_pharmacy = current_admissions[["subject_id", "hadm_id", "admittime", "dischtime"]].copy()


#scan pharmacy records in chunks for current admissions
if pharmacy_path.exists():
    pharmacy_usecols = ["subject_id", "hadm_id", "starttime", "entertime", "verifiedtime", "medication", "proc_type",
        "status", "route", "frequency", "disp_sched", "infusion_type", "sliding_scale", "lockout_interval",
        "basal_rate", "one_hr_max", "doses_per_24_hrs", "duration", "duration_interval", "dispensation"]
    #loop through each item in this feature block
    for chunk_number, pharmacy_chunk in enumerate(pd.read_csv(pharmacy_path, usecols=pharmacy_usecols, chunksize=1000000, low_memory=False)):
        pharmacy_chunk = pharmacy_chunk[pharmacy_chunk["hadm_id"].isin(current_hadm_ids)].copy()
        if pharmacy_chunk.empty:
            continue
        #loop through each item in this feature block
        for time_col in ["starttime", "entertime", "verifiedtime"]:
            pharmacy_chunk[time_col] = pd.to_datetime(pharmacy_chunk[time_col], errors="coerce")
        pharmacy_chunk["pharmacy_event_time"] = pharmacy_chunk["starttime"].fillna(pharmacy_chunk["entertime"]).fillna(pharmacy_chunk["verifiedtime"])
        #join the derived features back to the admission table
        pharmacy_chunk = pharmacy_chunk.merge(admission_times_for_pharmacy, on=["subject_id", "hadm_id"], how="left")
        pharmacy_chunk = pharmacy_chunk[(pharmacy_chunk["pharmacy_event_time"].notna()) & (pharmacy_chunk["admittime"].notna()) &
            (pharmacy_chunk["pharmacy_event_time"] >= pharmacy_chunk["admittime"]) &
            ((pharmacy_chunk["dischtime"].isna()) | (pharmacy_chunk["pharmacy_event_time"] <= pharmacy_chunk["dischtime"]))].copy()
        if pharmacy_chunk.empty:
            continue
        route_text = pharmacy_chunk["route"].fillna("").astype(str).str.lower()
        frequency_text = pharmacy_chunk["frequency"].fillna("").astype(str).str.lower()
        schedule_text = pharmacy_chunk["disp_sched"].fillna("").astype(str).str.lower()
        infusion_text = pharmacy_chunk["infusion_type"].fillna("").astype(str).str.lower()
        status_text = pharmacy_chunk["status"].fillna("").astype(str).str.lower()
        proc_text = pharmacy_chunk["proc_type"].fillna("").astype(str).str.lower()
        dispensation_text = pharmacy_chunk["dispensation"].fillna("").astype(str).str.lower()

        #grouped medication/process categories from pharmacy fields
        medication_text = pharmacy_chunk["medication"].fillna("").astype(str).str.lower()
        doses_per_24h = pd.to_numeric(pharmacy_chunk["doses_per_24_hrs"], errors="coerce")
        hours_to_discharge = (pharmacy_chunk["dischtime"] - pharmacy_chunk["pharmacy_event_time"]).dt.total_seconds() / 3600
        pharmacy_chunk["oral_pharmacy_order"] = route_text.str.contains(r"po|oral|mouth|by mouth", regex=True).astype(int)
        pharmacy_chunk["iv_pharmacy_order"] = (route_text.str.contains("iv|intravenous|ivpb|iv drip", regex=True) |
            proc_text.str.contains("iv|intravenous", regex=True)).astype(int)
        pharmacy_chunk["iv_route_pharmacy_order"] = pharmacy_chunk["iv_pharmacy_order"]
        pharmacy_chunk["intramuscular_pharmacy_order"] = route_text.str.contains(r"im|intramuscular", regex=True).astype(int)
        pharmacy_chunk["subcutaneous_pharmacy_order"] = route_text.str.contains(r"sc|sq|subcutaneous", regex=True).astype(int)
        pharmacy_chunk["infusion_pharmacy_order"] = (infusion_text.ne("") | route_text.str.contains("infusion", regex=True) |
            proc_text.str.contains("infusion", regex=True)).astype(int)
        pharmacy_chunk["prn_pharmacy_order"] = (frequency_text.str.contains("prn|as needed", regex=True) |
            schedule_text.str.contains("prn|as needed", regex=True)).astype(int)
        pharmacy_chunk["scheduled_pharmacy_order"] = ((pharmacy_chunk["prn_pharmacy_order"] == 0) &
            (frequency_text.ne("") | schedule_text.ne(""))).astype(int)
        pharmacy_chunk["high_frequency_med_order"] = ((doses_per_24h >= 4) |
            frequency_text.str.contains(r"q1h|q2h|q3h|q4h|q6h|every 1|every 2|every 3|every 4|every 6", regex=True)).astype(int)
        psychotropic_text = medication_text.str.contains(
            "haloperidol|olanzapine|quetiapine|risperidone|clozapine|aripiprazole|ziprasidone|lurasidone|paliperidone|fluphenazine|chlorpromazine|lorazepam|diazepam|clonazepam|alprazolam|midazolam|lithium|valpro|divalproex|carbamazepine|lamotrigine|sertraline|fluoxetine|citalopram|escitalopram|paroxetine|venlafaxine|duloxetine|mirtazapine|trazodone", regex=True)
        antipsychotic_text = medication_text.str.contains("haloperidol|olanzapine|quetiapine|risperidone|clozapine|aripiprazole|ziprasidone|lurasidone|paliperidone|fluphenazine|chlorpromazine", regex=True)
        benzodiazepine_text = medication_text.str.contains("lorazepam|diazepam|clonazepam|alprazolam|midazolam|temazepam|chlordiazepoxide|oxazepam", regex=True)
        pharmacy_chunk["prn_psychotropic_order"] = ((pharmacy_chunk["prn_pharmacy_order"] == 1) & psychotropic_text).astype(int)
        pharmacy_chunk["scheduled_psychotropic_order"] = ((pharmacy_chunk["scheduled_pharmacy_order"] == 1) & psychotropic_text).astype(int)
        pharmacy_chunk["psychotropic_order_last_24h_before_discharge"] = (psychotropic_text & hours_to_discharge.between(0, 24, inclusive="both")).astype(int)
        pharmacy_chunk["psychotropic_order_last_48h_before_discharge"] = (psychotropic_text & hours_to_discharge.between(0, 48, inclusive="both")).astype(int)
        pharmacy_chunk["prn_psychotropic_order_last_24h_before_discharge"] = ((pharmacy_chunk["prn_pharmacy_order"] == 1) & psychotropic_text &
            hours_to_discharge.between(0, 24, inclusive="both")).astype(int)
        pharmacy_chunk["prn_antipsychotic_order"] = ((pharmacy_chunk["prn_pharmacy_order"] == 1) & antipsychotic_text).astype(int)
        pharmacy_chunk["prn_benzodiazepine_order"] = ((pharmacy_chunk["prn_pharmacy_order"] == 1) & benzodiazepine_text).astype(int)
        pharmacy_chunk["naloxone_order"] = medication_text.str.contains("naloxone|narcan", regex=True).astype(int)
        pharmacy_chunk["methadone_or_buprenorphine_order"] = medication_text.str.contains("methadone|buprenorphine|suboxone|subutex", regex=True).astype(int)
        pharmacy_chunk["withdrawal_treatment_order"] = medication_text.str.contains("chlordiazepoxide|phenobarbital|thiamine|folic acid|methadone|buprenorphine|naloxone", regex=True).astype(int)
        pharmacy_chunk["floor_stock_pharmacy_order"] = dispensation_text.str.contains("floor stock|omnicell|bulk item", regex=True).astype(int)
        pharmacy_chunk["patient_may_take_own_med_order"] = dispensation_text.str.contains("patient may take own|self med", regex=True).astype(int)
        pharmacy_chunk["dosing_by_pharmacy_order"] = dispensation_text.str.contains("dosing by pharmacy|pharmacy dosing", regex=True).astype(int)
        pharmacy_chunk["discharge_med_proc_type_order"] = proc_text.str.contains("discharge|take home|meds to beds|outpatient", regex=True).astype(int)
        pharmacy_chunk["med_order_with_duration"] = (pharmacy_chunk["duration"].notna() | pharmacy_chunk["duration_interval"].notna()).astype(int)
        pharmacy_chunk["lockout_interval_order"] = pharmacy_chunk["lockout_interval"].notna().astype(int)
        pharmacy_chunk["basal_rate_order"] = pharmacy_chunk["basal_rate"].notna().astype(int)
        pharmacy_chunk["sliding_scale_order"] = pharmacy_chunk["sliding_scale"].fillna("").astype(str).str.lower().isin(["1", "true", "yes", "y"]).astype(int)
        pharmacy_chunk["one_hr_max_order"] = pharmacy_chunk["one_hr_max"].notna().astype(int)
        pharmacy_chunk["discontinued_pharmacy_order"] = status_text.str.contains("discontinue|discontinued|cancel|inactive|stop", regex=True).astype(int)
        pharmacy_chunk["verified_pharmacy_order"] = (pharmacy_chunk["verifiedtime"].notna() |
            status_text.str.contains("verified|active|approved", regex=True)).astype(int)
        pharmacy_chunk["unverified_pharmacy_order"] = (pharmacy_chunk["verified_pharmacy_order"] == 0).astype(int)
        #summarise row-level records to admission-level features
        pharmacy_feature_rows.append(pharmacy_chunk.groupby(id_cols, as_index=False).agg(
            num_pharmacy_records=("medication", "size"),
            num_iv_pharmacy_orders=("iv_pharmacy_order", "sum"),
            num_infusion_pharmacy_orders=("infusion_pharmacy_order", "sum"),
            num_prn_pharmacy_orders=("prn_pharmacy_order", "sum"),
            num_scheduled_pharmacy_orders=("scheduled_pharmacy_order", "sum"),
            num_high_frequency_med_orders=("high_frequency_med_order", "sum"),
            num_med_orders_with_duration=("med_order_with_duration", "sum"),
            num_discontinued_pharmacy_orders=("discontinued_pharmacy_order", "sum"),
            num_verified_pharmacy_orders=("verified_pharmacy_order", "sum"),
            num_unverified_pharmacy_orders=("unverified_pharmacy_order", "sum"),
            num_floor_stock_pharmacy_orders=("floor_stock_pharmacy_order", "sum"),
            num_patient_may_take_own_med_orders=("patient_may_take_own_med_order", "sum"),
            num_dosing_by_pharmacy_orders=("dosing_by_pharmacy_order", "sum"),
            num_discharge_med_proc_type_orders=("discharge_med_proc_type_order", "sum"),
            had_lockout_interval_order=("lockout_interval_order", "max"),
            had_basal_rate_order=("basal_rate_order", "max"),
            had_sliding_scale_order=("sliding_scale_order", "max"),
            had_one_hr_max_order=("one_hr_max_order", "max"),
            num_oral_pharmacy_orders=("oral_pharmacy_order", "sum"),
            num_iv_route_pharmacy_orders=("iv_route_pharmacy_order", "sum"),
            num_intramuscular_pharmacy_orders=("intramuscular_pharmacy_order", "sum"),
            num_subcutaneous_pharmacy_orders=("subcutaneous_pharmacy_order", "sum"),
            num_prn_psychotropic_orders=("prn_psychotropic_order", "sum"),
            num_scheduled_psychotropic_orders=("scheduled_psychotropic_order", "sum"),
            psychotropic_orders_last_24h_before_discharge=("psychotropic_order_last_24h_before_discharge", "sum"),
            psychotropic_orders_last_48h_before_discharge=("psychotropic_order_last_48h_before_discharge", "sum"),
            prn_psychotropic_orders_last_24h_before_discharge=("prn_psychotropic_order_last_24h_before_discharge", "sum"),
            num_prn_antipsychotic_orders=("prn_antipsychotic_order", "sum"),
            had_prn_antipsychotic=("prn_antipsychotic_order", "max"),
            num_prn_benzodiazepine_orders=("prn_benzodiazepine_order", "sum"),
            had_prn_benzodiazepine=("prn_benzodiazepine_order", "max"),
            had_naloxone_order=("naloxone_order", "max"),
            had_methadone_or_buprenorphine=("methadone_or_buprenorphine_order", "max"),
            had_withdrawal_treatment_order=("withdrawal_treatment_order", "max")))
        pharmacy_medication_rows.append(pharmacy_chunk[id_cols + ["medication"]].dropna(subset=["medication"]).drop_duplicates())
        pharmacy_route_rows.append(pharmacy_chunk[id_cols + ["route"]].dropna(subset=["route"]).drop_duplicates())
        pharmacy_frequency_rows.append(pharmacy_chunk[id_cols + ["frequency"]].dropna(subset=["frequency"]).drop_duplicates())
        print("Processed pharmacy chunk:", chunk_number, "rows kept:", len(pharmacy_chunk))

if len(pharmacy_feature_rows) > 0:
    #summarise row-level records to admission-level features
    pharmacy_features = pd.concat(pharmacy_feature_rows, ignore_index=True).groupby(id_cols, as_index=False).sum()
    if len(pharmacy_medication_rows) > 0:
        pharmacy_medication_counts = pd.concat(pharmacy_medication_rows, ignore_index=True).drop_duplicates().groupby(id_cols, as_index=False)["medication"].nunique()
        pharmacy_medication_counts = pharmacy_medication_counts.rename(columns={"medication": "num_unique_pharmacy_medications"})
        pharmacy_features = pharmacy_features.merge(pharmacy_medication_counts, on=id_cols, how="left")
    if len(pharmacy_route_rows) > 0:
        pharmacy_route_counts = pd.concat(pharmacy_route_rows, ignore_index=True).drop_duplicates().groupby(id_cols, as_index=False)["route"].nunique()
        pharmacy_route_counts = pharmacy_route_counts.rename(columns={"route": "num_medication_route_types"})
        pharmacy_features = pharmacy_features.merge(pharmacy_route_counts, on=id_cols, how="left")
    if len(pharmacy_frequency_rows) > 0:
        pharmacy_frequency_counts = pd.concat(pharmacy_frequency_rows, ignore_index=True).drop_duplicates().groupby(id_cols, as_index=False)["frequency"].nunique()
        pharmacy_frequency_counts = pharmacy_frequency_counts.rename(columns={"frequency": "num_medication_frequency_types"})
        pharmacy_features = pharmacy_features.merge(pharmacy_frequency_counts, on=id_cols, how="left")
    #join the derived features back to the admission table
    pharmacy_features = pharmacy_features.merge(admission_times_for_pharmacy, on=id_cols, how="left")
    pharmacy_los_days = ((pharmacy_features["dischtime"] - pharmacy_features["admittime"]).dt.total_seconds() / 86400).clip(lower=1)
    pharmacy_features["psychotropic_order_density_per_day"] = ((pharmacy_features.get("num_prn_psychotropic_orders", 0) +
        pharmacy_features.get("num_scheduled_psychotropic_orders", 0)) / pharmacy_los_days).replace([np.inf, -np.inf], 0).fillna(0)
    pharmacy_features["pharmacy_logistics_complexity_score"] = (pharmacy_features.get("num_floor_stock_pharmacy_orders", 0) +
        pharmacy_features.get("num_patient_may_take_own_med_orders", 0) +
        pharmacy_features.get("num_dosing_by_pharmacy_orders", 0) +
        pharmacy_features.get("num_discharge_med_proc_type_orders", 0) +
        pharmacy_features.get("num_unverified_pharmacy_orders", 0))
    pharmacy_features["pharmacy_infusion_complexity_score"] = (pharmacy_features.get("num_infusion_pharmacy_orders", 0) +
        pharmacy_features.get("had_lockout_interval_order", 0) + pharmacy_features.get("had_basal_rate_order", 0) +
        pharmacy_features.get("had_sliding_scale_order", 0) + pharmacy_features.get("had_one_hr_max_order", 0))
    for binary_col in ["had_prn_antipsychotic", "had_prn_benzodiazepine", "had_naloxone_order",
            "had_methadone_or_buprenorphine", "had_withdrawal_treatment_order", "had_lockout_interval_order",
            "had_basal_rate_order", "had_sliding_scale_order", "had_one_hr_max_order"]:
        if binary_col in pharmacy_features.columns:
            pharmacy_features[binary_col] = (pharmacy_features[binary_col] > 0).astype(int)
    pharmacy_features = pharmacy_features.drop(columns=["admittime", "dischtime"], errors="ignore")
else:
    pharmacy_features = df[id_cols].copy()

#loop through each item in this feature block
for col in pharmacy_engineered_cols:
    if col not in pharmacy_features.columns:
        pharmacy_features[col] = 0

df = df.drop(columns=[col for col in pharmacy_engineered_cols if col in df.columns], errors="ignore")
#join the derived features back to the admission table
df = df.merge(pharmacy_features[id_cols + pharmacy_engineered_cols], on=id_cols, how="left")
for col in pharmacy_engineered_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    if col != "psychotropic_order_density_per_day":
        df[col] = df[col].astype(int)

print("Pharmacy medication-intensity table-derived feature extraction complete.")
print("Pharmacy table-derived feature count:", len(pharmacy_engineered_cols))
print("Dataset shape after pharmacy features:", df.shape)

Starting pharmacy medication-intensity feature extraction...
Processed pharmacy chunk: 0 rows kept: 462964
Processed pharmacy chunk: 1 rows kept: 464266
Processed pharmacy chunk: 2 rows kept: 470947
Processed pharmacy chunk: 3 rows kept: 453763
Processed pharmacy chunk: 4 rows kept: 456580
Processed pharmacy chunk: 5 rows kept: 478238
Processed pharmacy chunk: 6 rows kept: 463489
Processed pharmacy chunk: 7 rows kept: 445428
Processed pharmacy chunk: 8 rows kept: 453305
Processed pharmacy chunk: 9 rows kept: 450548
Processed pharmacy chunk: 10 rows kept: 458192
Processed pharmacy chunk: 11 rows kept: 464805
Processed pharmacy chunk: 12 rows kept: 452873
Processed pharmacy chunk: 13 rows kept: 457683
Processed pharmacy chunk: 14 rows kept: 461595
Processed pharmacy chunk: 15 rows kept: 457044
Processed pharmacy chunk: 16 rows kept: 458229
Processed pharmacy chunk: 17 rows kept: 385047
Pharmacy medication-intensity table-derived feature extraction complete.
Pharmacy table-derived feature

**Create microbiology and infection features**

Microbiology events are summarised into test counts, positive-culture indicators, organism burden, and suspected infection flags. These features describe infection-related clinical burden during the admission.

In [ ]:
#microbiology and infection burden features
micro_usecols = ["subject_id", "hadm_id", "charttime", "spec_type_desc", "org_name", "interpretation"]
micro = pd.read_csv(hosp_path / "microbiologyevents.csv.gz", usecols=micro_usecols)
micro = micro[micro["hadm_id"].isin(current_hadm_ids)].copy()
if not micro.empty:
    micro["charttime"] = pd.to_datetime(micro["charttime"], errors="coerce")
    #join the derived features back to the admission table
    micro = micro.merge(current_admissions[["subject_id", "hadm_id", "admittime"]], on=id_cols, how="left")
    hours_from_admission = (micro["charttime"] - micro["admittime"]).dt.total_seconds() / 3600
    spec_text = micro["spec_type_desc"].astype(str).str.lower()
    org_text = micro["org_name"].astype(str).str.lower()
    interpretation_text = micro["interpretation"].astype(str).str.lower()
    micro["positive_culture_flag"] = ((micro["org_name"].notna()) & ~org_text.isin(["nan", "none", "no growth"])).astype(int)
    micro["blood_culture_positive"] = (spec_text.str.contains("blood", regex=False) & (micro["positive_culture_flag"] == 1)).astype(int)
    micro["urine_culture_positive"] = (spec_text.str.contains("urine", regex=False) & (micro["positive_culture_flag"] == 1)).astype(int)
    micro["respiratory_culture_positive"] = (spec_text.str.contains("sputum|respiratory|tracheal|bronch", regex=True) & (micro["positive_culture_flag"] == 1)).astype(int)
    micro["culture_positive_first_72h"] = ((micro["positive_culture_flag"] == 1) & (hours_from_admission >= 0) & (hours_from_admission <= 72)).astype(int)
    micro["abnormal_microbiology_interpretation"] = interpretation_text.str.contains("positive|abnormal|resistant", regex=True).fillna(False).astype(int)
    #summarise row-level records to admission-level features
    micro_features = micro.groupby(id_cols, as_index=False).agg(
        microbiology_test_count=("spec_type_desc", "size"),
        positive_culture_flag=("positive_culture_flag", "max"),
        blood_culture_positive=("blood_culture_positive", "max"),
        urine_culture_positive=("urine_culture_positive", "max"),
        respiratory_culture_positive=("respiratory_culture_positive", "max"),
        num_positive_blood_culture=("blood_culture_positive", "sum"),
        num_positive_urine_culture=("urine_culture_positive", "sum"),
        num_positive_respiratory_culture=("respiratory_culture_positive", "sum"),
        culture_positive_first_72h=("culture_positive_first_72h", "max"),
        abnormal_microbiology_interpretation=("abnormal_microbiology_interpretation", "max"),
        distinct_organism_count=("org_name", lambda x: x.dropna().nunique()),
        num_distinct_specimen_types=("spec_type_desc", lambda x: x.dropna().nunique()))
    micro_features["num_distinct_organisms"] = micro_features["distinct_organism_count"]
    micro_features["polymicrobial_flag"] = (micro_features["num_distinct_organisms"] >= 2).astype(int)
else:
    micro_features = pd.DataFrame(columns=id_cols)

df = df.drop(columns=[col for col in micro_features.columns if col not in id_cols], errors="ignore")
#join the derived features back to the admission table
df = df.merge(micro_features, on=id_cols, how="left")
microbiology_engineered_cols = [col for col in micro_features.columns if col not in id_cols]
#loop through each item in this feature block
for col in microbiology_engineered_cols:
    df[col] = df[col].fillna(0).astype(int)
if "microbiology_test_count" not in df.columns:
    df["microbiology_test_count"] = 0
if "had_antibiotic_exposure" not in df.columns:
    df["had_antibiotic_exposure"] = 0

microbiology_test_count_series = pd.to_numeric(df["microbiology_test_count"], errors="coerce").fillna(0)
df["has_microbiology_test"] = (microbiology_test_count_series > 0).astype(int)
df["suspected_infection_flag"] = ((df["has_microbiology_test"] == 1) & (df["had_antibiotic_exposure"] == 1)).astype(int)
df["infection_burden_category"] = pd.cut(microbiology_test_count_series,
    bins=[-1, 0, 2, 5, np.inf], labels=["0 tests", "1-2 tests", "3-5 tests", "6+ tests"])

/tmp/ipykernel_52805/3942877897.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["has_microbiology_test"] = (microbiology_test_count_series > 0).astype(int)
/tmp/ipykernel_52805/3942877897.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["suspected_infection_flag"] = ((df["has_microbiology_test"] == 1) & (df["had_antibiotic_exposure"] == 1)).astype(int)
/tmp/ipykernel_52805/3942877897.py:53: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which h

**Create OMR body measure features**

The latest available BMI, weight, and height measurements before admission are attached to each admission. Missingness flags are kept because absence of body-measure data can itself be informative in routinely collected records.

In [ ]:
#omr body measure features: latest available bmi, weight, and height before admission
omr = pd.read_csv(hosp_path / "omr.csv.gz")
omr = omr[omr["subject_id"].isin(current_subject_ids)].copy()
omr["chartdate"] = pd.to_datetime(omr["chartdate"], errors="coerce")
omr["result_value_numeric"] = pd.to_numeric(omr["result_value"].astype(str).str.extract(r"([-+]?\d*\.?\d+)")[0], errors="coerce")
omr_name = omr["result_name"].astype(str).str.lower()
omr["omr_measure"] = np.select([omr_name.str.contains("bmi", regex=False),
        omr_name.str.contains("weight", regex=False), omr_name.str.contains("height", regex=False)],
    ["bmi", "weight", "height"], default="")
omr = omr[(omr["omr_measure"] != "") & omr["result_value_numeric"].notna()].copy()
omr.loc[(omr["omr_measure"] == "weight") & omr_name.str.contains("lb|pound", regex=True), "result_value_numeric"] = (
    omr.loc[(omr["omr_measure"] == "weight") & omr_name.str.contains("lb|pound", regex=True), "result_value_numeric"] * 0.453592)
omr.loc[(omr["omr_measure"] == "height") & omr_name.str.contains("inch|inches", regex=True), "result_value_numeric"] = (
    omr.loc[(omr["omr_measure"] == "height") & omr_name.str.contains("inch|inches", regex=True), "result_value_numeric"] * 2.54)

#summarise row-level records to admission-level features
omr_groups = {(sid, measure): group.sort_values("chartdate") for (sid, measure), group in omr.groupby(["subject_id", "omr_measure"])}
omr_rows = []
#loop through each item in this feature block
for _, row in current_admissions.iterrows():
    result = {"subject_id": row["subject_id"], "hadm_id": row["hadm_id"],
        "latest_bmi_before_admission": np.nan, "previous_bmi_before_admission": np.nan,
        "latest_weight_kg_before_admission": np.nan, "previous_weight_kg_before_admission": np.nan,
        "latest_height_cm_before_admission": np.nan, "previous_height_cm_before_admission": np.nan,
        "days_since_latest_body_measure": np.nan}
    for measure, output_col in [("bmi", "latest_bmi_before_admission"), ("weight", "latest_weight_kg_before_admission"),
            ("height", "latest_height_cm_before_admission")]:
        group = omr_groups.get((row["subject_id"], measure))
        if group is not None and pd.notna(row["admittime"]):
            measure_dates = group["chartdate"].to_numpy(dtype="datetime64[ns]")
            admission_time64 = np.datetime64(row["admittime"])
            previous_index = np.searchsorted(measure_dates, admission_time64, side="right") - 1
            if previous_index >= 0:
                result[output_col] = group.iloc[previous_index]["result_value_numeric"]
                days_since_measure = (row["admittime"] - group.iloc[previous_index]["chartdate"]).total_seconds() / 86400
                if pd.isna(result["days_since_latest_body_measure"]):
                    result["days_since_latest_body_measure"] = days_since_measure
                else:
                    result["days_since_latest_body_measure"] = min(result["days_since_latest_body_measure"], days_since_measure)
            if previous_index >= 1:
                previous_output_col = output_col.replace("latest_", "previous_")
                result[previous_output_col] = group.iloc[previous_index - 1]["result_value_numeric"]
    omr_rows.append(result)

omr_features = pd.DataFrame(omr_rows)
df = df.drop(columns=[col for col in omr_features.columns if col not in id_cols], errors="ignore")
#join the derived features back to the admission table
df = df.merge(omr_features, on=id_cols, how="left")
df["bmi_change_recent"] = df["latest_bmi_before_admission"] - df["previous_bmi_before_admission"]
df["weight_change_recent"] = df["latest_weight_kg_before_admission"] - df["previous_weight_kg_before_admission"]
df["height_change_recent"] = df["latest_height_cm_before_admission"] - df["previous_height_cm_before_admission"]
df["missing_bmi_flag"] = df["latest_bmi_before_admission"].isna().astype(int)
df["obesity_flag"] = (df["latest_bmi_before_admission"] >= 30).fillna(False).astype(int)
df["underweight_flag"] = (df["latest_bmi_before_admission"] < 18.5).fillna(False).astype(int)
df["bmi_category"] = pd.cut(df["latest_bmi_before_admission"], bins=[0, 18.5, 25, 30, np.inf],
    labels=["Underweight", "Normal", "Overweight", "Obese"])
df["bmi_category"] = df["bmi_category"].astype("object").fillna("Missing")

/tmp/ipykernel_52805/780337511.py:49: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["bmi_change_recent"] = df["latest_bmi_before_admission"] - df["previous_bmi_before_admission"]
/tmp/ipykernel_52805/780337511.py:50: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["weight_change_recent"] = df["latest_weight_kg_before_admission"] - df["previous_weight_kg_before_admission"]
/tmp/ipykernel_52805/780337511.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times,

**Create service and transfer timing features**

Service transitions and transfer timing are summarised to capture care-pathway movement during the admission. These features describe whether care stayed on one service or moved between medical, surgical, ICU, and psychiatric pathways.

In [ ]:
#service and transfer timing features
services = pd.read_csv(hosp_path / "services.csv.gz")
services = services[services["hadm_id"].isin(current_hadm_ids)].copy()
if not services.empty:
    services["transfertime"] = pd.to_datetime(services["transfertime"], errors="coerce")
    services["prev_service_clean"] = services["prev_service"].astype(str).str.upper()
    services["curr_service_clean"] = services["curr_service"].astype(str).str.upper()
    services["medical_to_psych_service_transfer"] = ((~services["prev_service_clean"].str.contains("PSYCH", regex=False)) & services["curr_service_clean"].str.contains("PSYCH", regex=False)).astype(int)
    services["psych_to_medical_service_transfer"] = (services["prev_service_clean"].str.contains("PSYCH", regex=False) & (~services["curr_service_clean"].str.contains("PSYCH", regex=False))).astype(int)
    services["psych_service_involved_anytime"] = (services["prev_service_clean"].str.contains("PSYCH", regex=False) |
        services["curr_service_clean"].str.contains("PSYCH", regex=False)).astype(int)
    services["medicine_service_involved_anytime"] = (services["prev_service_clean"].str.contains("MED", regex=False) |
        services["curr_service_clean"].str.contains("MED", regex=False)).astype(int)
    #summarise row-level records to admission-level features
    service_timing_features = services.groupby(id_cols, as_index=False).agg(
        medical_to_psych_service_transfer=("medical_to_psych_service_transfer", "max"),
        psych_to_medical_service_transfer=("psych_to_medical_service_transfer", "max"),
        psych_service_involved_anytime=("psych_service_involved_anytime", "max"),
        medicine_service_involved_anytime=("medicine_service_involved_anytime", "max"))
    service_timing_features["medicine_and_psych_services_both_flag"] = ((service_timing_features["medicine_service_involved_anytime"] == 1) &
        (service_timing_features["psych_service_involved_anytime"] == 1)).astype(int)
else:
    service_timing_features = pd.DataFrame(columns=id_cols)

df = df.drop(columns=[col for col in service_timing_features.columns if col not in id_cols], errors="ignore")
#join the derived features back to the admission table
df = df.merge(service_timing_features, on=id_cols, how="left")
for col in ["medical_to_psych_service_transfer", "psych_to_medical_service_transfer", "psych_service_involved_anytime",
        "medicine_service_involved_anytime", "medicine_and_psych_services_both_flag"]:
    df[col] = df[col].fillna(0).astype(int)
first_service_series = df["first_service"] if "first_service" in df.columns else pd.Series("", index=df.index)
last_service_series = df["last_service"] if "last_service" in df.columns else pd.Series("", index=df.index)
df["same_first_last_service"] = (first_service_series.astype(str) == last_service_series.astype(str)).astype(int)

transfers = pd.read_csv(hosp_path / "transfers.csv.gz")
transfers = transfers[transfers["hadm_id"].isin(current_hadm_ids)].copy()
if not transfers.empty:
    transfers["intime"] = pd.to_datetime(transfers["intime"], errors="coerce")
    transfer_features = transfers.sort_values("intime").groupby(id_cols, as_index=False).agg(
        first_transfer_time=("intime", "first"),
        last_careunit_before_discharge=("careunit", lambda x: x.dropna().iloc[-1] if len(x.dropna()) > 0 else "Unknown"))
    #join the derived features back to the admission table
    transfer_features = transfer_features.merge(current_admissions[id_cols + ["admittime"]], on=id_cols, how="left")
    transfer_features["time_to_first_transfer_hours"] = ((transfer_features["first_transfer_time"] - transfer_features["admittime"]).dt.total_seconds() / 3600)
    transfer_features.loc[transfer_features["time_to_first_transfer_hours"] < 0, "time_to_first_transfer_hours"] = np.nan
    transfer_features = transfer_features.drop(columns=["first_transfer_time", "admittime"])
else:
    transfer_features = pd.DataFrame(columns=id_cols)

df = df.drop(columns=[col for col in transfer_features.columns if col not in id_cols], errors="ignore")
df = df.merge(transfer_features, on=id_cols, how="left")
df["last_careunit_before_discharge"] = df["last_careunit_before_discharge"].fillna("No transfer record")

/tmp/ipykernel_52805/2383118014.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["same_first_last_service"] = (first_service_series.astype(str) == last_service_series.astype(str)).astype(int)


**Add remaining ICD and procedure specificity features**

This block adds finer diagnosis and procedure specificity after the broad psychiatric and comorbidity indicators have already been created. It includes substance-use subtypes, simplified medical comorbidity flags, procedure family indicators, and compact procedure burden summaries.

The aim is to make broad diagnosis counts more interpretable. For example, two admissions may have the same diagnosis burden but differ clinically if one includes psychosis/substance-use complexity and another reflects cardiopulmonary or renal illness.


In [ ]:
#add icd and procedure specificity features
#sections:
#1. map diagnosis/procedure codes to interpretable psychiatric and medical groups.
#2. add procedure-specific flags such as ect, restraints, dialysis, ventilation, and central lines.
#3. keep the features as grouped clinical indicators rather than sparse raw icd/procedure codes.
print("Starting ICD/procedure specificity extraction...")

additional_substance_cols = ["has_cannabis_related_disorder", "has_tobacco_or_nicotine_related_disorder",
    "has_sedative_hypnotic_related_disorder", "has_polysubstance_related_disorder"]
additional_elixhauser_cols = ["elixhauser_congestive_heart_failure", "elixhauser_valvular_disease",
    "elixhauser_pulmonary_circulation", "elixhauser_peripheral_vascular", "elixhauser_hypertension",
    "elixhauser_paralysis", "elixhauser_other_neurological", "elixhauser_chronic_pulmonary",
    "elixhauser_diabetes_uncomplicated", "elixhauser_diabetes_complicated", "elixhauser_hypothyroidism",
    "elixhauser_renal_failure", "elixhauser_liver_disease", "elixhauser_peptic_ulcer",
    "elixhauser_aids_hiv", "elixhauser_lymphoma", "elixhauser_metastatic_cancer",
    "elixhauser_solid_tumour", "elixhauser_rheumatoid_collagen", "elixhauser_coagulopathy",
    "elixhauser_obesity", "elixhauser_weight_loss", "elixhauser_fluid_electrolyte",
    "elixhauser_blood_loss_anaemia", "elixhauser_deficiency_anaemia", "elixhauser_alcohol_abuse",
    "elixhauser_drug_abuse", "elixhauser_psychoses", "elixhauser_depression"]

if "index_dx" not in globals():
    diagnoses = pd.read_csv(hosp_path / "diagnoses_icd.csv.gz", usecols=["subject_id", "hadm_id", "icd_code", "icd_version"])
    diagnoses = diagnoses[diagnoses["subject_id"].isin(current_subject_ids)].copy()
    diagnoses["icd_code_clean"] = diagnoses["icd_code"].astype(str).str.upper().str.replace(".", "", regex=False).str.strip()
    diagnoses["icd3"] = diagnoses["icd_code_clean"].str[:3]
    diagnoses["icd4"] = diagnoses["icd_code_clean"].str[:4]
    diagnoses["icd3_numeric"] = pd.to_numeric(diagnoses["icd_code_clean"].str.extract(r"^(\d{3})")[0], errors="coerce")
    index_dx = diagnoses[diagnoses["hadm_id"].isin(current_hadm_ids)].copy()

#finer substance-use icd categories
index_dx["has_cannabis_related_disorder"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith("F12")) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith(("3043", "3052")))).astype(int)
index_dx["has_tobacco_or_nicotine_related_disorder"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith("F17")) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith(("3051", "V1582")))).astype(int)
index_dx["has_sedative_hypnotic_related_disorder"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith("F13")) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith(("3041", "3054")))).astype(int)
index_dx["has_polysubstance_related_disorder"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith("F19")) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith(("3048", "3049")))).astype(int)

#individual simplified elixhauser-style comorbidity flags based on broad icd-9/icd-10 code families
index_dx["elixhauser_congestive_heart_failure"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith("I50")) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith("428"))).astype(int)
index_dx["elixhauser_valvular_disease"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("I05", "I06", "I07", "I08", "I09", "I34", "I35", "I36", "I37", "I38"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith(("394", "395", "396", "397", "424")))).astype(int)
index_dx["elixhauser_pulmonary_circulation"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("I26", "I27", "I28"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith(("415", "416", "417")))).astype(int)
index_dx["elixhauser_peripheral_vascular"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("I70", "I71", "I72", "I73", "I74", "I77"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith(("440", "441", "443", "444", "447")))).astype(int)
index_dx["elixhauser_hypertension"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("I10", "I11", "I12", "I13", "I15"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd3_numeric"].between(401, 405))).astype(int)
index_dx["elixhauser_paralysis"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("G81", "G82"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith(("342", "343", "344")))).astype(int)
index_dx["elixhauser_other_neurological"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("G10", "G11", "G12", "G20", "G21", "G22", "G25", "G31", "G32", "G35", "G36", "G37", "G40", "G41"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith(("331", "332", "333", "334", "335", "340", "341", "345")))).astype(int)
index_dx["elixhauser_chronic_pulmonary"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("J40", "J41", "J42", "J43", "J44", "J45", "J46", "J47"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd3_numeric"].between(490, 496))).astype(int)
index_dx["elixhauser_diabetes_uncomplicated"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("E100", "E101", "E109", "E110", "E111", "E119"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith(("2500", "2501", "2502", "2503")))).astype(int)
index_dx["elixhauser_diabetes_complicated"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("E102", "E103", "E104", "E105", "E106", "E107", "E112", "E113", "E114", "E115", "E116", "E117"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith(("2504", "2505", "2506", "2507", "2508", "2509")))).astype(int)
index_dx["elixhauser_hypothyroidism"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith("E03")) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith("244"))).astype(int)
index_dx["elixhauser_renal_failure"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("N17", "N18", "N19"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith(("584", "585", "586")))).astype(int)
index_dx["elixhauser_liver_disease"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("K70", "K71", "K72", "K73", "K74", "K75", "K76"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd3_numeric"].between(570, 573))).astype(int)
index_dx["elixhauser_peptic_ulcer"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("K25", "K26", "K27", "K28"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith(("531", "532", "533", "534")))).astype(int)
index_dx["elixhauser_aids_hiv"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("B20", "B21", "B22", "B24"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith(("042", "043", "044")))).astype(int)
index_dx["elixhauser_lymphoma"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("C81", "C82", "C83", "C84", "C85", "C88", "C90", "C91", "C92", "C93", "C94", "C95"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith(("200", "201", "202", "203", "204", "205", "206", "207", "208")))).astype(int)
index_dx["elixhauser_metastatic_cancer"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("C77", "C78", "C79", "C80"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd3_numeric"].between(196, 199))).astype(int)
index_dx["elixhauser_solid_tumour"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith("C")) |
    ((index_dx["icd_version"] == 9) & index_dx["icd3_numeric"].between(140, 195))).astype(int)
index_dx["elixhauser_rheumatoid_collagen"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("M05", "M06", "M32", "M33", "M34", "M35"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith(("710", "714")))).astype(int)
index_dx["elixhauser_coagulopathy"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("D65", "D66", "D67", "D68"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith(("286", "2871", "2873", "2874", "2875")))).astype(int)
index_dx["elixhauser_obesity"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith("E66")) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith("2780"))).astype(int)
index_dx["elixhauser_weight_loss"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("R634", "E43", "E44"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith(("260", "261", "262", "263", "7832")))).astype(int)
index_dx["elixhauser_fluid_electrolyte"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith("E87")) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith("276"))).astype(int)
index_dx["elixhauser_blood_loss_anaemia"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith("D500")) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith("2800"))).astype(int)
index_dx["elixhauser_deficiency_anaemia"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("D50", "D51", "D52", "D53"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith(("280", "281")))).astype(int)
index_dx["elixhauser_alcohol_abuse"] = index_dx.get("has_alcohol_related_disorder", 0).astype(int)
index_dx["elixhauser_drug_abuse"] = index_dx[["has_opioid_related_disorder", "has_stimulant_related_disorder",
    "has_cannabis_related_disorder", "has_sedative_hypnotic_related_disorder", "has_polysubstance_related_disorder"]].max(axis=1).astype(int)
index_dx["elixhauser_psychoses"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("F20", "F21", "F22", "F23", "F24", "F25", "F28", "F29"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd3_numeric"].between(295, 299))).astype(int)
index_dx["elixhauser_depression"] = (((index_dx["icd_version"] == 10) & index_dx["icd_code_clean"].str.startswith(("F32", "F33"))) |
    ((index_dx["icd_version"] == 9) & index_dx["icd_code_clean"].str.startswith(("2962", "2963", "311")))).astype(int)

#summarise row-level records to admission-level features
additional_icd_features = index_dx.groupby(id_cols, as_index=False)[additional_substance_cols + additional_elixhauser_cols].max()
additional_icd_features["finer_substance_use_category_count"] = additional_icd_features[additional_substance_cols].sum(axis=1)
additional_icd_features["elixhauser_individual_group_count"] = additional_icd_features[additional_elixhauser_cols].sum(axis=1)

df = df.drop(columns=[col for col in additional_icd_features.columns if col not in id_cols], errors="ignore")
#join the derived features back to the admission table
df = df.merge(additional_icd_features, on=id_cols, how="left")
additional_icd_engineered_cols = additional_substance_cols + ["finer_substance_use_category_count"] + additional_elixhauser_cols + ["elixhauser_individual_group_count"]
#loop through each item in this feature block
for col in additional_icd_engineered_cols:
    df[col] = df[col].fillna(0).astype(int)

#procedure specificity from hospital procedure icd codes and icu procedure event descriptions
procedure_specificity_cols = ["had_mechanical_ventilation_procedure", "had_dialysis_procedure",
    "had_central_line_procedure", "had_ect_procedure", "had_restraint_related_procedure",
    "ect_procedure_count", "ect_within_first_72h", "ect_during_admission_flag"]
procedure_features = pd.DataFrame(columns=id_cols + procedure_specificity_cols)

procedures_icd_path = hosp_path / "procedures_icd.csv.gz"
if procedures_icd_path.exists():
    procedures = pd.read_csv(procedures_icd_path, usecols=["subject_id", "hadm_id", "icd_code", "icd_version", "chartdate"])
    procedures = procedures[procedures["hadm_id"].isin(current_hadm_ids)].copy()
    procedures["icd_code_clean"] = procedures["icd_code"].astype(str).str.upper().str.replace(".", "", regex=False).str.strip()
    procedures["icd3_numeric"] = pd.to_numeric(procedures["icd_code_clean"].str.extract(r"^(\d{2,3})")[0], errors="coerce")
    procedures["had_mechanical_ventilation_procedure"] = (
        ((procedures["icd_version"] == 10) & procedures["icd_code_clean"].str.startswith(("5A1935Z", "5A1945Z", "5A1955Z"))) |
        ((procedures["icd_version"] == 9) & procedures["icd_code_clean"].str.startswith(("9670", "9671", "9672")))).astype(int)
    procedures["had_dialysis_procedure"] = (
        ((procedures["icd_version"] == 10) & procedures["icd_code_clean"].str.startswith(("5A1D", "3E1M39Z"))) |
        ((procedures["icd_version"] == 9) & procedures["icd_code_clean"].str.startswith(("3995", "5498")))).astype(int)
    procedures["had_central_line_procedure"] = (
        ((procedures["icd_version"] == 10) & procedures["icd_code_clean"].str.startswith(("02HV", "05HM", "05HN", "05HP", "05HQ"))) |
        ((procedures["icd_version"] == 9) & procedures["icd_code_clean"].str.startswith(("3893", "3897")))).astype(int)
    procedures["had_ect_procedure"] = (
        ((procedures["icd_version"] == 10) & procedures["icd_code_clean"].str.startswith("GZB")) |
        ((procedures["icd_version"] == 9) & procedures["icd_code_clean"].str.startswith("9427"))).astype(int)
    procedures["had_restraint_related_procedure"] = procedures["icd_code_clean"].str.startswith(("Z9114", "V4987")).astype(int)
    #join the derived features back to the admission table
    procedures = procedures.merge(current_admissions[id_cols + ["admittime"]], on=id_cols, how="left")
    procedures["chartdate"] = pd.to_datetime(procedures["chartdate"], errors="coerce")
    procedures["hours_from_admission"] = ((procedures["chartdate"] - procedures["admittime"]).dt.total_seconds() / 3600)
    procedures["ect_within_first_72h"] = ((procedures["had_ect_procedure"] == 1) &
        (procedures["hours_from_admission"] >= 0) & (procedures["hours_from_admission"] <= 72)).astype(int)
    procedures["ect_during_admission_flag"] = procedures["had_ect_procedure"]
    procedures["ect_procedure_count"] = procedures["had_ect_procedure"]
    #summarise row-level records to admission-level features
    procedure_features = procedures.groupby(id_cols, as_index=False).agg(
        had_mechanical_ventilation_procedure=("had_mechanical_ventilation_procedure", "max"),
        had_dialysis_procedure=("had_dialysis_procedure", "max"),
        had_central_line_procedure=("had_central_line_procedure", "max"),
        had_ect_procedure=("had_ect_procedure", "max"),
        had_restraint_related_procedure=("had_restraint_related_procedure", "max"),
        ect_procedure_count=("ect_procedure_count", "sum"),
        ect_within_first_72h=("ect_within_first_72h", "max"),
        ect_during_admission_flag=("ect_during_admission_flag", "max"))

procedureevents_path = icu_path / "procedureevents.csv.gz"
if procedureevents_path.exists():
    procedure_event_usecols = ["subject_id", "hadm_id", "ordercategoryname", "ordercategorydescription", "statusdescription"]
    procedure_event_features = []
    #loop through each item in this feature block
    for chunk_number, procedure_chunk in enumerate(pd.read_csv(procedureevents_path, usecols=procedure_event_usecols, chunksize=500000)):
        procedure_chunk = procedure_chunk[procedure_chunk["hadm_id"].isin(current_hadm_ids)].copy()
        if procedure_chunk.empty:
            continue
        procedure_text = (procedure_chunk["ordercategoryname"].fillna("") + " " +
            procedure_chunk["ordercategorydescription"].fillna("") + " " +
            procedure_chunk["statusdescription"].fillna("")).str.lower()
        procedure_chunk["had_mechanical_ventilation_procedure"] = procedure_text.str.contains("ventilation|ventilator|intubat", regex=True).astype(int)
        procedure_chunk["had_dialysis_procedure"] = procedure_text.str.contains("dialysis|crrt|cvvh|hemofiltration", regex=True).astype(int)
        procedure_chunk["had_central_line_procedure"] = procedure_text.str.contains("central line|central venous|cvc|arterial line", regex=True).astype(int)
        procedure_chunk["had_ect_procedure"] = procedure_text.str.contains("electroconvulsive|\bect\b", regex=True).astype(int)
        procedure_chunk["had_restraint_related_procedure"] = procedure_text.str.contains("restraint", regex=True).astype(int)
        procedure_chunk["ect_procedure_count"] = procedure_chunk["had_ect_procedure"]
        procedure_chunk["ect_within_first_72h"] = 0
        procedure_chunk["ect_during_admission_flag"] = procedure_chunk["had_ect_procedure"]
        #summarise row-level records to admission-level features
        procedure_event_features.append(procedure_chunk.groupby(id_cols, as_index=False).agg(
            had_mechanical_ventilation_procedure=("had_mechanical_ventilation_procedure", "max"),
            had_dialysis_procedure=("had_dialysis_procedure", "max"),
            had_central_line_procedure=("had_central_line_procedure", "max"),
            had_ect_procedure=("had_ect_procedure", "max"),
            had_restraint_related_procedure=("had_restraint_related_procedure", "max"),
            ect_procedure_count=("ect_procedure_count", "sum"),
            ect_within_first_72h=("ect_within_first_72h", "max"),
            ect_during_admission_flag=("ect_during_admission_flag", "max")))
        print("Processed ICU procedure event chunk:", chunk_number, "rows kept:", len(procedure_chunk))
    if procedure_event_features:
        procedure_event_features = pd.concat(procedure_event_features, ignore_index=True).groupby(id_cols, as_index=False).max()
        procedure_features = pd.concat([procedure_features, procedure_event_features], ignore_index=True).groupby(id_cols, as_index=False).max()

if procedure_features.empty:
    procedure_features = df[id_cols].copy()
    for col in procedure_specificity_cols:
        procedure_features[col] = 0

df = df.drop(columns=[col for col in procedure_features.columns if col not in id_cols], errors="ignore")
df = df.merge(procedure_features, on=id_cols, how="left")
for col in procedure_specificity_cols:
    df[col] = df[col].fillna(0).astype(int)

procedure_specificity_engineered_cols = procedure_specificity_cols

print("Additional ICD/procedure feature additions complete.")
print("Additional ICD table-derived feature count:", len(additional_icd_engineered_cols))
print("Procedure specificity table-derived feature count:", len(procedure_specificity_engineered_cols))
print("Dataset shape after additional ICD/procedure features:", df.shape)

Starting ICD/procedure specificity extraction...
Processed ICU procedure event chunk: 0 rows kept: 236908
Processed ICU procedure event chunk: 1 rows kept: 146752
Additional ICD/procedure feature additions complete.
Additional ICD table-derived feature count: 35
Procedure specificity table-derived feature count: 8
Dataset shape after additional ICD/procedure features: (238565, 726)


**Add EMAR medication administration features**

EMAR records describe whether ordered medications were actually administered, not given, delayed, refused, started, stopped, or otherwise interrupted. This section turns those repeated administration rows into admission-level process features.

This is useful because medication orders alone do not show whether medication delivery was smooth. Not-given rates, refusals, and near-discharge medication friction can signal instability or care complexity that is different from simple medication exposure.


In [ ]:
#add actual medication administration table-derived columns from emar and emar detail tables
#sections in this cell:
#1. summarise actual medication administration events and first 24h/72h windows.
#2. derive not-given/refusal/process-friction table-derived columns from emar detail.
#3. keep administration complexity as rates/counts so it can complement prescription/order intent.
print("Starting EMAR medication-administration feature extraction...")

emar_engineered_cols = ["emar_administration_record_count", "emar_unique_medication_count",
    "emar_administered_event_count", "emar_not_given_event_count", "emar_missed_or_not_given_flag",
    "emar_administration_count_first_24h", "emar_administration_count_first_72h",
    "emar_iv_administration_count", "emar_had_iv_administration", "emar_infusion_administration_count",
    "emar_had_infusion_administration", "emar_complete_dose_not_given_count", "emar_complete_dose_not_given_flag",
    "emar_dose_given_record_count", "emar_new_iv_bag_count", "emar_missed_or_not_given_count_first_24h",
    "emar_missed_or_not_given_count_first_72h", "emar_not_given_rate", "emar_complete_dose_not_given_rate",
    "emar_administered_to_order_ratio", "num_no_barcode_med_admins", "no_barcode_med_admin_rate",
    "num_nonformulary_visual_verification", "num_dose_due_records", "dose_due_to_given_gap_count",
    "num_admins_with_remaining_dose", "emar_refused_event_count", "emar_refusal_rate",
    "emar_psychotropic_refused_count", "emar_psychotropic_refusal_rate", "emar_psychotropic_not_given_count",
    "emar_nonpsychotropic_not_given_count", "emar_delayed_administration_count",
    "emar_total_delay_hours", "emar_mean_delay_hours"]

emar_path = hosp_path / "emar.csv.gz"
emar_detail_path = hosp_path / "emar_detail.csv.gz"
emar_key_rows = []
emar_medication_rows = []
emar_feature_rows = []

#medication administration events from emar
if emar_path.exists():
    emar_usecols = ["subject_id", "hadm_id", "emar_id", "emar_seq", "charttime", "scheduletime", "medication", "event_txt"]
    admission_times_for_emar = current_admissions[["hadm_id", "admittime", "dischtime"]].copy()
    #loop through each item in this feature block
    for chunk_number, emar_chunk in enumerate(pd.read_csv(emar_path, usecols=emar_usecols, chunksize=1000000)):
        emar_chunk = emar_chunk[emar_chunk["hadm_id"].isin(current_hadm_ids)].copy()
        if emar_chunk.empty:
            continue
        emar_chunk["charttime"] = pd.to_datetime(emar_chunk["charttime"], errors="coerce")
        emar_chunk["scheduletime"] = pd.to_datetime(emar_chunk["scheduletime"], errors="coerce")
        #join the derived features back to the admission table
        emar_chunk = emar_chunk.merge(admission_times_for_emar, on="hadm_id", how="left")
        emar_chunk = emar_chunk[(emar_chunk["charttime"].notna()) & (emar_chunk["admittime"].notna()) &
            (emar_chunk["charttime"] >= emar_chunk["admittime"]) &
            ((emar_chunk["dischtime"].isna()) | (emar_chunk["charttime"] <= emar_chunk["dischtime"]))].copy()
        if emar_chunk.empty:
            continue
        hours_from_admission = (emar_chunk["charttime"] - emar_chunk["admittime"]).dt.total_seconds() / 3600
        hours_to_discharge = (emar_chunk["dischtime"] - emar_chunk["charttime"]).dt.total_seconds() / 3600
        event_text = emar_chunk["event_txt"].fillna("").str.lower()
        medication_text = emar_chunk["medication"].fillna("").str.lower()
        psychotropic_text = medication_text.str.contains(
            "haloperidol|olanzapine|quetiapine|risperidone|clozapine|aripiprazole|ziprasidone|lurasidone|paliperidone|fluphenazine|chlorpromazine|lorazepam|diazepam|clonazepam|alprazolam|midazolam|lithium|valpro|divalproex|carbamazepine|lamotrigine|sertraline|fluoxetine|citalopram|escitalopram|paroxetine|venlafaxine|duloxetine|mirtazapine|trazodone", regex=True)
        emar_chunk["emar_administered_event"] = event_text.str.contains("administer|given|new bag|restart|rate change", regex=True).astype(int)
        emar_chunk["emar_not_given_event"] = event_text.str.contains("not given|missed|held|refused|stopped|cancel", regex=True).astype(int)
        emar_chunk["emar_refused_event"] = event_text.str.contains("refused|patient refused|declined", regex=True).astype(int)
        delay_hours = ((emar_chunk["charttime"] - emar_chunk["scheduletime"]).dt.total_seconds() / 3600).clip(lower=0)
        emar_chunk["emar_delayed_administration"] = ((emar_chunk["emar_administered_event"] == 1) & delay_hours.gt(2)).astype(int)
        emar_chunk["emar_delay_hours"] = np.where(emar_chunk["emar_delayed_administration"] == 1, delay_hours.fillna(0), 0)
        emar_chunk["emar_started_event"] = event_text.str.contains("started|start", regex=True).astype(int)
        emar_chunk["emar_stopped_event"] = event_text.str.contains("stopped|stop|discontinued", regex=True).astype(int)
        emar_chunk["emar_started_stopped_last_24h"] = (((emar_chunk["emar_started_event"] == 1) |
            (emar_chunk["emar_stopped_event"] == 1)) & hours_to_discharge.between(0, 24, inclusive="both")).astype(int)
        emar_chunk["emar_delayed_or_not_given_last_24h"] = (((emar_chunk["emar_delayed_administration"] == 1) |
            (emar_chunk["emar_not_given_event"] == 1)) & hours_to_discharge.between(0, 24, inclusive="both")).astype(int)
        emar_chunk["emar_psychotropic_refused"] = ((emar_chunk["emar_refused_event"] == 1) & psychotropic_text).astype(int)
        emar_chunk["emar_psychotropic_not_given"] = ((emar_chunk["emar_not_given_event"] == 1) & psychotropic_text).astype(int)
        emar_chunk["emar_nonpsychotropic_not_given"] = ((emar_chunk["emar_not_given_event"] == 1) & ~psychotropic_text).astype(int)
        emar_chunk["emar_first_24h"] = ((hours_from_admission >= 0) & (hours_from_admission <= 24)).astype(int)
        emar_chunk["emar_first_72h"] = ((hours_from_admission >= 0) & (hours_from_admission <= 72)).astype(int)
        emar_chunk["emar_not_given_first_24h"] = ((emar_chunk["emar_not_given_event"] == 1) & (emar_chunk["emar_first_24h"] == 1)).astype(int)
        emar_chunk["emar_not_given_first_72h"] = ((emar_chunk["emar_not_given_event"] == 1) & (emar_chunk["emar_first_72h"] == 1)).astype(int)
        #summarise row-level records to admission-level features
        emar_feature_rows.append(emar_chunk.groupby(id_cols, as_index=False).agg(
            emar_administration_record_count=("emar_id", "count"),
            emar_administered_event_count=("emar_administered_event", "sum"),
            emar_not_given_event_count=("emar_not_given_event", "sum"),
            emar_administration_count_first_24h=("emar_first_24h", "sum"),
            emar_administration_count_first_72h=("emar_first_72h", "sum"),
            emar_missed_or_not_given_count_first_24h=("emar_not_given_first_24h", "sum"),
            emar_missed_or_not_given_count_first_72h=("emar_not_given_first_72h", "sum"),
            emar_refused_event_count=("emar_refused_event", "sum"),
            emar_psychotropic_refused_count=("emar_psychotropic_refused", "sum"),
            emar_psychotropic_not_given_count=("emar_psychotropic_not_given", "sum"),
            emar_nonpsychotropic_not_given_count=("emar_nonpsychotropic_not_given", "sum"),
            emar_delayed_administration_count=("emar_delayed_administration", "sum"),
            num_delayed_administered_events=("emar_delayed_administration", "sum"),
            num_started_emar_events=("emar_started_event", "sum"),
            num_stopped_emar_events=("emar_stopped_event", "sum"),
            emar_started_stopped_activity_last_24h=("emar_started_stopped_last_24h", "sum"),
            emar_delayed_or_not_given_last_24h=("emar_delayed_or_not_given_last_24h", "sum"),
            emar_total_delay_hours=("emar_delay_hours", "sum")))
        emar_key_rows.append(emar_chunk[["subject_id", "hadm_id", "emar_id", "emar_seq"]].drop_duplicates())
        emar_medication_rows.append(emar_chunk[id_cols + ["medication"]].dropna(subset=["medication"]).drop_duplicates())
        print("Processed EMAR chunk:", chunk_number, "rows kept:", len(emar_chunk))

#aggregate emar administration counts and friction rates
if emar_feature_rows:
    #summarise row-level records to admission-level features
    emar_features = pd.concat(emar_feature_rows, ignore_index=True).groupby(id_cols, as_index=False).agg(
        emar_administration_record_count=("emar_administration_record_count", "sum"),
        emar_administered_event_count=("emar_administered_event_count", "sum"),
        emar_not_given_event_count=("emar_not_given_event_count", "sum"),
        emar_administration_count_first_24h=("emar_administration_count_first_24h", "sum"),
        emar_administration_count_first_72h=("emar_administration_count_first_72h", "sum"),
        emar_missed_or_not_given_count_first_24h=("emar_missed_or_not_given_count_first_24h", "sum"),
        emar_missed_or_not_given_count_first_72h=("emar_missed_or_not_given_count_first_72h", "sum"),
        emar_refused_event_count=("emar_refused_event_count", "sum"),
        emar_psychotropic_refused_count=("emar_psychotropic_refused_count", "sum"),
        emar_psychotropic_not_given_count=("emar_psychotropic_not_given_count", "sum"),
        emar_nonpsychotropic_not_given_count=("emar_nonpsychotropic_not_given_count", "sum"),
        emar_delayed_administration_count=("emar_delayed_administration_count", "sum"),
        num_delayed_administered_events=("num_delayed_administered_events", "sum"),
        num_started_emar_events=("num_started_emar_events", "sum"),
        num_stopped_emar_events=("num_stopped_emar_events", "sum"),
        emar_started_stopped_activity_last_24h=("emar_started_stopped_activity_last_24h", "sum"),
        emar_delayed_or_not_given_last_24h=("emar_delayed_or_not_given_last_24h", "sum"),
        emar_total_delay_hours=("emar_total_delay_hours", "sum"))
    if emar_medication_rows:
        emar_medication_counts = pd.concat(emar_medication_rows, ignore_index=True).drop_duplicates().groupby(id_cols, as_index=False)["medication"].nunique()
        emar_medication_counts = emar_medication_counts.rename(columns={"medication": "emar_unique_medication_count"})
        emar_features = emar_features.merge(emar_medication_counts, on=id_cols, how="left")
    else:
        emar_features["emar_unique_medication_count"] = 0
    emar_features["emar_unique_medication_count"] = emar_features["emar_unique_medication_count"].fillna(0).astype(int)
    emar_features["emar_missed_or_not_given_flag"] = (emar_features["emar_not_given_event_count"] > 0).astype(int)
else:
    emar_features = df[id_cols].copy()
    for col in ["emar_administration_record_count", "emar_unique_medication_count", "emar_administered_event_count",
            "emar_not_given_event_count", "emar_administration_count_first_24h", "emar_administration_count_first_72h",
            "emar_missed_or_not_given_count_first_24h", "emar_missed_or_not_given_count_first_72h",
            "emar_missed_or_not_given_flag", "emar_refused_event_count", "emar_psychotropic_refused_count",
            "emar_psychotropic_not_given_count", "emar_nonpsychotropic_not_given_count",
            "emar_delayed_administration_count", "num_delayed_administered_events", "num_started_emar_events",
            "num_stopped_emar_events", "emar_started_stopped_activity_last_24h",
            "emar_delayed_or_not_given_last_24h", "emar_total_delay_hours"]:
        emar_features[col] = 0

#emar detail is joined through the emar keys so administration-detail features remain admission-level
emar_detail_feature_cols = ["emar_iv_administration_count", "emar_had_iv_administration",
    "emar_infusion_administration_count", "emar_had_infusion_administration", "emar_complete_dose_not_given_count",
    "emar_complete_dose_not_given_flag", "emar_dose_given_record_count", "emar_new_iv_bag_count",
    "num_no_barcode_med_admins", "num_nonformulary_visual_verification", "num_dose_due_records",
    "dose_due_to_given_gap_count", "num_admins_with_remaining_dose", "emar_refused_event_count", "emar_refusal_rate",
    "emar_psychotropic_refused_count", "emar_psychotropic_refusal_rate", "emar_psychotropic_not_given_count",
    "emar_nonpsychotropic_not_given_count", "emar_delayed_administration_count",
    "emar_total_delay_hours", "emar_mean_delay_hours"]

if emar_key_rows and emar_detail_path.exists():
    emar_keys = pd.concat(emar_key_rows, ignore_index=True).drop_duplicates()
    emar_detail_feature_rows = []
    emar_detail_usecols = ["subject_id", "emar_id", "emar_seq", "administration_type", "complete_dose_not_given",
        "dose_given", "dose_due", "barcode_type", "reason_for_no_barcode", "route", "infusion_rate", "new_iv_bag_hung",
        "continued_infusion_in_other_location", "infusion_complete", "product_amount_given", "will_remainder_of_dose_be_given",
        "non_formulary_visual_verification"]
    for chunk_number, detail_chunk in enumerate(pd.read_csv(emar_detail_path, usecols=emar_detail_usecols, chunksize=1000000, low_memory=False)):
        detail_chunk = detail_chunk.merge(emar_keys, on=["subject_id", "emar_id", "emar_seq"], how="inner")
        if detail_chunk.empty:
            continue
        route_text = detail_chunk["route"].fillna("").str.lower()
        admin_text = detail_chunk["administration_type"].fillna("").str.lower()
        detail_chunk["emar_iv_administration"] = route_text.str.contains("iv|intravenous|ivpb|iv drip", regex=True).astype(int)
        detail_chunk["emar_infusion_administration"] = (
            admin_text.str.contains("infusion", regex=True) |
            detail_chunk["infusion_rate"].notna() |
            detail_chunk["continued_infusion_in_other_location"].fillna("").astype(str).str.lower().isin(["1", "true", "yes", "y"]) |
            detail_chunk["infusion_complete"].fillna("").astype(str).str.lower().isin(["1", "true", "yes", "y"])).astype(int)
        detail_chunk["emar_complete_dose_not_given"] = detail_chunk["complete_dose_not_given"].fillna("").astype(str).str.lower().isin(["1", "true", "yes", "y"]).astype(int)
        detail_chunk["emar_dose_given_record"] = (pd.to_numeric(detail_chunk["dose_given"], errors="coerce").fillna(0) > 0).astype(int)
        detail_chunk["emar_new_iv_bag"] = detail_chunk["new_iv_bag_hung"].fillna("").astype(str).str.lower().isin(["1", "true", "yes", "y"]).astype(int)
        detail_chunk["no_barcode_med_admin"] = (detail_chunk["reason_for_no_barcode"].notna() |
            detail_chunk["barcode_type"].fillna("").astype(str).str.lower().str.contains("no barcode|manual|override", regex=True)).astype(int)
        detail_chunk["nonformulary_visual_verification"] = detail_chunk["non_formulary_visual_verification"].fillna("").astype(str).str.lower().isin(["1", "true", "yes", "y"]).astype(int)
        dose_due_numeric = pd.to_numeric(detail_chunk["dose_due"], errors="coerce")
        dose_given_numeric = pd.to_numeric(detail_chunk["dose_given"], errors="coerce")
        detail_chunk["dose_due_record"] = dose_due_numeric.notna().astype(int)
        detail_chunk["dose_due_to_given_gap"] = ((dose_due_numeric.notna()) & (dose_given_numeric.notna()) &
            (dose_due_numeric != dose_given_numeric)).astype(int)
        detail_chunk["admin_with_remaining_dose"] = detail_chunk["will_remainder_of_dose_be_given"].fillna("").astype(str).str.lower().isin(["1", "true", "yes", "y"]).astype(int)
        emar_detail_feature_rows.append(detail_chunk.groupby(id_cols, as_index=False).agg(
            emar_iv_administration_count=("emar_iv_administration", "sum"),
            emar_infusion_administration_count=("emar_infusion_administration", "sum"),
            emar_complete_dose_not_given_count=("emar_complete_dose_not_given", "sum"),
            emar_dose_given_record_count=("emar_dose_given_record", "sum"),
            emar_new_iv_bag_count=("emar_new_iv_bag", "sum"),
            num_no_barcode_med_admins=("no_barcode_med_admin", "sum"),
            num_nonformulary_visual_verification=("nonformulary_visual_verification", "sum"),
            num_dose_due_records=("dose_due_record", "sum"),
            dose_due_to_given_gap_count=("dose_due_to_given_gap", "sum"),
            num_admins_with_remaining_dose=("admin_with_remaining_dose", "sum")))
        print("Processed EMAR detail chunk:", chunk_number, "rows kept:", len(detail_chunk))
    if emar_detail_feature_rows:
        emar_detail_features = pd.concat(emar_detail_feature_rows, ignore_index=True).groupby(id_cols, as_index=False).sum()
        emar_detail_features["emar_had_iv_administration"] = (emar_detail_features["emar_iv_administration_count"] > 0).astype(int)
        emar_detail_features["emar_had_infusion_administration"] = (emar_detail_features["emar_infusion_administration_count"] > 0).astype(int)
        emar_detail_features["emar_complete_dose_not_given_flag"] = (emar_detail_features["emar_complete_dose_not_given_count"] > 0).astype(int)
    else:
        emar_detail_features = df[id_cols].copy()
        #loop through each item in this feature block
        for col in emar_detail_feature_cols:
            emar_detail_features[col] = 0
else:
    emar_detail_features = df[id_cols].copy()
    for col in emar_detail_feature_cols:
        emar_detail_features[col] = 0

#join the derived features back to the admission table
emar_features = emar_features.merge(emar_detail_features, on=id_cols, how="left")
df = df.drop(columns=[col for col in emar_features.columns if col not in id_cols], errors="ignore")
df = df.merge(emar_features, on=id_cols, how="left")
for col in [col for col in emar_engineered_cols if not col.endswith("_rate") and
        col not in ["emar_administered_to_order_ratio", "no_barcode_med_admin_rate", "emar_total_delay_hours",
            "emar_mean_delay_hours"]]:
    if col not in df.columns:
        df[col] = 0
    df[col] = df[col].fillna(0).astype(int)

df["emar_not_given_rate"] = np.where(df["emar_administration_record_count"] > 0,
    df["emar_not_given_event_count"] / df["emar_administration_record_count"], 0)
df["emar_complete_dose_not_given_rate"] = np.where(df["emar_administration_record_count"] > 0,
    df["emar_complete_dose_not_given_count"] / df["emar_administration_record_count"], 0)
df["no_barcode_med_admin_rate"] = np.where(df["emar_administration_record_count"] > 0,
    df["num_no_barcode_med_admins"] / df["emar_administration_record_count"], 0)
df["emar_refusal_rate"] = np.where(df["emar_administration_record_count"] > 0,
    df["emar_refused_event_count"] / df["emar_administration_record_count"], 0)
df["emar_psychotropic_refusal_rate"] = np.where(df["emar_administration_record_count"] > 0,
    df["emar_psychotropic_refused_count"] / df["emar_administration_record_count"], 0)
df["delayed_administered_rate"] = np.where(df["emar_administration_record_count"] > 0,
    df["num_delayed_administered_events"] / df["emar_administration_record_count"], 0)
df["emar_mean_delay_hours"] = np.where(df["emar_delayed_administration_count"] > 0,
    df["emar_total_delay_hours"] / df["emar_delayed_administration_count"], 0)
if "num_prescription_rows" in df.columns:
    df["emar_administered_to_order_ratio"] = np.where(df["num_prescription_rows"] > 0,
        df["emar_administered_event_count"] / df["num_prescription_rows"], 0)
else:
    df["emar_administered_to_order_ratio"] = 0

print("EMAR medication-administration table-derived feature extraction complete.")
print("EMAR table-derived feature count:", len(emar_engineered_cols))
print("Dataset shape after EMAR features:", df.shape)
print("Admissions with any EMAR administration record:", int((df["emar_administration_record_count"] > 0).sum()))

Starting EMAR medication-administration feature extraction...
Processed EMAR chunk: 0 rows kept: 514351
Processed EMAR chunk: 1 rows kept: 521301
Processed EMAR chunk: 2 rows kept: 524217
Processed EMAR chunk: 3 rows kept: 539620
Processed EMAR chunk: 4 rows kept: 509300
Processed EMAR chunk: 5 rows kept: 551875
Processed EMAR chunk: 6 rows kept: 525185
Processed EMAR chunk: 7 rows kept: 486877
Processed EMAR chunk: 8 rows kept: 510276
Processed EMAR chunk: 9 rows kept: 509599
Processed EMAR chunk: 10 rows kept: 520844
Processed EMAR chunk: 11 rows kept: 477375
Processed EMAR chunk: 12 rows kept: 512367
Processed EMAR chunk: 13 rows kept: 543066
Processed EMAR chunk: 14 rows kept: 547762
Processed EMAR chunk: 15 rows kept: 509186
Processed EMAR chunk: 16 rows kept: 526910
Processed EMAR chunk: 17 rows kept: 484768
Processed EMAR chunk: 18 rows kept: 512074
Processed EMAR chunk: 19 rows kept: 518388
Processed EMAR chunk: 20 rows kept: 526108
Processed EMAR chunk: 21 rows kept: 515420
Pr

/tmp/ipykernel_52805/1923925374.py:217: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["emar_not_given_rate"] = np.where(df["emar_administration_record_count"] > 0,
/tmp/ipykernel_52805/1923925374.py:219: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["emar_complete_dose_not_given_rate"] = np.where(df["emar_administration_record_count"] > 0,
/tmp/ipykernel_52805/1923925374.py:221: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance. 

**Create HCPCS event burden features**

HCPCS events are summarised as grouped utilisation and billing-procedure burden features. Raw HCPCS codes are not retained; only compact counts and clinically interpretable categories are used.

In [ ]:
#hcpcs event burden features using compact observation and total-use fields
#this keeps only the hcpcs fields that showed usable signal and drops sparse detailed categories.
print("Starting compact HCPCS event feature extraction...")

hcpcs_engineered_cols = ["num_hcpcs_events", "num_unique_hcpcs_codes",
    "num_hospital_observation_hcpcs_events", "hcpcs_observation_to_total_ratio"]

hcpcs_features = df[id_cols].copy()
#loop through each item in this feature block
for col in hcpcs_engineered_cols:
    hcpcs_features[col] = 0

hcpcs_path = hosp_path / "hcpcsevents.csv.gz"
d_hcpcs_path = hosp_path / "d_hcpcs.csv.gz"

if hcpcs_path.exists():
    hcpcs_available_cols = pd.read_csv(hcpcs_path, nrows=0).columns.tolist()
    hcpcs_code_col = "hcpcs_cd" if "hcpcs_cd" in hcpcs_available_cols else "code"
    hcpcs_usecols = [col for col in ["subject_id", "hadm_id", hcpcs_code_col, "short_description"] if col in hcpcs_available_cols]
    hcpcs_rows = []
    d_hcpcs = pd.DataFrame()

    if d_hcpcs_path.exists():
        d_hcpcs_available_cols = pd.read_csv(d_hcpcs_path, nrows=0).columns.tolist()
        d_code_col = "code" if "code" in d_hcpcs_available_cols else hcpcs_code_col
        d_hcpcs_usecols = [col for col in [d_code_col, "category", "long_description", "short_description"] if col in d_hcpcs_available_cols]
        d_hcpcs = pd.read_csv(d_hcpcs_path, usecols=d_hcpcs_usecols, low_memory=False)
        if d_code_col != hcpcs_code_col and d_code_col in d_hcpcs.columns:
            d_hcpcs = d_hcpcs.rename(columns={d_code_col: hcpcs_code_col})

    #loop through each item in this feature block
    for chunk_number, hcpcs_chunk in enumerate(pd.read_csv(hcpcs_path, usecols=hcpcs_usecols, chunksize=500000, low_memory=False)):
        hcpcs_chunk = hcpcs_chunk[hcpcs_chunk["hadm_id"].isin(current_hadm_ids)].copy()
        if hcpcs_chunk.empty:
            continue

        if not d_hcpcs.empty and hcpcs_code_col in hcpcs_chunk.columns:
            #join the derived features back to the admission table
            hcpcs_chunk = hcpcs_chunk.merge(d_hcpcs, on=hcpcs_code_col, how="left", suffixes=("", "_dictionary"))

        description_cols = [col for col in hcpcs_chunk.columns if col in ["short_description", "short_description_dictionary", "long_description", "category"]]
        if description_cols:
            description_text = hcpcs_chunk[description_cols].fillna("").astype(str).agg(" ".join, axis=1).str.lower()
        else:
            description_text = pd.Series("", index=hcpcs_chunk.index)

        hcpcs_chunk["hcpcs_event"] = 1
        hcpcs_chunk["hospital_observation_hcpcs_event"] = description_text.str.contains(
            "observation|hospital observation|obs care|observation care", regex=True, na=False).astype(int)

        agg_dict = {"num_hcpcs_events": ("hcpcs_event", "sum"),
            "num_hospital_observation_hcpcs_events": ("hospital_observation_hcpcs_event", "sum")}
        if hcpcs_code_col in hcpcs_chunk.columns:
            agg_dict["num_unique_hcpcs_codes"] = (hcpcs_code_col, "nunique")

        #summarise row-level records to admission-level features
        hcpcs_rows.append(hcpcs_chunk.groupby(id_cols, as_index=False).agg(**agg_dict))
        print("Processed HCPCS chunk:", chunk_number, "rows kept:", len(hcpcs_chunk))

    if hcpcs_rows:
        hcpcs_features = pd.concat(hcpcs_rows, ignore_index=True).groupby(id_cols, as_index=False).sum(min_count=1)

if "num_unique_hcpcs_codes" not in hcpcs_features.columns:
    hcpcs_features["num_unique_hcpcs_codes"] = 0
hcpcs_features["hcpcs_observation_to_total_ratio"] = (
    pd.to_numeric(hcpcs_features.get("num_hospital_observation_hcpcs_events", 0), errors="coerce").fillna(0) /
    pd.to_numeric(hcpcs_features.get("num_hcpcs_events", 0), errors="coerce").fillna(0).replace(0, np.nan)).fillna(0)

for col in hcpcs_engineered_cols:
    if col not in hcpcs_features.columns:
        hcpcs_features[col] = 0

df = df.drop(columns=[col for col in hcpcs_engineered_cols if col in df.columns], errors="ignore")
#join the derived features back to the admission table
df = df.merge(hcpcs_features[id_cols + hcpcs_engineered_cols], on=id_cols, how="left")
for col in hcpcs_engineered_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

print("Compact HCPCS feature extraction complete.")
print("HCPCS compact feature count:", len(hcpcs_engineered_cols))
print("Dataset shape after HCPCS features:", df.shape)

Starting compact HCPCS event feature extraction...
Processed HCPCS chunk: 0 rows kept: 80660
Compact HCPCS feature extraction complete.
HCPCS compact feature count: 4
Dataset shape after HCPCS features: (238565, 771)


**Create care-team complexity features**

Care-team features count distinct order-entry and ICU event caregivers without retaining raw provider or caregiver identifiers. These table-derived columns are proxies for care complexity and fragmentation, not staff-level effects.

In [ ]:
#care-team complexity features without retaining raw provider or caregiver identifiers
print("Starting care-team complexity feature extraction...")

care_team_engineered_cols = ["num_unique_order_providers", "num_unique_enter_providers", "num_unique_caregivers_icu",
    "num_provider_order_changes", "num_care_team_touchpoints"]
care_team_features = df[id_cols].copy()
#loop through each item in this feature block
for col in care_team_engineered_cols:
    if col not in care_team_features.columns:
        care_team_features[col] = 0

#poe order-provider counts are created in the poe block above and merged into df.
for col in ["num_unique_order_providers", "num_provider_order_changes"]:
    if col in df.columns:
        care_team_features[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

#emar enter-provider counts capture medication-administration team touchpoints without storing provider ids.
emar_enter_provider_rows = []
if emar_path.exists():
    emar_provider_usecols = ["subject_id", "hadm_id", "enter_provider_id"]
    #loop through each item in this feature block
    for chunk_number, emar_provider_chunk in enumerate(pd.read_csv(emar_path, usecols=emar_provider_usecols, chunksize=1000000, low_memory=False)):
        emar_provider_chunk = emar_provider_chunk[emar_provider_chunk["hadm_id"].isin(current_hadm_ids)].copy()
        if emar_provider_chunk.empty:
            continue
        emar_enter_provider_rows.append(emar_provider_chunk[id_cols + ["enter_provider_id"]].dropna(subset=["enter_provider_id"]).drop_duplicates())
        print("Processed EMAR provider chunk:", chunk_number, "rows kept:", len(emar_provider_chunk))
if emar_enter_provider_rows:
    #summarise row-level records to admission-level features
    emar_enter_provider_counts = pd.concat(emar_enter_provider_rows, ignore_index=True).drop_duplicates().groupby(id_cols, as_index=False)["enter_provider_id"].nunique()
    emar_enter_provider_counts = emar_enter_provider_counts.rename(columns={"enter_provider_id": "num_unique_enter_providers"})
    care_team_features = care_team_features.drop(columns=["num_unique_enter_providers"], errors="ignore").merge(
        emar_enter_provider_counts, on=id_cols, how="left")

#icu caregiver counts use selected icu event tables rather than raw caregiver ids.
icu_caregiver_rows = []
icu_caregiver_tables = ["inputevents.csv.gz", "procedureevents.csv.gz", "outputevents.csv.gz", "datetimeevents.csv.gz", "ingredientevents.csv.gz"]
for table_name in icu_caregiver_tables:
    table_path = icu_path / table_name
    if not table_path.exists():
        continue
    icu_usecols = ["subject_id", "hadm_id", "caregiver_id"]
    for chunk_number, icu_chunk in enumerate(pd.read_csv(table_path, usecols=icu_usecols, chunksize=1000000, low_memory=False)):
        icu_chunk = icu_chunk[icu_chunk["hadm_id"].isin(current_hadm_ids)].copy()
        if icu_chunk.empty:
            continue
        icu_caregiver_rows.append(icu_chunk[id_cols + ["caregiver_id"]].dropna(subset=["caregiver_id"]).drop_duplicates())
        print("Processed ICU caregiver table:", table_name, "chunk:", chunk_number, "rows kept:", len(icu_chunk))
if icu_caregiver_rows:
    icu_caregiver_counts = pd.concat(icu_caregiver_rows, ignore_index=True).drop_duplicates().groupby(id_cols, as_index=False)["caregiver_id"].nunique()
    icu_caregiver_counts = icu_caregiver_counts.rename(columns={"caregiver_id": "num_unique_caregivers_icu"})
    care_team_features = care_team_features.drop(columns=["num_unique_caregivers_icu"], errors="ignore").merge(
        icu_caregiver_counts, on=id_cols, how="left")

for col in ["num_unique_order_providers", "num_unique_enter_providers", "num_unique_caregivers_icu", "num_provider_order_changes"]:
    if col not in care_team_features.columns:
        care_team_features[col] = 0
    care_team_features[col] = pd.to_numeric(care_team_features[col], errors="coerce").fillna(0).astype(int)
care_team_features["num_care_team_touchpoints"] = (care_team_features["num_unique_order_providers"] +
    care_team_features["num_unique_enter_providers"] + care_team_features["num_unique_caregivers_icu"])

df = df.drop(columns=[col for col in care_team_engineered_cols if col in df.columns], errors="ignore")
#join the derived features back to the admission table
df = df.merge(care_team_features[id_cols + care_team_engineered_cols], on=id_cols, how="left")
for col in care_team_engineered_cols:
    df[col] = df[col].fillna(0).astype(int)

print("Care-team complexity table-derived feature extraction complete.")
print("Care-team table-derived feature count:", len(care_team_engineered_cols))
print("Dataset shape after care-team features:", df.shape)

Starting care-team complexity feature extraction...
Processed EMAR provider chunk: 0 rows kept: 523076
Processed EMAR provider chunk: 1 rows kept: 529401
Processed EMAR provider chunk: 2 rows kept: 534011
Processed EMAR provider chunk: 3 rows kept: 549029
Processed EMAR provider chunk: 4 rows kept: 518639
Processed EMAR provider chunk: 5 rows kept: 560873
Processed EMAR provider chunk: 6 rows kept: 533424
Processed EMAR provider chunk: 7 rows kept: 495922
Processed EMAR provider chunk: 8 rows kept: 519572
Processed EMAR provider chunk: 9 rows kept: 518025
Processed EMAR provider chunk: 10 rows kept: 530392
Processed EMAR provider chunk: 11 rows kept: 486671
Processed EMAR provider chunk: 12 rows kept: 521678
Processed EMAR provider chunk: 13 rows kept: 552951
Processed EMAR provider chunk: 14 rows kept: 558078
Processed EMAR provider chunk: 15 rows kept: 518202
Processed EMAR provider chunk: 16 rows kept: 536686
Processed EMAR provider chunk: 17 rows kept: 493951
Processed EMAR provide

**Create compact ICU event-burden features**

This block summarises ICU input, output, datetime, and procedure events into compact admission-level burden features. It includes event counts, rates, interruption/status markers, urine output, and selected high-acuity ICU intervention categories.

These are medical-acuity features rather than psychiatric features. They are included because some lower-performing subgroups appear more medically complex, and structured ICU event burden may help identify illness severity not captured by diagnosis codes alone.


In [ ]:
#compact icu event-burden table-derived columns from icu event tables
#sections:
#1. count icu input/output/datetime/procedure events as care-intensity proxies.
#2. add outputevent urine-output features for medical acuity in older or icu-linked admissions.
#3. add interruption/status indicators from inputevents and procedureevents where available.
#4. normalise selected counts by length of stay to reduce pure exposure effects.
#these features describe medical care intensity without adding raw item ids or caregiver identifiers.
print("Starting compact ICU event-burden feature extraction...")

icu_event_engineered_cols = ["num_inputevents", "num_outputevents", "num_procedureevents", "num_datetimeevents",
    "num_icu_input_events", "num_icu_output_events", "num_icu_datetime_events", "num_icu_procedure_events",
    "num_icu_event_warnings", "num_continuous_infusion_events", "num_inputevent_rate_changes",
    "num_paused_inputevents", "num_stopped_inputevents", "num_change_dose_rate_inputevents",
    "num_paused_procedureevents", "num_stopped_procedureevents", "num_active_icu_interventions",
    "icu_event_interruption_score", "icu_event_density_per_day", "num_chart_warnings", "had_chart_warning",
    "num_outputevents_last_24h_before_discharge", "urine_output_total", "urine_output_per_icu_day",
    "low_urine_output_flag", "late_icu_intervention_activity"]

icu_event_features = df[id_cols].copy()
#loop through each item in this feature block
for col in icu_event_engineered_cols:
    icu_event_features[col] = 0

admission_times_for_icu_events = current_admissions[["subject_id", "hadm_id", "admittime", "dischtime"]].copy()

icu_item_dictionary = d_items.copy() if "d_items" in globals() else pd.read_csv(icu_path / "d_items.csv.gz")
icu_item_dictionary["label_lower"] = icu_item_dictionary["label"].fillna("").astype(str).str.lower()
icu_item_dictionary["category_lower"] = icu_item_dictionary["category"].fillna("").astype(str).str.lower()
urine_output_itemids = set(icu_item_dictionary.loc[
    icu_item_dictionary["category_lower"].eq("output") &
    icu_item_dictionary["label_lower"].str.contains("urine|foley|void|urinary|gu irrigant", regex=True, na=False),
    "itemid"].astype(int))

#define count icu events in admission
def count_icu_events_in_admission(table_path, time_col, count_col, extra_builder=None, chunksize=1000000):
    if not table_path.exists():
        return df[id_cols].copy()
    usecols = ["subject_id", "hadm_id", time_col]
    if extra_builder is not None:
        usecols = extra_builder("usecols", usecols)
    rows = []
    #loop through each item in this feature block
    for chunk_number, chunk in enumerate(pd.read_csv(table_path, usecols=usecols, chunksize=chunksize, low_memory=False)):
        chunk = chunk[chunk["hadm_id"].isin(current_hadm_ids)].copy()
        if chunk.empty:
            continue
        chunk[time_col] = pd.to_datetime(chunk[time_col], errors="coerce")
        #join the derived features back to the admission table
        chunk = chunk.merge(admission_times_for_icu_events, on=["subject_id", "hadm_id"], how="left")
        chunk = chunk[(chunk[time_col].notna()) & (chunk["admittime"].notna()) &
            (chunk[time_col] >= chunk["admittime"]) &
            ((chunk["dischtime"].isna()) | (chunk[time_col] <= chunk["dischtime"]))].copy()
        if chunk.empty:
            continue
        chunk[count_col] = 1
        if extra_builder is not None:
            chunk = extra_builder("features", chunk)
        agg_dict = {count_col: (count_col, "sum")}
        if extra_builder is not None:
            agg_dict.update(extra_builder("agg", None))
        #summarise row-level records to admission-level features
        rows.append(chunk.groupby(id_cols, as_index=False).agg(**agg_dict))
        print("Processed ICU event chunk:", table_path.name, chunk_number, "rows kept:", len(chunk))
    if rows:
        return pd.concat(rows, ignore_index=True).groupby(id_cols, as_index=False).sum(min_count=1)
    return df[id_cols].copy()

#define inputevent extra
def inputevent_extra(mode, payload):
    if mode == "usecols":
        return payload + ["endtime", "rate", "originalrate", "ordercategorydescription", "statusdescription"]
    if mode == "features":
        text = payload["ordercategorydescription"].fillna("").astype(str).str.lower()
        status_text = payload["statusdescription"].fillna("").astype(str).str.lower()
        payload["continuous_infusion_event"] = (payload["rate"].notna() | payload["originalrate"].notna() |
            text.str.contains("continuous|infusion|drip", regex=True)).astype(int)
        payload["inputevent_rate_change"] = ((pd.to_numeric(payload["rate"], errors="coerce") !=
            pd.to_numeric(payload["originalrate"], errors="coerce")) & payload["rate"].notna() & payload["originalrate"].notna()).astype(int)
        payload["paused_inputevent"] = status_text.str.contains("pause|paused", regex=True).astype(int)
        payload["stopped_inputevent"] = status_text.str.contains("stop|stopped", regex=True).astype(int)
        payload["change_dose_rate_inputevent"] = status_text.str.contains("change.*dose|change.*rate|changedose|changerate", regex=True).astype(int)
        payload["late_icu_intervention_activity"] = ((payload["dischtime"] - payload["starttime"]).dt.total_seconds().between(0, 24 * 3600, inclusive="both")).astype(int)
        return payload
    if mode == "agg":
        return {"num_continuous_infusion_events": ("continuous_infusion_event", "sum"),
            "num_inputevent_rate_changes": ("inputevent_rate_change", "sum"),
            "num_paused_inputevents": ("paused_inputevent", "sum"),
            "num_stopped_inputevents": ("stopped_inputevent", "sum"),
            "num_change_dose_rate_inputevents": ("change_dose_rate_inputevent", "sum"),
            "late_icu_intervention_activity": ("late_icu_intervention_activity", "sum")}

#define outputevent extra
def outputevent_extra(mode, payload):
    if mode == "usecols":
        return payload + ["itemid", "value", "valueuom"]
    if mode == "features":
        payload["outputevent_last_24h_before_discharge"] = ((payload["dischtime"] - payload["charttime"]).dt.total_seconds().between(0, 24 * 3600, inclusive="both")).astype(int)
        payload["value_numeric"] = pd.to_numeric(payload["value"], errors="coerce")
        payload["urine_output_value"] = np.where(payload["itemid"].isin(urine_output_itemids), payload["value_numeric"], 0)
        return payload
    if mode == "agg":
        return {"num_outputevents_last_24h_before_discharge": ("outputevent_last_24h_before_discharge", "sum"),
            "urine_output_total": ("urine_output_value", "sum")}

#define datetimeevent extra
def datetimeevent_extra(mode, payload):
    if mode == "usecols":
        return payload + ["warning"]
    if mode == "features":
        payload["icu_event_warning"] = pd.to_numeric(payload["warning"], errors="coerce").fillna(0).astype(int)
        return payload
    if mode == "agg":
        return {"num_icu_event_warnings": ("icu_event_warning", "sum")}

#define procedureevent extra
def procedureevent_extra(mode, payload):
    if mode == "usecols":
        return payload + ["endtime", "statusdescription"]
    if mode == "features":
        status_text = payload["statusdescription"].fillna("").astype(str).str.lower()
        payload["paused_procedureevent"] = status_text.str.contains("pause|paused", regex=True).astype(int)
        payload["stopped_procedureevent"] = status_text.str.contains("stop|stopped", regex=True).astype(int)
        payload["late_icu_intervention_activity"] = ((payload["dischtime"] - payload["starttime"]).dt.total_seconds().between(0, 24 * 3600, inclusive="both")).astype(int)
        return payload
    if mode == "agg":
        return {"num_paused_procedureevents": ("paused_procedureevent", "sum"),
            "num_stopped_procedureevents": ("stopped_procedureevent", "sum"),
            "late_icu_intervention_activity": ("late_icu_intervention_activity", "sum")}

input_counts = count_icu_events_in_admission(icu_path / "inputevents.csv.gz", "starttime", "num_icu_input_events", inputevent_extra)
output_counts = count_icu_events_in_admission(icu_path / "outputevents.csv.gz", "charttime", "num_icu_output_events", outputevent_extra)
datetime_counts = count_icu_events_in_admission(icu_path / "datetimeevents.csv.gz", "charttime", "num_icu_datetime_events", datetimeevent_extra)
procedure_counts = count_icu_events_in_admission(icu_path / "procedureevents.csv.gz", "starttime", "num_icu_procedure_events", procedureevent_extra)

for event_frame in [input_counts, output_counts, datetime_counts, procedure_counts]:
    #join the derived features back to the admission table
    icu_event_features = icu_event_features.merge(event_frame, on=id_cols, how="left", suffixes=("", "_new"))
    for col in [col for col in icu_event_engineered_cols if col + "_new" in icu_event_features.columns]:
        icu_event_features[col] = pd.to_numeric(icu_event_features[col], errors="coerce").fillna(0) + pd.to_numeric(
            icu_event_features[col + "_new"], errors="coerce").fillna(0)
        icu_event_features = icu_event_features.drop(columns=[col + "_new"])

icu_event_features["num_inputevents"] = icu_event_features["num_icu_input_events"]
icu_event_features["num_outputevents"] = icu_event_features["num_icu_output_events"]
icu_event_features["num_procedureevents"] = icu_event_features["num_icu_procedure_events"]
icu_event_features["num_datetimeevents"] = icu_event_features["num_icu_datetime_events"]
icu_event_features["num_chart_warnings"] = icu_event_features["num_icu_event_warnings"]
icu_event_features["had_chart_warning"] = (icu_event_features["num_chart_warnings"].fillna(0) > 0).astype(int)
icu_event_features["num_active_icu_interventions"] = (icu_event_features["num_icu_input_events"].fillna(0) +
    icu_event_features["num_icu_procedure_events"].fillna(0))
icu_event_features["late_icu_intervention_activity"] = (icu_event_features["late_icu_intervention_activity"].fillna(0) > 0).astype(int)
icu_event_features["icu_event_interruption_score"] = (icu_event_features["num_paused_inputevents"].fillna(0) +
    icu_event_features["num_stopped_inputevents"].fillna(0) +
    icu_event_features["num_change_dose_rate_inputevents"].fillna(0) +
    icu_event_features["num_paused_procedureevents"].fillna(0) +
    icu_event_features["num_stopped_procedureevents"].fillna(0))

icu_los_days = df[id_cols + ["hospital_los_days"]].copy() if "hospital_los_days" in df.columns else df[id_cols].assign(hospital_los_days=1)
#join the derived features back to the admission table
icu_event_features = icu_event_features.merge(icu_los_days, on=id_cols, how="left")
icu_event_features["icu_event_density_per_day"] = ((icu_event_features["num_icu_input_events"].fillna(0) +
    icu_event_features["num_icu_output_events"].fillna(0) + icu_event_features["num_icu_datetime_events"].fillna(0) +
    icu_event_features["num_icu_procedure_events"].fillna(0)) /
    icu_event_features["hospital_los_days"].fillna(1).clip(lower=1)).replace([np.inf, -np.inf], 0).fillna(0)
icu_event_features["urine_output_per_icu_day"] = (icu_event_features["urine_output_total"].fillna(0) /
    icu_event_features["hospital_los_days"].fillna(1).clip(lower=1)).replace([np.inf, -np.inf], 0).fillna(0)
icu_event_features["low_urine_output_flag"] = ((icu_event_features["num_icu_output_events"].fillna(0) > 0) &
    (icu_event_features["urine_output_per_icu_day"].fillna(0) < 500)).astype(int)
icu_event_features = icu_event_features.drop(columns=["hospital_los_days"], errors="ignore")

for col in icu_event_engineered_cols:
    if col not in icu_event_features.columns:
        icu_event_features[col] = 0

df = df.drop(columns=[col for col in icu_event_engineered_cols if col in df.columns], errors="ignore")
df = df.merge(icu_event_features[id_cols + icu_event_engineered_cols], on=id_cols, how="left")
for col in icu_event_engineered_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    if col not in ["icu_event_density_per_day", "urine_output_total", "urine_output_per_icu_day"]:
        df[col] = df[col].astype(int)

print("Compact ICU event-burden table-derived feature extraction complete.")
print("ICU event table-derived feature count:", len(icu_event_engineered_cols))
print("Dataset shape after ICU event features:", df.shape)

Starting compact ICU event-burden feature extraction...
Processed ICU event chunk: inputevents.csv.gz 0 rows kept: 523805
Processed ICU event chunk: inputevents.csv.gz 1 rows kept: 508586
Processed ICU event chunk: inputevents.csv.gz 2 rows kept: 483591
Processed ICU event chunk: inputevents.csv.gz 3 rows kept: 507484
Processed ICU event chunk: inputevents.csv.gz 4 rows kept: 501301
Processed ICU event chunk: inputevents.csv.gz 5 rows kept: 497466
Processed ICU event chunk: inputevents.csv.gz 6 rows kept: 498380
Processed ICU event chunk: inputevents.csv.gz 7 rows kept: 496966
Processed ICU event chunk: inputevents.csv.gz 8 rows kept: 487826
Processed ICU event chunk: inputevents.csv.gz 9 rows kept: 523238
Processed ICU event chunk: inputevents.csv.gz 10 rows kept: 468130
Processed ICU event chunk: outputevents.csv.gz 0 rows kept: 501081
Processed ICU event chunk: outputevents.csv.gz 1 rows kept: 481072
Processed ICU event chunk: outputevents.csv.gz 2 rows kept: 477679
Processed ICU ev

**Final checks and save T2.1 dataset**

The final dataset is checked for shape, column names, missing values, and readmission label distribution. This extracted dataset forms the initial modelling dataset and will be used for cleaning and preprocessing in T2.2.

In [47]:
#set file paths and create output folders
print("Final T2.1 dataset shape:", df.shape)
print("\nICU type distribution:")
print(df["icu_type"].value_counts(dropna=False))

print("\nService feature columns:")
service_cols = ["num_service_records", "num_unique_services", "num_service_transfers",
    "had_service_transfer", "first_service", "last_service"]
print(df[service_cols].head())
print(df[service_cols].isna().sum())

print("\nTransfer feature columns:")
transfer_cols = ["num_transfer_events", "num_careunit_transfers", "num_unique_careunits",
    "had_ed_transfer_record", "had_unknown_transfer_careunit"]
print(df[transfer_cols].head())
print(df[transfer_cols].isna().sum())

print("\nProcedure feature columns:")
procedure_cols = ["num_procedures", "num_unique_procedure_codes", "had_procedure"]
print(df[procedure_cols].head())
print(df[procedure_cols].isna().sum())

print("\nDRG feature columns:")
drg_cols = ["drg_severity", "drg_mortality", "drg_code_count"]
print(df[drg_cols].head())
print(df[drg_cols].isna().sum())
print("\nVital sign feature columns:")
vital_cols = [col for col in df.columns if "_vital_" in col]
print(vital_cols)

if len(vital_cols) > 0:
    print("\nVital sign measured indicator counts:")
    vital_measured_cols = [col for col in vital_cols if col.endswith("_vital_measured")]
    print(df[vital_measured_cols].sum().sort_values(ascending=False))
    print("\nMissing vital sign values:")
    print(df[vital_cols].isna().sum().sort_values(ascending=False).head(30))

print("\nFinal columns:")
print(df.columns.tolist())

print("\nReadmission label distribution:")
print(df["readmitted_30d"].value_counts())

print("\nReadmission label distribution (%):")
print(df["readmitted_30d"].value_counts(normalize=True) * 100)

missing_summary = pd.DataFrame({"missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)}).sort_values(
    "missing_percent", ascending=False)

print("\nMissing values summary:")
print(missing_summary)

categorical_cols = ["gender", "anchor_year_group", "admission_type",
    "admission_location", "discharge_location", "insurance",
    "language", "marital_status", "race", "first_icu_careunit", "last_icu_careunit"]

#loop through each item in this feature block
for col in categorical_cols:
    print(f"\n{'='*50}")
    print(col)
    print(f"{'='*50}")
    print(df[col].value_counts(dropna=False).head(10))

print("\nPreview final dataset:")
print("\nDataset memory usage (MB):")
print(round(df.memory_usage(deep=True).sum() / 1024**2, 2))
print(df.head())
print("\nNumber of unique patients:")
print(df["subject_id"].nunique())

print("\nNumber of unique admissions:")
print(df["hadm_id"].nunique())

#save output
print("\nDuplicate subject_id + hadm_id rows:")
print(df.duplicated(subset=["subject_id", "hadm_id"]).sum())
output_file = output_path / "t2_1_initial_modelling_dataset.csv"
#save this table or artefact for later review
df.to_csv(output_file, index=False)
print("\nSaved T2.1 dataset to:")
print(output_file)

#create extraction audit summary for reporting
t2_1_audit = pd.DataFrame({"check": ["Rows in final extracted dataset", "Unique patients",
        "Unique admissions", "Duplicate subject_id + hadm_id rows", "Missing subject_id",
        "Missing hadm_id", "Missing readmitted_30d"],
    "value": [len(df), df["subject_id"].nunique(), df["hadm_id"].nunique(),
        df.duplicated(subset=["subject_id", "hadm_id"]).sum(), df["subject_id"].isna().sum(),
        df["hadm_id"].isna().sum(), df["readmitted_30d"].isna().sum()]})

print("T2.1 extraction audit summary:")
print(t2_1_audit)

Final T2.1 dataset shape: (238565, 800)

ICU type distribution:
icu_type
No ICU                 198054
Surgical/Trauma ICU     17110
Medical ICU             10685
Cardiac ICU              8952
Neuro ICU                3631
Other ICU                 133
Name: count, dtype: int64

Service feature columns:
   num_service_records  num_unique_services  num_service_transfers  \
0                    1                    1                      0   
1                    1                    1                      0   
2                    1                    1                      0   
3                    1                    1                      0   
4                    1                    1                      0   

   had_service_transfer first_service last_service  
0                     0           MED          MED  
1                     0           MED          MED  
2                     0           MED          MED  
3                     0           MED          MED  
4        

In [48]:
#compute explanation tables for model interpretation
print("Final T2.1 dataset shape:", df.shape)

feature_groups = {"Demographic/admission": ["gender", "anchor_age", "admission_type", "race", "hospital_los_days"],
    "ICU": ["had_icu_stay", "icu_stay_count", "total_icu_los_days", "icu_type"],
    "Care pathway": ["num_service_transfers", "had_service_transfer", "num_transfer_events", "num_careunit_transfers", "num_unique_careunits", "psych_service_involved_anytime", "medicine_and_psych_services_both_flag", "internal_transfer_from_psych_flag", "emergency_room_admission_flag", "transfer_from_hospital_flag", "transfer_from_snf_flag"],
    "Severity": ["drg_severity", "drg_mortality"],
    "Procedures": ["num_procedures", "num_unique_procedure_codes", "had_procedure", "had_mechanical_ventilation_procedure", "had_dialysis_procedure", "had_central_line_procedure", "had_ect_procedure", "had_restraint_related_procedure", "ect_procedure_count", "ect_within_first_72h", "ect_during_admission_flag"],
    "Diagnosis": ["num_total_diagnoses", "num_psych_diagnoses", "num_nonpsych_diagnoses", "has_self_harm_or_suicidal_ideation", "has_alcohol_related_disorder", "has_opioid_related_disorder", "has_stimulant_related_disorder", "has_cannabis_related_disorder", "has_tobacco_or_nicotine_related_disorder", "has_sedative_hypnotic_related_disorder", "has_polysubstance_related_disorder"],
    "Comorbidity": [col for col in df.columns if col.startswith("elixhauser_") or col in ["physical_comorbidity_count", "charlson_comorbidity_index_simplified", "elixhauser_comorbidity_group_count_simplified"]],
    "Medication": ["num_unique_drugs", "num_psych_med_classes", "had_antibiotic_exposure", "had_opioid_exposure", "had_steroid_exposure", "had_iv_prescription", "num_unique_antipsychotics", "antipsychotic_prescription_count", "antipsychotic_polypharmacy_2plus", "antipsychotic_polypharmacy_3plus", "antipsychotic_plus_benzodiazepine", "antipsychotic_plus_mood_stabiliser", "had_long_acting_injectable_antipsychotic", "emar_administration_record_count", "emar_not_given_event_count", "emar_had_iv_administration", "emar_had_infusion_administration", "emar_not_given_rate", "emar_complete_dose_not_given_rate", "emar_administered_to_order_ratio", "emar_missed_or_not_given_count_first_24h", "emar_missed_or_not_given_count_first_72h"],
    "Admission/social/LOS proxies": ["discharged_against_advice", "not_married_flag", "single_or_divorced_or_widowed", "non_english_language_flag", "public_insurance_flag", "medicaid_flag", "medicare_flag", "insurance_missing_flag", "admitted_from_facility_flag", "admitted_from_hospital_transfer_flag", "discharged_home_flag", "discharged_to_facility_flag", "discharged_to_psych_facility_flag", "los_under_2_days", "los_under_7_days", "los_7_to_30_days", "los_30plus_days", "los_60plus_days"],
    "Microbiology/infection": ["microbiology_test_count", "positive_culture_flag", "blood_culture_positive", "urine_culture_positive", "respiratory_culture_positive", "distinct_organism_count", "suspected_infection_flag"],
    "ED timing": ["has_ed_timing", "ed_length_of_stay_hours", "ed_to_admission_hours", "ed_to_ward_delay_hours"],
    "Prior utilisation windows": [col for col in df.columns if col.startswith("previous_total_admissions_") or col.startswith("previous_psych_admissions_") or col.startswith("previous_nonpsych_admissions_") or col in ["days_since_previous_hospital_admission", "previous_psych_to_total_admission_ratio", "has_previous_icu_admission", "days_since_previous_icu_admission", "previous_2plus_psych_admissions", "previous_3plus_psych_admissions", "previous_2plus_total_admissions_365d", "previous_3plus_total_admissions_365d", "frequent_psych_admitter_flag"]],
    "Body measures": ["latest_bmi_before_admission", "latest_weight_kg_before_admission", "latest_height_cm_before_admission", "obesity_flag", "underweight_flag", "missing_bmi_flag"],
    "Laboratory": [col for col in df.columns if col.endswith("_lab_mean_value")],
    "Vital signs": [col for col in df.columns if col.endswith("_vital_mean_value")]}

for group, cols in feature_groups.items():
    existing_cols = [col for col in cols if col in df.columns]
    print("\n" + group + ": " + str(len(existing_cols)) + " columns")
    print(existing_cols)

Final T2.1 dataset shape: (238565, 800)

Demographic/admission: 5 columns
['gender', 'anchor_age', 'admission_type', 'race', 'hospital_los_days']

ICU: 4 columns
['had_icu_stay', 'icu_stay_count', 'total_icu_los_days', 'icu_type']

Care pathway: 11 columns
['num_service_transfers', 'had_service_transfer', 'num_transfer_events', 'num_careunit_transfers', 'num_unique_careunits', 'psych_service_involved_anytime', 'medicine_and_psych_services_both_flag', 'internal_transfer_from_psych_flag', 'emergency_room_admission_flag', 'transfer_from_hospital_flag', 'transfer_from_snf_flag']

Severity: 2 columns
['drg_severity', 'drg_mortality']

Procedures: 11 columns
['num_procedures', 'num_unique_procedure_codes', 'had_procedure', 'had_mechanical_ventilation_procedure', 'had_dialysis_procedure', 'had_central_line_procedure', 'had_ect_procedure', 'had_restraint_related_procedure', 'ect_procedure_count', 'ect_within_first_72h', 'ect_during_admission_flag']

Diagnosis: 11 columns
['num_total_diagnoses'